
## 1. Load & Audit Data

### 1.1 Load Required Datasets

**Purpose:** Load the processed CERT r6.2 user-day dataset, historical user baselines, and approved attack scenarios to establish the inputs required for user eligibility assessment and scenario assignment.


In [1]:

# Purpose: Load the CERT r6.2 user-day dataset, historical user
# baselines, and approved attack scenarios without modifying them.

from pathlib import Path
import pandas as pd

# Project structure: FYP_Data_Exfiltration/Attack_Simulation/notebooks/
PROJECT_ROOT = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
)

DATA_ENGINEERING = PROJECT_ROOT / "Data_Engineering" / "Data_engineering"
ATTACK_DATA = PROJECT_ROOT / "Attack_Simulation" / "data"

dataset_paths = {
    "User-Day Dataset": (
        DATA_ENGINEERING / "data" / "processed" / "user_day_r62_v1.parquet"
    ),
    "User Baseline": (
        DATA_ENGINEERING / "data" / "features" / "user_baseline_r62_v1.parquet"
    ),
    "Approved Scenarios": (
        ATTACK_DATA / "approved_attack_scenarios_r62.csv"
    )
}

# Check all input files before loading.
for name, path in dataset_paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name} not found: {path}")

user_day = pd.read_parquet(dataset_paths["User-Day Dataset"])
user_baseline = pd.read_parquet(dataset_paths["User Baseline"])
approved_scenarios = pd.read_csv(dataset_paths["Approved Scenarios"])

dataset_summary = pd.DataFrame([
    {
        "dataset": "User-Day Dataset",
        "rows": user_day.shape[0],
        "columns": user_day.shape[1]
    },
    {
        "dataset": "User Baseline",
        "rows": user_baseline.shape[0],
        "columns": user_baseline.shape[1]
    },
    {
        "dataset": "Approved Scenarios",
        "rows": approved_scenarios.shape[0],
        "columns": approved_scenarios.shape[1]
    }
])

print("DATASET LOADING SUMMARY")
print("Datasets loaded:", len(dataset_summary))
display(dataset_summary)

assert len(user_baseline) == 3995
assert len(approved_scenarios) == 1143

print("\nDataset loading: PASSED")


DATASET LOADING SUMMARY
Datasets loaded: 3


,dataset,rows,columns
0,User-Day Dataset,1394010,34
1,User Baseline,3995,36
2,Approved Scenarios,1143,14



Dataset loading: PASSED



### 1.2 Dataset Structure Audit

**Purpose:** Inspect the column names, data types, and missing values of the three loaded datasets to identify the fields required for user eligibility assessment and scenario assignment.


In [2]:

# Purpose: Audit the structure, data types, and missing values
# of the three datasets without modifying their contents.

datasets = {
    "User-Day Dataset": user_day,
    "User Baseline": user_baseline,
    "Approved Scenarios": approved_scenarios
}

for name, df in datasets.items():
    print(f"\n{'=' * 75}")
    print(f"DATASET STRUCTURE AUDIT: {name}")
    print(f"{'=' * 75}")

    print(f"Full dataset dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns")

    structure_audit = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(dtype) for dtype in df.dtypes],
        "missing_count": df.isna().sum().values,
        "missing_pct": (
            df.isna().mean().values * 100
        ).round(2)
    })

    display(structure_audit)

    print(f"Total missing cells: {df.isna().sum().sum():,}")
    print(f"Duplicate column names: {df.columns.duplicated().sum()}")



DATASET STRUCTURE AUDIT: User-Day Dataset
Full dataset dimensions: 1,394,010 rows x 34 columns


,column,dtype,missing_count,missing_pct
0,user,str,0,0.00
1,day,object,0,0.00
2,day_of_week,int32,0,0.00
3,is_weekend,int32,0,0.00
4,email_event_count,int64,0,0.00
5,email_send_count,float64,0,0.00
6,email_view_count,float64,0,0.00
7,external_email_count,float64,0,0.00
8,internal_email_count,float64,0,0.00
9,total_sent_email_bytes,float64,0,0.00


Total missing cells: 881
Duplicate column names: 0

DATASET STRUCTURE AUDIT: User Baseline
Full dataset dimensions: 3,995 rows x 36 columns


,column,dtype,missing_count,missing_pct
0,user,str,0,0.00
1,baseline_period_start,object,0,0.00
2,baseline_period_end,object,0,0.00
3,baseline_workdays,int64,0,0.00
4,baseline_first_day,object,0,0.00
5,baseline_last_day,object,0,0.00
6,email_active_days,int64,0,0.00
7,email_active_rate,float64,0,0.00
8,email_count_mean_active,float64,1,0.03
9,email_count_p50,float64,1,0.03


Total missing cells: 39,564
Duplicate column names: 0

DATASET STRUCTURE AUDIT: Approved Scenarios
Full dataset dimensions: 1,143 rows x 14 columns


,column,dtype,missing_count,missing_pct
0,scenario_id,str,0,0.0
1,duration,int64,0,0.0
2,off_days,int64,0,0.0
3,active_days,int64,0,0.0
4,switching_level,str,0,0.0
5,min_switches,int64,0,0.0
6,max_switches,int64,0,0.0
7,target_switches,int64,0,0.0
8,target_switching_rate,float64,0,0.0
9,two_channel_days,int64,0,0.0


Total missing cells: 0
Duplicate column names: 0



### 1.3 Date Coverage & User Integrity Audit

**Purpose:** Verify dataset date coverage, user consistency, and user-day uniqueness before applying historical channel eligibility and attack-period constraints.


In [3]:

# Purpose: Verify date coverage, user consistency, and user-day
# uniqueness without modifying the original datasets.

user_day_dates = pd.to_datetime(user_day["day"], errors="coerce")
baseline_start = pd.to_datetime(
    user_baseline["baseline_period_start"], errors="coerce"
)
baseline_end = pd.to_datetime(
    user_baseline["baseline_period_end"], errors="coerce"
)

day_users = set(user_day["user"].dropna())
baseline_users = set(user_baseline["user"].dropna())

print("DATE COVERAGE AUDIT")
print("User-Day earliest date:", user_day_dates.min())
print("User-Day latest date:", user_day_dates.max())
print("Baseline earliest start:", baseline_start.min())
print("Baseline latest end:", baseline_end.max())

print("\nUSER INTEGRITY AUDIT")
print("Unique User-Day users:", len(day_users))
print("Unique Baseline users:", len(baseline_users))
print("Users in both datasets:", len(day_users & baseline_users))
print("Baseline users missing from User-Day:", len(baseline_users - day_users))
print("User-Day users missing from Baseline:", len(day_users - baseline_users))

print("\nDATA QUALITY AUDIT")
print("Missing/invalid User-Day dates:", user_day_dates.isna().sum())
print("Missing/invalid Baseline start dates:", baseline_start.isna().sum())
print("Missing/invalid Baseline end dates:", baseline_end.isna().sum())
print("Duplicate User-Day (user, day) records:",
      user_day.duplicated(subset=["user", "day"]).sum())
print("Duplicate Baseline users:",
      user_baseline.duplicated(subset=["user"]).sum())

print("\nATTACK PERIOD COVERAGE")
attack_start = pd.Timestamp("2011-01-01")
attack_end = pd.Timestamp("2011-06-01")

attack_period_mask = user_day_dates.between(
    attack_start, attack_end, inclusive="both"
)

print("Attack period:", attack_start.date(), "to", attack_end.date())
print("User-Day records within attack period:",
      f"{attack_period_mask.sum():,}")
print("Unique users within attack period:",
      user_day.loc[attack_period_mask, "user"].nunique())


DATE COVERAGE AUDIT
User-Day earliest date: 2010-01-02 00:00:00
User-Day latest date: 2011-06-01 00:00:00
Baseline earliest start: 2010-01-02 00:00:00
Baseline latest end: 2010-12-31 00:00:00

USER INTEGRITY AUDIT
Unique User-Day users: 4000
Unique Baseline users: 3995
Users in both datasets: 3995
Baseline users missing from User-Day: 0
User-Day users missing from Baseline: 5

DATA QUALITY AUDIT
Missing/invalid User-Day dates: 0
Missing/invalid Baseline start dates: 0
Missing/invalid Baseline end dates: 0
Duplicate User-Day (user, day) records: 0
Duplicate Baseline users: 0

ATTACK PERIOD COVERAGE
Attack period: 2011-01-01 to 2011-06-01
User-Day records within attack period: 395,562
Unique users within attack period: 3741



### 1.4 Baseline Exclusion Verification

**Purpose:** Identify users present in the User-Day dataset but absent from the historical baseline, and verify whether they correspond exactly to the five known malicious users excluded from attack simulation.


In [5]:

# Purpose: Verify excluded baseline users against the processed
# CERT r6.2 ground-truth user dataset.

ground_truth_path = (
    DATA_ENGINEERING
    / "data"
    / "processed"
    / "ground_truth_users_r62.parquet"
)

assert ground_truth_path.is_file(), (
    f"Ground-truth file not found: {ground_truth_path}"
)

ground_truth = pd.read_parquet(ground_truth_path)

assert "user" in ground_truth.columns, (
    "Ground-truth dataset does not contain a user column."
)

ground_truth_users = set(ground_truth["user"].dropna())

print("GROUND-TRUTH USER VERIFICATION")
print("Ground-truth users:", sorted(ground_truth_users))
print("Excluded baseline users:", sorted(missing_baseline_users))

print("\nVALIDATION")
print(
    "Exact match:",
    ground_truth_users == missing_baseline_users
)

assert ground_truth_users == missing_baseline_users, (
    "Ground-truth users do not match baseline exclusions."
)

print("\nGround-truth verification: PASSED")


GROUND-TRUTH USER VERIFICATION
Ground-truth users: ['ACM2278', 'CDE1846', 'CMP2946', 'MBG3183', 'PLJ1771']
Excluded baseline users: ['ACM2278', 'CDE1846', 'CMP2946', 'MBG3183', 'PLJ1771']

VALIDATION
Exact match: True

Ground-truth verification: PASSED



## 2. Historical Channel Eligibility

### 2.1 Identify Historically Active Channels

**Purpose:** Identify each user's historically active Email, USB, and HTTP channels using baseline activity-day counts, establishing which channels can be used for synthetic attack assignment.


In [6]:

# Purpose: Identify historically active Email, USB, and HTTP
# channels for each user using baseline activity-day counts.

channel_eligibility = user_baseline[
    ["user", "email_active_days", "usb_active_days",
     "http_upload_active_days"]
].copy()

channel_columns = {
    "Email": "email_active_days",
    "USB": "usb_active_days",
    "HTTP": "http_upload_active_days"
}

for channel, column in channel_columns.items():
    channel_eligibility[f"{channel.lower()}_eligible"] = (
        channel_eligibility[column] > 0
    )

eligibility_columns = [
    "email_eligible", "usb_eligible", "http_eligible"
]

channel_eligibility["eligible_channel_count"] = (
    channel_eligibility[eligibility_columns].sum(axis=1)
)

channel_eligibility["eligible_channels"] = (
    channel_eligibility[eligibility_columns]
    .apply(
        lambda row: "+".join(
            channel
            for channel, eligible in zip(
                ["Email", "USB", "HTTP"], row
            )
            if eligible
        ) or "None",
        axis=1
    )
)

print("HISTORICAL CHANNEL ELIGIBILITY")
print("Total baseline users:", f"{len(channel_eligibility):,}")

print("\nCHANNEL COMBINATION DISTRIBUTION")
display(
    channel_eligibility["eligible_channels"]
    .value_counts()
    .rename_axis("channel_combination")
    .reset_index(name="users")
)

print("\nELIGIBLE CHANNEL COUNT DISTRIBUTION")
display(
    channel_eligibility["eligible_channel_count"]
    .value_counts()
    .sort_index()
    .rename_axis("eligible_channel_count")
    .reset_index(name="users")
)

print("\nFull dataset dimensions:", channel_eligibility.shape)
print("Preview: First 15 rows only (not the complete dataset)")
display(channel_eligibility.head(15))

assert channel_eligibility["user"].is_unique
assert len(channel_eligibility) == len(user_baseline)

print("\nHistorical channel identification: PASSED")


HISTORICAL CHANNEL ELIGIBILITY
Total baseline users: 3,995

CHANNEL COMBINATION DISTRIBUTION


,channel_combination,users
0,Email,2915
1,Email+USB,464
2,Email+USB+HTTP,319
3,Email+HTTP,296
4,None,1



ELIGIBLE CHANNEL COUNT DISTRIBUTION


,eligible_channel_count,users
0,0,1
1,1,2915
2,2,760
3,3,319



Full dataset dimensions: (3995, 9)
Preview: First 15 rows only (not the complete dataset)


,user,email_active_days,usb_active_days,http_upload_active_days,email_eligible,usb_eligible,http_eligible,eligible_channel_count,eligible_channels
0,AAB0162,153,0,0,True,False,False,1,Email
1,AAB0398,199,0,0,True,False,False,1,Email
2,AAC0610,97,133,0,True,True,False,2,Email+USB
3,AAC0668,173,0,0,True,False,False,1,Email
4,AAC3270,17,0,0,True,False,False,1,Email
5,AAD2188,32,0,0,True,False,False,1,Email
6,AAD3030,168,0,0,True,False,False,1,Email
7,AAF0819,155,203,0,True,True,False,2,Email+USB
8,AAF3937,38,0,0,True,False,False,1,Email
9,AAG1447,181,0,0,True,False,False,1,Email



Historical channel identification: PASSED



### 2.2 Verify Channel Definitions Against Daily Activity

**Purpose:** Independently verify that historical channel activity-day counts correspond to external email events, USB copy-to-removable events, and HTTP upload events recorded in the User-Day dataset.


In [9]:

# Purpose: Verify baseline active-day counts against the documented
# channel definitions and baseline workday eligibility conditions.

baseline_days = user_day.loc[
    (pd.to_datetime(user_day["day"]) < pd.Timestamp("2011-01-01"))
    & (user_day["is_weekend"] == 0)
    & (user_day["logon_count"] > 0)
].copy()

channel_mapping = {
    "Email": ("external_email_count", "email_active_days"),
    "USB": ("usb_copy_count", "usb_active_days"),
    "HTTP": ("www_upload_count", "http_upload_active_days")
}

validation_results = []

for channel, (daily_col, baseline_col) in channel_mapping.items():

    recalculated = (
        baseline_days[daily_col].gt(0)
        .groupby(baseline_days["user"])
        .sum()
        .rename("recalculated_days")
    )

    comparison = user_baseline[
        ["user", baseline_col]
    ].merge(
        recalculated,
        left_on="user",
        right_index=True,
        how="left"
    )

    comparison["recalculated_days"] = (
        comparison["recalculated_days"].fillna(0).astype(int)
    )

    mismatches = (
        comparison[baseline_col] != comparison["recalculated_days"]
    ).sum()

    validation_results.append({
        "channel": channel,
        "daily_column": daily_col,
        "baseline_column": baseline_col,
        "users_checked": len(comparison),
        "mismatched_users": int(mismatches)
    })

channel_definition_validation = pd.DataFrame(validation_results)

print("DOCUMENTED CHANNEL DEFINITION VALIDATION")
print("Eligible historical workday records:", f"{len(baseline_days):,}")
display(channel_definition_validation)

assert (
    channel_definition_validation["mismatched_users"] == 0
).all(), "Baseline active-day counts differ from documented calculations."

print("\nChannel definition validation: PASSED")


DOCUMENTED CHANNEL DEFINITION VALIDATION
Eligible historical workday records: 977,519


,channel,daily_column,baseline_column,users_checked,mismatched_users
0,Email,external_email_count,email_active_days,3995,0
1,USB,usb_copy_count,usb_active_days,3995,0
2,HTTP,www_upload_count,http_upload_active_days,3995,0



Channel definition validation: PASSED



### 2.3 Baseline P95 Validation

**Purpose:** Verify that every historically active exfiltration channel has a valid P95 baseline value, ensuring reliable calculation of channel-specific attack injection intensities.


In [10]:

# Purpose: Validate P95 availability and numerical validity for
# historically active Email, USB, and HTTP channels.

import numpy as np

channel_checks = {
    "Email": ("email_active_days", "email_count_p95"),
    "USB": ("usb_active_days", "usb_copy_p95"),
    "HTTP": ("http_upload_active_days", "http_upload_p95")
}

validation_results = []

for channel, (active_col, p95_col) in channel_checks.items():
    active = user_baseline[active_col].gt(0)
    p95 = user_baseline[p95_col]

    valid_p95 = p95.notna() & np.isfinite(p95) & p95.ge(0)

    validation_results.append({
        "channel": channel,
        "active_users": int(active.sum()),
        "valid_active_p95": int((active & valid_p95).sum()),
        "invalid_active_p95": int((active & ~valid_p95).sum()),
        "inactive_with_p95": int((~active & p95.notna()).sum())
    })

p95_validation = pd.DataFrame(validation_results)

print("BASELINE P95 VALIDATION")
display(p95_validation)

assert (p95_validation["invalid_active_p95"] == 0).all(), (
    "Some historically active channels have invalid P95 values."
)

assert (p95_validation["inactive_with_p95"] == 0).all(), (
    "Unexpected P95 values found for inactive channels."
)

print("\nBaseline P95 validation: PASSED")


BASELINE P95 VALIDATION


,channel,active_users,valid_active_p95,invalid_active_p95,inactive_with_p95
0,Email,3994,3994,0,0
1,USB,783,783,0,0
2,HTTP,615,615,0,0



Baseline P95 validation: PASSED



## 3. Eligible Attack Dates

### 3.1 Identify Eligible Attack Workdays

**Purpose:** Identify eligible attack workdays between January 1 and June 1, 2011, using the approved conditions: non-weekend days and at least one user logon. These dates will determine each user's available attack scheduling opportunities.


In [11]:

# Purpose: Identify eligible attack workdays during the approved
# injection period without modifying the original User-Day dataset.

ATTACK_START = pd.Timestamp("2011-01-01")
ATTACK_END = pd.Timestamp("2011-06-01")

attack_dates = pd.to_datetime(user_day["day"])

attack_period = user_day.loc[
    attack_dates.between(
        ATTACK_START, ATTACK_END, inclusive="both"
    )
].copy()

attack_period["day"] = pd.to_datetime(attack_period["day"])

attack_period["eligible_workday"] = (
    attack_period["is_weekend"].eq(0)
    & attack_period["logon_count"].gt(0)
)

eligible_attack_days = attack_period.loc[
    attack_period["eligible_workday"]
].copy()

print("ATTACK PERIOD ELIGIBILITY SUMMARY")
print("Attack start:", ATTACK_START.date())
print("Attack end:", ATTACK_END.date())
print("Total attack-period records:", f"{len(attack_period):,}")
print("Eligible workday records:", f"{len(eligible_attack_days):,}")
print(
    "Ineligible records:",
    f"{len(attack_period) - len(eligible_attack_days):,}"
)
print(
    "Users with at least one eligible workday:",
    f"{eligible_attack_days['user'].nunique():,}"
)

print("\nELIGIBLE WORKDAYS PER USER")
eligible_days_per_user = (
    eligible_attack_days.groupby("user")
    .size()
    .rename("eligible_workdays")
    .reset_index()
)

display(
    eligible_days_per_user["eligible_workdays"]
    .describe()
    .to_frame(name="value")
)

print("\nFull eligible dataset dimensions:", eligible_attack_days.shape)
print("Preview: First 15 rows only (not the complete dataset)")
display(
    eligible_attack_days[
        ["user", "day", "is_weekend", "logon_count", "eligible_workday"]
    ].head(15)
)

assert eligible_attack_days["eligible_workday"].all()
assert eligible_attack_days["day"].between(
    ATTACK_START, ATTACK_END
).all()

print("\nEligible attack workday identification: PASSED")


ATTACK PERIOD ELIGIBILITY SUMMARY
Attack start: 2011-01-01
Attack end: 2011-06-01
Total attack-period records: 395,562
Eligible workday records: 387,437
Ineligible records: 8,125
Users with at least one eligible workday: 3,740

ELIGIBLE WORKDAYS PER USER


,value
count,3740.000000
mean,103.592781
std,10.031760
min,1.000000
25%,105.000000
50%,105.000000
75%,105.000000
max,108.000000



Full eligible dataset dimensions: (387437, 35)
Preview: First 15 rows only (not the complete dataset)


,user,day,is_weekend,logon_count,eligible_workday
252,AAB0162,2011-01-04,0,1.0,True
253,AAB0162,2011-01-05,0,1.0,True
254,AAB0162,2011-01-06,0,1.0,True
255,AAB0162,2011-01-07,0,1.0,True
256,AAB0162,2011-01-10,0,1.0,True
257,AAB0162,2011-01-11,0,1.0,True
258,AAB0162,2011-01-12,0,1.0,True
259,AAB0162,2011-01-13,0,1.0,True
260,AAB0162,2011-01-14,0,1.0,True
261,AAB0162,2011-01-17,0,1.0,True



Eligible attack workday identification: PASSED



### 3.2 Attack Duration Eligibility

**Purpose:** Evaluate user eligibility for each approved attack duration (5, 10, 20, 30, 40, and 50 eligible workdays), combining future workday availability with historical multi-channel eligibility.


In [12]:

# Purpose: Evaluate attack-duration feasibility for users with
# at least two historically active exfiltration channels.

ATTACK_DURATIONS = (5, 10, 20, 30, 40, 50)

user_duration_eligibility = channel_eligibility.merge(
    eligible_days_per_user,
    on="user",
    how="left",
    validate="one_to_one"
)

user_duration_eligibility["eligible_workdays"] = (
    user_duration_eligibility["eligible_workdays"]
    .fillna(0)
    .astype(int)
)

user_duration_eligibility["multi_channel_eligible"] = (
    user_duration_eligibility["eligible_channel_count"] >= 2
)

duration_results = []

for duration in ATTACK_DURATIONS:
    duration_col = f"eligible_{duration}d"

    user_duration_eligibility[duration_col] = (
        user_duration_eligibility["multi_channel_eligible"]
        & user_duration_eligibility["eligible_workdays"].ge(duration)
    )

    eligible = user_duration_eligibility[duration_col]

    duration_results.append({
        "duration": duration,
        "eligible_users": int(eligible.sum()),
        "two_channel_users": int((
            eligible
            & user_duration_eligibility["eligible_channel_count"].eq(2)
        ).sum()),
        "three_channel_users": int((
            eligible
            & user_duration_eligibility["eligible_channel_count"].eq(3)
        ).sum()),
        "remaining_to_1000": max(0, 1000 - int(eligible.sum()))
    })

duration_eligibility_summary = pd.DataFrame(duration_results)

print("ATTACK DURATION ELIGIBILITY SUMMARY")
display(duration_eligibility_summary)

print("\nUSER ELIGIBILITY OVERVIEW")
print("Total baseline users:", f"{len(user_duration_eligibility):,}")
print(
    "Users with 2+ historical channels:",
    f"{user_duration_eligibility['multi_channel_eligible'].sum():,}"
)

print("\nFull eligibility dataset dimensions:",
      user_duration_eligibility.shape)

print("Preview: First 15 rows only (not the complete dataset)")
display(
    user_duration_eligibility[
        ["user", "eligible_channels", "eligible_channel_count",
         "eligible_workdays"] +
        [f"eligible_{d}d" for d in ATTACK_DURATIONS]
    ].head(15)
)

assert len(user_duration_eligibility) == len(user_baseline)
assert user_duration_eligibility["user"].is_unique

print("\nAttack duration eligibility assessment: PASSED")


ATTACK DURATION ELIGIBILITY SUMMARY


,duration,eligible_users,two_channel_users,three_channel_users,remaining_to_1000
0,5,995,702,293,5
1,10,993,700,293,7
2,20,992,700,292,8
3,30,988,697,291,12
4,40,983,694,289,17
5,50,982,693,289,18



USER ELIGIBILITY OVERVIEW
Total baseline users: 3,995
Users with 2+ historical channels: 1,079

Full eligibility dataset dimensions: (3995, 17)
Preview: First 15 rows only (not the complete dataset)


,user,eligible_channels,eligible_channel_count,eligible_workdays,eligible_5d,eligible_10d,eligible_20d,eligible_30d,eligible_40d,eligible_50d
0,AAB0162,Email,1,104,False,False,False,False,False,False
1,AAB0398,Email,1,105,False,False,False,False,False,False
2,AAC0610,Email+USB,2,105,True,True,True,True,True,True
3,AAC0668,Email,1,105,False,False,False,False,False,False
4,AAC3270,Email,1,105,False,False,False,False,False,False
5,AAD2188,Email,1,105,False,False,False,False,False,False
6,AAD3030,Email,1,105,False,False,False,False,False,False
7,AAF0819,Email+USB,2,105,True,True,True,True,True,True
8,AAF3937,Email,1,105,False,False,False,False,False,False
9,AAG1447,Email,1,105,False,False,False,False,False,False



Attack duration eligibility assessment: PASSED



### 3.3 Investigate Ineligible Multi-Channel Users

**Purpose:** Investigate why historically multi-channel users fail the minimum five-day attack duration requirement, distinguishing users with no eligible attack workdays from users with insufficient eligible workdays.


In [13]:

# Purpose: Investigate historically multi-channel users who cannot
# support the minimum five-day attack duration.

multi_channel_users = user_duration_eligibility.loc[
    user_duration_eligibility["multi_channel_eligible"]
].copy()

ineligible_min_duration = multi_channel_users.loc[
    ~multi_channel_users["eligible_5d"]
].copy()

ineligible_min_duration["ineligibility_reason"] = (
    ineligible_min_duration["eligible_workdays"].apply(
        lambda days: (
            "No eligible attack workdays"
            if days == 0
            else "Fewer than 5 eligible workdays"
        )
    )
)

print("MINIMUM-DURATION INELIGIBILITY ANALYSIS")
print("Historically multi-channel users:", len(multi_channel_users))
print("Eligible for at least 5 days:",
      int(multi_channel_users["eligible_5d"].sum()))
print("Ineligible for 5 days:", len(ineligible_min_duration))

print("\nINELIGIBILITY REASONS")
display(
    ineligible_min_duration["ineligibility_reason"]
    .value_counts()
    .rename_axis("reason")
    .reset_index(name="users")
)

print("\nELIGIBLE WORKDAY DISTRIBUTION AMONG INELIGIBLE USERS")
display(
    ineligible_min_duration["eligible_workdays"]
    .value_counts()
    .sort_index()
    .rename_axis("eligible_workdays")
    .reset_index(name="users")
)

print("\nCHANNEL COMBINATIONS OF INELIGIBLE USERS")
display(
    ineligible_min_duration["eligible_channels"]
    .value_counts()
    .rename_axis("channel_combination")
    .reset_index(name="users")
)

assert len(ineligible_min_duration) == 84

print("\nMinimum-duration ineligibility investigation: PASSED")


MINIMUM-DURATION INELIGIBILITY ANALYSIS
Historically multi-channel users: 1079
Eligible for at least 5 days: 995
Ineligible for 5 days: 84

INELIGIBILITY REASONS


,reason,users
0,No eligible attack workdays,84



ELIGIBLE WORKDAY DISTRIBUTION AMONG INELIGIBLE USERS


,eligible_workdays,users
0,0,84



CHANNEL COMBINATIONS OF INELIGIBLE USERS


,channel_combination,users
0,Email+USB,36
1,Email+USB+HTTP,26
2,Email+HTTP,22



Minimum-duration ineligibility investigation: PASSED



يعني 84 شخص مؤهل لا يوجد لديهم 5 ايام عمل مستمرة؟؟
مو بالضبط. فيه فرق مهم بين 5 أيام عمل متتالية و5 أيام عمل مؤهلة.


الـ84 مستخدمًا:
- مؤهلون من ناحية النشاط التاريخي؛ لأن لديهم قناتين أو أكثر.
- لكن ليس لديهم أي يوم عمل مؤهل (0 أيام) خلال الفترة من 1 يناير إلى 1 يونيو 2011.
ونحن حتى الآن لم نشترط أن تكون أيام العمل الخمسة متتالية.

مثلًا، لو كان لدى مستخدم 5 أيام عمل مؤهلة موزعة على أسبوعين، فهو مؤهل مبدئيًا لهجوم مدته 5 أيام.

الخلاصة: الـ84 مستخدمًا ليسوا مستبعدين بسبب عدم توفر 5 أيام متتالية، بل لأن عدد أيام العمل التي تحقق شروطنا يساوي صفرًا.

والاختبار القادم سيكشف هل السبب عدم وجود سجلات لهم خلال فترة الهجوم، أم وجود سجلات لا تحقق شروط أيام العمل وتسجيل الدخول.



### 3.4 Investigate Attack-Period Availability

**Purpose:** Determine whether historically multi-channel users with zero eligible attack workdays are absent from the attack-period dataset or have records that fail the approved workday and logon conditions.


In [14]:

# Purpose: Investigate why 84 historically multi-channel users
# have zero eligible attack workdays during the injection period.

ineligible_users = set(ineligible_min_duration["user"])

ineligible_period_records = attack_period.loc[
    attack_period["user"].isin(ineligible_users)
].copy()

period_activity = (
    ineligible_period_records.groupby("user")
    .agg(
        attack_period_records=("day", "size"),
        non_weekend_records=(
            "is_weekend", lambda x: int(x.eq(0).sum())
        ),
        logon_records=(
            "logon_count", lambda x: int(x.gt(0).sum())
        ),
        eligible_records=(
            "eligible_workday", lambda x: int(x.sum())
        )
    )
)

availability_audit = ineligible_min_duration[
    ["user", "eligible_channels"]
].merge(
    period_activity,
    on="user",
    how="left",
    validate="one_to_one"
)

count_columns = [
    "attack_period_records",
    "non_weekend_records",
    "logon_records",
    "eligible_records"
]

availability_audit[count_columns] = (
    availability_audit[count_columns].fillna(0).astype(int)
)

availability_audit["availability_reason"] = (
    availability_audit["attack_period_records"]
    .eq(0)
    .map({
        True: "No attack-period records",
        False: "Records exist but no eligible workdays"
    })
)

print("ATTACK-PERIOD AVAILABILITY AUDIT")
print("Users investigated:", len(availability_audit))

print("\nAVAILABILITY REASONS")
display(
    availability_audit["availability_reason"]
    .value_counts()
    .rename_axis("reason")
    .reset_index(name="users")
)

print("\nRECORD AVAILABILITY SUMMARY")
display(availability_audit[count_columns].describe())

print("\nFull audit dimensions:", availability_audit.shape)
print("Preview: First 15 rows only (not the complete dataset)")
display(availability_audit.head(15))

assert len(availability_audit) == 84
assert (availability_audit["eligible_records"] == 0).all()

print("\nAttack-period availability investigation: PASSED")


ATTACK-PERIOD AVAILABILITY AUDIT
Users investigated: 84

AVAILABILITY REASONS


,reason,users
0,No attack-period records,83
1,Records exist but no eligible workdays,1



RECORD AVAILABILITY SUMMARY


,attack_period_records,non_weekend_records,logon_records,eligible_records
count,84.000000,84.0,84.000000,84.0
mean,0.011905,0.0,0.011905,0.0
std,0.109109,0.0,0.109109,0.0
min,0.000000,0.0,0.000000,0.0
25%,0.000000,0.0,0.000000,0.0
50%,0.000000,0.0,0.000000,0.0
75%,0.000000,0.0,0.000000,0.0
max,1.000000,0.0,1.000000,0.0



Full audit dimensions: (84, 7)
Preview: First 15 rows only (not the complete dataset)


,user,eligible_channels,attack_period_records,non_weekend_records,logon_records,eligible_records,availability_reason
0,ABM3687,Email+USB,0,0,0,0,No attack-period records
1,ACF1806,Email+USB,0,0,0,0,No attack-period records
2,ADB1105,Email+USB+HTTP,0,0,0,0,No attack-period records
3,ADS2950,Email+USB+HTTP,0,0,0,0,No attack-period records
4,AJG0240,Email+USB+HTTP,0,0,0,0,No attack-period records
5,AKB3530,Email+USB,0,0,0,0,No attack-period records
6,AKS3496,Email+USB+HTTP,0,0,0,0,No attack-period records
7,BAO2928,Email+USB+HTTP,0,0,0,0,No attack-period records
8,BCC2536,Email+USB+HTTP,0,0,0,0,No attack-period records
9,BMN0178,Email+USB+HTTP,0,0,0,0,No attack-period records



Attack-period availability investigation: PASSED



### 3.5 Combined User Eligibility Assessment

**Purpose:** Evaluate user eligibility by combining historical exfiltration-channel activity, attack-period workday availability, and approved attack durations. Exclude known malicious users and distinguish preliminary channel candidates from users who satisfy the minimum attack-duration requirements.


In [15]:

# Purpose: Evaluate user eligibility using historical channel
# availability, attack-period workdays, duration requirements,
# and ground-truth exclusions in one consolidated assessment.

ATTACK_DURATIONS = (5, 10, 20, 30, 40, 50)

# 1. Identify historically active exfiltration channels.
combined_eligibility = user_baseline[
    [
        "user",
        "email_active_days",
        "usb_active_days",
        "http_upload_active_days"
    ]
].copy()

channel_mapping = {
    "Email": "email_active_days",
    "USB": "usb_active_days",
    "HTTP": "http_upload_active_days"
}

for channel, column in channel_mapping.items():
    combined_eligibility[f"{channel.lower()}_eligible"] = (
        combined_eligibility[column].gt(0)
    )

channel_flags = [
    "email_eligible",
    "usb_eligible",
    "http_eligible"
]

combined_eligibility["historical_channel_count"] = (
    combined_eligibility[channel_flags].sum(axis=1)
)

combined_eligibility["historical_channels"] = (
    combined_eligibility[channel_flags].apply(
        lambda row: "+".join(
            channel
            for channel, active in zip(
                ["Email", "USB", "HTTP"], row
            )
            if active
        ) or "None",
        axis=1
    )
)

# 2. Count eligible workdays within the approved attack period.
attack_window = user_day.loc[
    pd.to_datetime(user_day["day"]).between(
        "2011-01-01", "2011-06-01", inclusive="both"
    )
].copy()

attack_window["eligible_workday"] = (
    attack_window["is_weekend"].eq(0)
    & attack_window["logon_count"].gt(0)
)

workday_counts = (
    attack_window.loc[attack_window["eligible_workday"]]
    .groupby("user")
    .size()
    .rename("eligible_workdays")
)

period_counts = (
    attack_window.groupby("user")
    .size()
    .rename("attack_period_records")
)

combined_eligibility = (
    combined_eligibility
    .merge(period_counts, on="user", how="left")
    .merge(workday_counts, on="user", how="left")
)

combined_eligibility[
    ["attack_period_records", "eligible_workdays"]
] = combined_eligibility[
    ["attack_period_records", "eligible_workdays"]
].fillna(0).astype(int)

# 3. Apply known malicious-user exclusions.
combined_eligibility["is_known_malicious"] = (
    combined_eligibility["user"].isin(ground_truth_users)
)

# 4. Evaluate combined eligibility for each duration.
combined_eligibility["historical_channel_candidate"] = (
    combined_eligibility["historical_channel_count"].ge(2)
    & ~combined_eligibility["is_known_malicious"]
)

combined_eligibility["present_in_attack_period"] = (
    combined_eligibility["attack_period_records"].gt(0)
)

for duration in ATTACK_DURATIONS:
    combined_eligibility[f"eligible_{duration}d"] = (
        combined_eligibility["historical_channel_candidate"]
        & combined_eligibility["eligible_workdays"].ge(duration)
    )

# 5. Summarize eligibility across the complete user population.
summary = {
    "Total baseline users": len(combined_eligibility),
    "Historical multi-channel candidates": int(
        combined_eligibility["historical_channel_candidate"].sum()
    ),
    "Multi-channel candidates present in attack period": int((
        combined_eligibility["historical_channel_candidate"]
        & combined_eligibility["present_in_attack_period"]
    ).sum()),
    "Multi-channel candidates with eligible workdays": int((
        combined_eligibility["historical_channel_candidate"]
        & combined_eligibility["eligible_workdays"].gt(0)
    ).sum()),
    "Eligible for at least 5 days": int(
        combined_eligibility["eligible_5d"].sum()
    )
}

print("COMBINED USER ELIGIBILITY SUMMARY")
display(
    pd.DataFrame(
        summary.items(),
        columns=["criterion", "users"]
    )
)

duration_summary = pd.DataFrame([
    {
        "duration": duration,
        "eligible_users": int(
            combined_eligibility[f"eligible_{duration}d"].sum()
        ),
        "two_channel_users": int((
            combined_eligibility[f"eligible_{duration}d"]
            & combined_eligibility["historical_channel_count"].eq(2)
        ).sum()),
        "three_channel_users": int((
            combined_eligibility[f"eligible_{duration}d"]
            & combined_eligibility["historical_channel_count"].eq(3)
        ).sum())
    }
    for duration in ATTACK_DURATIONS
])

print("\nCOMBINED DURATION ELIGIBILITY")
display(duration_summary)

print("\nVALIDATION CHECKS")
assert combined_eligibility["user"].is_unique
assert len(combined_eligibility) == len(user_baseline)

assert (
    combined_eligibility.loc[
        combined_eligibility["is_known_malicious"],
        "eligible_5d"
    ].sum() == 0
)

assert (
    combined_eligibility["eligible_5d"].sum()
    == user_duration_eligibility["eligible_5d"].sum()
), "Combined eligibility differs from previous assessment."

print("\nCombined user eligibility assessment: PASSED")


COMBINED USER ELIGIBILITY SUMMARY


,criterion,users
0,Total baseline users,3995
1,Historical multi-channel candidates,1079
2,Multi-channel candidates present in attack period,996
3,Multi-channel candidates with eligible workdays,995
4,Eligible for at least 5 days,995



COMBINED DURATION ELIGIBILITY


,duration,eligible_users,two_channel_users,three_channel_users
0,5,995,702,293
1,10,993,700,293
2,20,992,700,292
3,30,988,697,291
4,40,983,694,289
5,50,982,693,289



VALIDATION CHECKS

Combined user eligibility assessment: PASSED



### 3.6 Eligible Users & Supported Attack Durations

**Purpose:** Create a complete user-level eligibility table containing all users who satisfy the historical multi-channel and minimum attack-duration requirements. For each user, report the historically active channels, available attack workdays, and all supported attack durations.


In [16]:

# Purpose: Generate a complete 995-user eligibility table showing
# historical exfiltration channels and supported attack durations.

ATTACK_DURATIONS = (5, 10, 20, 30, 40, 50)

# Select users satisfying the minimum five-day requirement.
eligible_users_995 = combined_eligibility.loc[
    combined_eligibility["eligible_5d"]
].copy()

# Build a readable list of supported attack durations per user.
eligible_users_995["supported_durations"] = (
    eligible_users_995.apply(
        lambda row: ", ".join(
            f"{duration}d"
            for duration in ATTACK_DURATIONS
            if row[f"eligible_{duration}d"]
        ),
        axis=1
    )
)

# Identify the longest supported attack duration.
eligible_users_995["max_supported_duration"] = (
    eligible_users_995.apply(
        lambda row: max(
            duration
            for duration in ATTACK_DURATIONS
            if row[f"eligible_{duration}d"]
        ),
        axis=1
    )
)

# Prepare the complete eligibility table.
eligible_users_table = (
    eligible_users_995[
        [
            "user",
            "historical_channels",
            "historical_channel_count",
            "attack_period_records",
            "eligible_workdays",
            "supported_durations",
            "max_supported_duration"
        ]
    ]
    .rename(columns={
        "user": "user_id",
        "historical_channels": "eligible_channels",
        "historical_channel_count": "channel_count"
    })
    .sort_values(
        ["channel_count", "eligible_workdays", "user_id"],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)

print("COMPLETE ELIGIBLE USER TABLE")
print("Full table dimensions:", eligible_users_table.shape)
print("Unique users:", eligible_users_table["user_id"].nunique())

# Display the complete table, not a 15-row preview.
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None
):
    display(eligible_users_table)

# Validate the final table.
assert len(eligible_users_table) == 995
assert eligible_users_table["user_id"].is_unique
assert eligible_users_table["channel_count"].ge(2).all()
assert eligible_users_table["eligible_workdays"].ge(5).all()
assert eligible_users_table["supported_durations"].ne("").all()

print("\nComplete eligible user table: PASSED")


COMPLETE ELIGIBLE USER TABLE
Full table dimensions: (995, 7)
Unique users: 995


,user_id,eligible_channels,channel_count,attack_period_records,eligible_workdays,supported_durations,max_supported_duration
0,ACB0701,Email+USB+HTTP,3,111,107,"5d, 10d, 20d, 30d, 40d, 50d",50
1,AJS3021,Email+USB+HTTP,3,137,107,"5d, 10d, 20d, 30d, 40d, 50d",50
2,ALB3649,Email+USB+HTTP,3,111,107,"5d, 10d, 20d, 30d, 40d, 50d",50
3,ARZ3464,Email+USB+HTTP,3,115,107,"5d, 10d, 20d, 30d, 40d, 50d",50
4,BMC3636,Email+USB+HTTP,3,151,107,"5d, 10d, 20d, 30d, 40d, 50d",50
5,BYS3626,Email+USB+HTTP,3,116,107,"5d, 10d, 20d, 30d, 40d, 50d",50
6,CEC1352,Email+USB+HTTP,3,111,107,"5d, 10d, 20d, 30d, 40d, 50d",50
7,GDM3960,Email+USB+HTTP,3,151,107,"5d, 10d, 20d, 30d, 40d, 50d",50
8,GNT0221,Email+USB+HTTP,3,115,107,"5d, 10d, 20d, 30d, 40d, 50d",50
9,HAG3196,Email+USB+HTTP,3,130,107,"5d, 10d, 20d, 30d, 40d, 50d",50



Complete eligible user table: PASSED


In [21]:

# Purpose: Load and inspect the previous engineer's attack-host
# eligibility dataset to examine its channel and duration criteria.

from pathlib import Path
import pandas as pd

engineer_file = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
    r"\Data_Engineering\Data_engineering\data\features"
    r"\attack_host_eligibility_r62.parquet"
)

assert engineer_file.is_file(), (
    f"File not found: {engineer_file}"
)

engineer_eligibility = pd.read_parquet(engineer_file)

print("ENGINEER ATTACK-HOST ELIGIBILITY DATASET")
print("Source file:", engineer_file)
print("Full dataset dimensions:", engineer_eligibility.shape)

print("\nCOLUMN NAMES AND DATA TYPES")
column_types = pd.DataFrame({
    "column": engineer_eligibility.columns,
    "dtype": [str(dtype) for dtype in engineer_eligibility.dtypes]
})
display(column_types)

print("\nMISSING VALUES")
missing_values = pd.DataFrame({
    "column": engineer_eligibility.columns,
    "missing_count": engineer_eligibility.isna().sum().values
})
display(missing_values)

print("\nPreview: First 15 rows only (not the complete dataset)")
display(engineer_eligibility.head(15))

print("\nDataset inspection: COMPLETED")


ENGINEER ATTACK-HOST ELIGIBILITY DATASET
Source file: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Data_Engineering\Data_engineering\data\features\attack_host_eligibility_r62.parquet
Full dataset dimensions: (3995, 9)

COLUMN NAMES AND DATA TYPES


,column,dtype
0,user,str
1,baseline_workdays,int64
2,email_active_days,int64
3,usb_active_days,int64
4,http_upload_active_days,int64
5,future_workdays,int64
6,eligible_30,int32
7,eligible_40,int32
8,eligible_strict_50,int32



MISSING VALUES


,column,missing_count
0,user,0
1,baseline_workdays,0
2,email_active_days,0
3,usb_active_days,0
4,http_upload_active_days,0
5,future_workdays,0
6,eligible_30,0
7,eligible_40,0
8,eligible_strict_50,0



Preview: First 15 rows only (not the complete dataset)


,user,baseline_workdays,email_active_days,usb_active_days,http_upload_active_days,future_workdays,eligible_30,eligible_40,eligible_strict_50
0,AAB0162,251,153,0,0,104,0,0,0
1,AAB0398,251,199,0,0,105,0,0,0
2,AAC0610,251,97,133,0,105,0,0,0
3,AAC0668,251,173,0,0,105,0,0,0
4,AAC3270,251,17,0,0,105,0,0,0
5,AAD2188,251,32,0,0,105,0,0,0
6,AAD3030,251,168,0,0,105,0,0,0
7,AAF0819,251,155,203,0,105,0,0,0
8,AAF3937,251,38,0,0,105,0,0,0
9,AAG1447,251,181,0,0,105,0,0,0



Dataset inspection: COMPLETED



### 3.7 Verify Previous Engineer's Eligibility Rules

**Purpose:** Compare the previous engineer's 30-, 40-, and 50-day eligibility flags against historical three-channel activity and attack-period workday availability to determine whether the dataset combines baseline and future-period requirements.



### 3.8 Engineer Eligibility vs. Attack-Period Eligibility

**Purpose:** Verify whether users marked eligible by the previous engineer have sufficient eligible workdays strictly within the attack period (2011-01-01 to 2011-06-01), excluding all baseline dates, weekends, and days without logon activity.



### 3.9 Excluded Users vs. Engineer Eligibility

**Purpose:** Cross-check the 84 historically multi-channel users excluded from attack simulation due to zero eligible attack-period workdays against the previous engineer's 30-, 40-, and 50-day eligibility flags. Confirm that no excluded user was incorrectly selected for attack injection.


In [24]:

# Purpose: Identify the 84 historically multi-channel users with
# zero eligible attack-period workdays and check whether the
# previous engineer classified any of them as attack-eligible.

# Reconstruct the excluded cohort using verified eligibility results.
excluded_84 = combined_eligibility.loc[
    combined_eligibility["historical_channel_candidate"].eq(True)
    & combined_eligibility["eligible_workdays"].eq(0),
    ["user", "eligible_workdays"]
].copy()

assert len(excluded_84) == 84, (
    f"Expected 84 excluded users, found {len(excluded_84)}"
)

# Compare excluded users with the engineer's eligibility flags.
excluded_engineer_comparison = excluded_84.merge(
    engineer_eligibility[
        [
            "user",
            "email_active_days",
            "usb_active_days",
            "http_upload_active_days",
            "future_workdays",
            "eligible_30",
            "eligible_40",
            "eligible_strict_50"
        ]
    ],
    on="user",
    how="left",
    validate="one_to_one"
)

flags = ["eligible_30", "eligible_40", "eligible_strict_50"]

excluded_engineer_comparison["selected_by_engineer"] = (
    excluded_engineer_comparison[flags].eq(1).any(axis=1)
)

print("EXCLUDED USERS CROSS-CHECK")
print("Historically multi-channel excluded users:", len(excluded_84))
print(
    "Excluded users selected by engineer:",
    int(excluded_engineer_comparison["selected_by_engineer"].sum())
)
print(
    "Excluded users not selected by engineer:",
    int((~excluded_engineer_comparison["selected_by_engineer"]).sum())
)

print("\nELIGIBILITY FLAGS AMONG EXCLUDED USERS")
display(
    excluded_engineer_comparison[flags]
    .eq(1)
    .sum()
    .rename("selected_users")
    .to_frame()
)

print("\nEXCLUDED USERS INCORRECTLY SELECTED")
incorrectly_selected = excluded_engineer_comparison.loc[
    excluded_engineer_comparison["selected_by_engineer"]
].copy()

print("Full matching records:", len(incorrectly_selected))
display(incorrectly_selected)

print("\nCROSS-CHECK RESULT")
if incorrectly_selected.empty:
    print("PASSED: None of the 84 excluded users was selected.")
else:
    print("FAILED: Some excluded users were selected by the engineer.")


EXCLUDED USERS CROSS-CHECK
Historically multi-channel excluded users: 84
Excluded users selected by engineer: 0
Excluded users not selected by engineer: 84

ELIGIBILITY FLAGS AMONG EXCLUDED USERS


,selected_users
eligible_30,0
eligible_40,0
eligible_strict_50,0



EXCLUDED USERS INCORRECTLY SELECTED
Full matching records: 0


,user,eligible_workdays,email_active_days,usb_active_days,http_upload_active_days,future_workdays,eligible_30,eligible_40,eligible_strict_50,selected_by_engineer



CROSS-CHECK RESULT
PASSED: None of the 84 excluded users was selected.



## 4. Duration & Channel Feasibility

### 4.1 Scenario Channel Requirements Audit

**Purpose:** Audit the 1,143 approved attack scenarios to determine the minimum number of historically active exfiltration channels required for each scenario.

Scenarios containing three-channel activity days require three historically active channels. All other approved channel-rotating scenarios require at least two channels.

This step validates scenario requirements before matching scenarios to eligible users. No user assignment or attack injection is performed.


In [25]:

# Purpose: Determine and validate the minimum historical channel
# requirements for all approved attack scenarios.

import pandas as pd

scenario_channel_requirements = approved_scenarios.copy()

# Derive the minimum number of historically active channels.
scenario_channel_requirements["required_channel_count"] = (
    scenario_channel_requirements["three_channel_days"]
    .gt(0)
    .map({True: 3, False: 2})
)

# Validate the daily activity structure.
scenario_channel_requirements["calculated_active_days"] = (
    scenario_channel_requirements["single_channel_days"]
    + scenario_channel_requirements["two_channel_days"]
    + scenario_channel_requirements["three_channel_days"]
)

# Validate the approved scenario rules.
assert len(scenario_channel_requirements) == 1143

assert (
    scenario_channel_requirements["calculated_active_days"]
    == scenario_channel_requirements["active_days"]
).all(), "Scenario activity-day totals do not match."

assert (
    scenario_channel_requirements["active_days"]
    + scenario_channel_requirements["off_days"]
    == scenario_channel_requirements["duration"]
).all(), "Scenario duration totals do not match."

assert (
    scenario_channel_requirements["target_switches"] > 0
).all(), "A channel-rotating scenario has no switches."

assert (
    scenario_channel_requirements.loc[
        scenario_channel_requirements["duration"].isin([5, 10, 20, 30]),
        "required_channel_count"
    ] == 2
).all(), "Unexpected three-channel requirement in shorter scenarios."

assert (
    scenario_channel_requirements.loc[
        scenario_channel_requirements["duration"].isin([40, 50]),
        "required_channel_count"
    ] == 3
).all(), "A 40/50-day scenario does not require three channels."

# Summarize requirements by duration.
scenario_channel_summary = (
    scenario_channel_requirements
    .groupby(["duration", "required_channel_count"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[2, 3], fill_value=0)
    .rename(columns={
        2: "two_channel_scenarios",
        3: "three_channel_scenarios"
    })
    .reset_index()
)

scenario_channel_summary["total_scenarios"] = (
    scenario_channel_summary["two_channel_scenarios"]
    + scenario_channel_summary["three_channel_scenarios"]
)

print("SCENARIO CHANNEL REQUIREMENTS AUDIT")
print("Total approved scenarios:", len(scenario_channel_requirements))

display(scenario_channel_summary)

print("\nOVERALL CHANNEL REQUIREMENTS")
display(
    scenario_channel_requirements["required_channel_count"]
    .value_counts()
    .sort_index()
    .rename_axis("required_channels")
    .reset_index(name="scenario_count")
)

print("\nScenario channel requirements audit: PASSED")


SCENARIO CHANNEL REQUIREMENTS AUDIT
Total approved scenarios: 1143


required_channel_count,duration,two_channel_scenarios,three_channel_scenarios,total_scenarios
0,5,9,0,9
1,10,24,0,24
2,20,69,0,69
3,30,99,0,99
4,40,0,258,258
5,50,0,684,684



OVERALL CHANNEL REQUIREMENTS


,required_channels,scenario_count
0,2,201
1,3,942



Scenario channel requirements audit: PASSED


باااااااااااااعووووووووووص


### 4.2 User–Scenario Compatibility

**Purpose:** Evaluate user compatibility with approved attack scenarios using both historical channel availability and eligible attack-period workdays.

Two-channel scenarios require at least two historically active channels, while three-channel scenarios require all three. Each user must also have sufficient eligible workdays within the attack period.

This step measures the available user pool for each duration without assigning scenarios or injecting attacks.


In [26]:

# Purpose: Calculate the number of compatible users for each
# approved scenario duration and historical channel requirement.

import pandas as pd

# Start with the independently verified 995-user cohort.
candidate_users = combined_eligibility.loc[
    combined_eligibility["eligible_5d"].eq(True)
].copy()

assert len(candidate_users) == 995
assert candidate_users["user"].is_unique

compatibility_results = []

for _, scenario_group in scenario_channel_summary.iterrows():

    duration = int(scenario_group["duration"])

    for required_channels, scenario_count_column in [
        (2, "two_channel_scenarios"),
        (3, "three_channel_scenarios")
    ]:

        scenario_count = int(scenario_group[scenario_count_column])

        if scenario_count == 0:
            continue

        compatible = candidate_users.loc[
            candidate_users["eligible_workdays"].ge(duration)
            & candidate_users["historical_channel_count"].ge(
                required_channels
            )
        ]

        compatibility_results.append({
            "duration": duration,
            "required_channels": required_channels,
            "scenario_count": scenario_count,
            "compatible_users": len(compatible),
            "two_channel_users": int(
                compatible["historical_channel_count"].eq(2).sum()
            ),
            "three_channel_users": int(
                compatible["historical_channel_count"].eq(3).sum()
            )
        })

user_scenario_compatibility = pd.DataFrame(
    compatibility_results
).sort_values("duration").reset_index(drop=True)

print("USER–SCENARIO COMPATIBILITY BY DURATION")
display(user_scenario_compatibility)

print("\nVALIDATION")

assert (
    user_scenario_compatibility["scenario_count"].sum()
    == len(scenario_channel_requirements)
)

assert (
    user_scenario_compatibility["compatible_users"]
    == (
        user_scenario_compatibility["two_channel_users"]
        + user_scenario_compatibility["three_channel_users"]
    )
).all()

assert (
    user_scenario_compatibility.loc[
        user_scenario_compatibility["required_channels"].eq(3),
        "two_channel_users"
    ].eq(0)
).all()

print("User–scenario compatibility audit: PASSED")


USER–SCENARIO COMPATIBILITY BY DURATION


,duration,required_channels,scenario_count,compatible_users,two_channel_users,three_channel_users
0,5,2,9,995,702,293
1,10,2,24,993,700,293
2,20,2,69,992,700,292
3,30,2,99,988,697,291
4,40,3,258,289,0,289
5,50,3,684,289,0,289



VALIDATION
User–scenario compatibility audit: PASSED


نفكك الباعوص


### 4.3 Assignment Capacity & Feasibility

**Purpose:** Assess the maximum number of unique users available for attack assignment, the capacity of each scenario duration, and the constraints affecting duration distribution.

This assessment uses the verified attack-period workdays, historical channel eligibility, and approved scenario requirements.

No fixed duration quotas are imposed. Scenario templates may be reused across different users, but each user can receive at most one attack.

This step evaluates assignment capacity only. It does not assign scenarios, schedule attacks, or modify the dataset.


In [27]:

# Purpose: Assess unique-user assignment capacity and duration
# flexibility without assigning scenarios or modifying data.

import pandas as pd

TARGET_ATTACKS = 1000
DURATIONS = (5, 10, 20, 30, 40, 50)

# Use the verified user–scenario compatibility results.
capacity = user_scenario_compatibility[
    [
        "duration",
        "required_channels",
        "scenario_count",
        "compatible_users",
        "two_channel_users",
        "three_channel_users"
    ]
].copy()

capacity = capacity.sort_values("duration").reset_index(drop=True)

# Every compatible user can receive at most one attack.
capacity["max_unique_assignments_at_duration"] = (
    capacity["compatible_users"]
)

# Calculate the available user pool.
eligible_pool = combined_eligibility.loc[
    combined_eligibility["eligible_5d"].eq(True)
].copy()

total_unique_users = eligible_pool["user"].nunique()

# Count users by the longest scenario duration they can support.
scenario_requirements = {
    int(row["duration"]): int(row["required_channels"])
    for _, row in capacity.iterrows()
}

def longest_compatible_duration(row):
    supported = [
        duration
        for duration in DURATIONS
        if (
            row["eligible_workdays"] >= duration
            and row["historical_channel_count"]
            >= scenario_requirements[duration]
        )
    ]
    return max(supported) if supported else None

eligible_pool["max_scenario_duration"] = eligible_pool.apply(
    longest_compatible_duration,
    axis=1
)

duration_flexibility = (
    eligible_pool["max_scenario_duration"]
    .value_counts()
    .reindex(DURATIONS, fill_value=0)
    .sort_index()
    .rename_axis("max_scenario_duration")
    .reset_index(name="unique_users")
)

print("ASSIGNMENT CAPACITY BY DURATION")
display(capacity)

print("\nUSER FLEXIBILITY BY MAXIMUM SUPPORTED DURATION")
display(duration_flexibility)

print("\nOVERALL UNIQUE-USER CAPACITY")
print("Target attacks:", TARGET_ATTACKS)
print("Available unique users:", total_unique_users)
print("Maximum possible unique-user assignments:", total_unique_users)
print("Shortfall from target:", max(0, TARGET_ATTACKS - total_unique_users))

print("\nVALIDATION")
assert total_unique_users == 995
assert eligible_pool["max_scenario_duration"].notna().all()
assert duration_flexibility["unique_users"].sum() == total_unique_users
assert capacity["scenario_count"].sum() == 1143

print("Assignment capacity assessment: PASSED")


ASSIGNMENT CAPACITY BY DURATION


,duration,required_channels,scenario_count,compatible_users,two_channel_users,three_channel_users,max_unique_assignments_at_duration
0,5,2,9,995,702,293,995
1,10,2,24,993,700,293,993
2,20,2,69,992,700,292,992
3,30,2,99,988,697,291,988
4,40,3,258,289,0,289,289
5,50,3,684,289,0,289,289



USER FLEXIBILITY BY MAXIMUM SUPPORTED DURATION


,max_scenario_duration,unique_users
0,5,2
1,10,1
2,20,4
3,30,699
4,40,0
5,50,289



OVERALL UNIQUE-USER CAPACITY
Target attacks: 1000
Available unique users: 995
Maximum possible unique-user assignments: 995
Shortfall from target: 5

VALIDATION
Assignment capacity assessment: PASSED


نسوي بروفايل


## Step 5 — Eligible User Profiling & Feasibility Analysis

### Objective
Analyze the 995 eligible users before designing scenario assignment and reuse rules.

This step examines actual user characteristics, historical channel activity, baseline behavior, attack-period availability, and scenario compatibility to support evidence-based assignment decisions.

### Analysis Plan
1. **Eligible User Overview:** Profile the 995 users and classify them by historical channel availability and supported attack durations.
2. **Historical Channel & Baseline Profiling:** Examine channel activity and P95 baseline values.
3. **Attack-Period Activity Profiling:** Analyze eligible workdays, activity patterns, and temporal gaps.
4. **User–Scenario Compatibility Analysis:** Identify feasible scenarios for each user.
5. **Scenario Reuse Feasibility:** Evaluate scenario coverage, reuse requirements, and assignment constraints.

**Scope:** Exploratory analysis only. No scenario assignment, attack injection, or file export is performed in this step.

---

### 5.1 — Eligible User Overview

**Objective:** Build and validate a profile of all 995 eligible attack hosts.

**Analysis Tasks:**
- Identify all eligible users and verify uniqueness.
- Examine historical channel counts.
- Calculate each user's maximum supported attack duration.
- Classify users into three groups:
  - **Three-channel long-duration:** 289 users eligible for three-channel scenarios lasting 40–50 workdays.
  - **Three-channel short-duration exceptions:** 4 users with three historical channels who will receive two-channel scenarios.
  - **Two-channel short-duration:** 702 users eligible for two-channel scenarios lasting 5–30 workdays.
- Identify and inspect the four short-duration exceptions.
- Validate the group totals against the previously established eligibility results.

**Expected Output:** A validated user profile DataFrame, group-level summaries, duration distributions, and the identities of the four exceptional users.

**Note:** The group counts are based on previous feasibility results and must be verified before proceeding.


In [28]:

# Purpose: Step 5.1 — Profile the 995 eligible users, examine their
# historical channel availability and attack-duration eligibility,
# and validate the three planned assignment groups.
# This cell performs analysis only; no assignment or export.

import pandas as pd

# ---------------------------------------------------------
# 1. Validate the required input columns
# ---------------------------------------------------------

required_columns = [
    "user",
    "historical_channel_count",
    "eligible_workdays",
    "eligible_5d",
    "eligible_10d",
    "eligible_20d",
    "eligible_30d",
    "eligible_40d",
    "eligible_50d",
]

missing_columns = [
    col for col in required_columns
    if col not in combined_eligibility.columns
]

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

# ---------------------------------------------------------
# 2. Build the eligible user profile
# ---------------------------------------------------------

durations = [5, 10, 20, 30, 40, 50]

eligible_user_profile = combined_eligibility.loc[
    (combined_eligibility["historical_channel_count"] >= 2)
    & (combined_eligibility["eligible_workdays"] >= 5),
    required_columns
].copy()

eligible_user_profile["max_supported_duration"] = (
    eligible_user_profile["eligible_workdays"].apply(
        lambda days: max(
            d for d in durations if days >= d
        )
    )
)

# ---------------------------------------------------------
# 3. Classify users according to the approved plan
# ---------------------------------------------------------

three_channel_long = (
    (eligible_user_profile["historical_channel_count"] == 3)
    & (eligible_user_profile["eligible_workdays"] >= 40)
)

three_channel_short = (
    (eligible_user_profile["historical_channel_count"] == 3)
    & (eligible_user_profile["eligible_workdays"] < 40)
)

two_channel_short = (
    eligible_user_profile["historical_channel_count"] == 2
)

eligible_user_profile["assignment_group"] = "Unclassified"

eligible_user_profile.loc[
    three_channel_long, "assignment_group"
] = "Three-channel long-duration"

eligible_user_profile.loc[
    three_channel_short, "assignment_group"
] = "Three-channel short-duration exception"

eligible_user_profile.loc[
    two_channel_short, "assignment_group"
] = "Two-channel short-duration"

# ---------------------------------------------------------
# 4. Validate the population and classification
# ---------------------------------------------------------

assert len(eligible_user_profile) == 995, (
    f"Expected 995 users, found {len(eligible_user_profile)}"
)

assert eligible_user_profile["user"].is_unique, (
    "Duplicate users detected."
)

assert (
    eligible_user_profile["assignment_group"] != "Unclassified"
).all(), "Unclassified users detected."

# Verify duration flags against eligible attack-period workdays.
for duration in durations:
    flag = f"eligible_{duration}d"

    assert (
        eligible_user_profile[flag].astype(bool)
        == (eligible_user_profile["eligible_workdays"] >= duration)
    ).all(), f"Duration eligibility mismatch: {duration} days"

# ---------------------------------------------------------
# 5. Summarize user groups
# ---------------------------------------------------------

group_summary = (
    eligible_user_profile
    .groupby("assignment_group", as_index=False)
    .agg(
        users=("user", "nunique"),
        min_workdays=("eligible_workdays", "min"),
        median_workdays=("eligible_workdays", "median"),
        max_workdays=("eligible_workdays", "max"),
    )
    .sort_values("assignment_group")
)

duration_summary = (
    eligible_user_profile
    .groupby(
        ["assignment_group", "max_supported_duration"],
        as_index=False
    )
    .agg(users=("user", "nunique"))
    .sort_values(
        ["assignment_group", "max_supported_duration"]
    )
)

# ---------------------------------------------------------
# 6. Inspect the four exceptional three-channel users
# ---------------------------------------------------------

exception_users = (
    eligible_user_profile.loc[
        three_channel_short,
        [
            "user",
            "historical_channel_count",
            "eligible_workdays",
            "max_supported_duration",
        ] + [f"eligible_{d}d" for d in durations]
    ]
    .sort_values(["eligible_workdays", "user"])
)

# ---------------------------------------------------------
# 7. Display results
# ---------------------------------------------------------

print("STEP 5.1 — ELIGIBLE USER OVERVIEW")
print("=" * 65)

print("\nPOPULATION VALIDATION")
print("Total eligible users:", len(eligible_user_profile))
print("Unique users:", eligible_user_profile["user"].nunique())

print("\nASSIGNMENT GROUP SUMMARY")
print(group_summary.to_string(index=False))

print("\nMAXIMUM SUPPORTED DURATION BY GROUP")
print(duration_summary.to_string(index=False))

print("\nTHREE-CHANNEL SHORT-DURATION EXCEPTIONS")
print("Exception count:", len(exception_users))
print(exception_users.to_string(index=False))

print("\nEXPECTED GROUP COUNTS")
expected_groups = {
    "Three-channel long-duration": 289,
    "Three-channel short-duration exception": 4,
    "Two-channel short-duration": 702,
}

actual_groups = (
    eligible_user_profile["assignment_group"]
    .value_counts()
    .to_dict()
)

for group, expected in expected_groups.items():
    actual = actual_groups.get(group, 0)
    status = "PASS" if actual == expected else "REVIEW"
    print(f"{group}: {actual} / {expected} [{status}]")

print("\nELIGIBLE USER PROFILE")
print("Full DataFrame shape:", eligible_user_profile.shape)
print("Preview: First 15 rows only (not the complete dataset)")

display(
    eligible_user_profile
    .sort_values(["assignment_group", "user"])
    .head(15)
)


STEP 5.1 — ELIGIBLE USER OVERVIEW

POPULATION VALIDATION
Total eligible users: 995
Unique users: 995

ASSIGNMENT GROUP SUMMARY
                      assignment_group  users  min_workdays  median_workdays  max_workdays
           Three-channel long-duration    289            75            105.0           107
Three-channel short-duration exception      4            15             29.0            30
            Two-channel short-duration    702             7            105.0           108

MAXIMUM SUPPORTED DURATION BY GROUP
                      assignment_group  max_supported_duration  users
           Three-channel long-duration                      50    289
Three-channel short-duration exception                      10      1
Three-channel short-duration exception                      20      1
Three-channel short-duration exception                      30      2
            Two-channel short-duration                       5      2
            Two-channel short-duration              

,user,historical_channel_count,eligible_workdays,eligible_5d,eligible_10d,eligible_20d,eligible_30d,eligible_40d,eligible_50d,max_supported_duration,assignment_group
27,ABD3426,3,106,True,True,True,True,True,True,50,Three-channel long-duration
31,ABH0349,3,105,True,True,True,True,True,True,50,Three-channel long-duration
43,ABO1173,3,105,True,True,True,True,True,True,50,Three-channel long-duration
51,ACB0701,3,107,True,True,True,True,True,True,50,Three-channel long-duration
56,ACG3819,3,106,True,True,True,True,True,True,50,Three-channel long-duration
71,ACV1946,3,105,True,True,True,True,True,True,50,Three-channel long-duration
80,ADG1737,3,105,True,True,True,True,True,True,50,Three-channel long-duration
84,ADM0923,3,105,True,True,True,True,True,True,50,Three-channel long-duration
97,AEB3249,3,105,True,True,True,True,True,True,50,Three-channel long-duration
120,AGF0742,3,106,True,True,True,True,True,True,50,Three-channel long-duration



### 5.2 — Historical Channel & Baseline Profiling

**Objective:** Examine historical channel activity and baseline P95 values for the 995 eligible users to identify potential constraints before scenario assignment.

**Analysis Tasks:**
1. Identify the exact historical channel combinations (Email, USB, and HTTP).
2. Verify that every historically eligible channel has a valid, positive P95 value.
3. Analyze P95 distributions across the three channels.
4. Evaluate the three approved stealth multipliers:
   - High Stealth: 0.5 × P95
   - Medium Stealth: 1.0 × P95
   - Low Stealth: 1.5 × P95
5. Examine the resulting values after applying ceiling, including zero-value and missing-value checks.
6. Compare the results across the three user groups established in Step 5.1.

**Expected Output:**
- Historical channel combination distribution.
- Baseline P95 descriptive statistics.
- Stealth multiplier feasibility summaries.
- Identification of users or channels requiring further investigation.

**Scope:** Exploratory analysis only. No scenario assignment, baseline modification, attack injection, or file export.


In [29]:

# Purpose: Step 5.2 — Audit historical channel combinations, baseline
# P95 values, and stealth multiplier feasibility for all 995 users.
# Analysis only: no scenario assignment, injection, or file export.

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Validate inputs and required columns
# ---------------------------------------------------------

assert "eligible_user_profile" in globals(), (
    "Run Step 5.1 before Step 5.2."
)

baseline_columns = [
    "user",
    "email_active_days",
    "usb_active_days",
    "http_upload_active_days",
    "email_count_p95",
    "usb_copy_p95",
    "http_upload_p95",
]

missing = [
    col for col in baseline_columns
    if col not in user_baseline.columns
]

assert not missing, f"Missing baseline columns: {missing}"
assert user_baseline["user"].is_unique
assert eligible_user_profile["user"].is_unique

# ---------------------------------------------------------
# 2. Merge historical baseline with eligible user profiles
# ---------------------------------------------------------

user_baseline_profile = eligible_user_profile.merge(
    user_baseline[baseline_columns],
    on="user",
    how="left",
    validate="one_to_one",
    indicator=True,
)

assert (user_baseline_profile["_merge"] == "both").all(), (
    "Some eligible users have no baseline record."
)

user_baseline_profile = user_baseline_profile.drop(
    columns="_merge"
)

assert len(user_baseline_profile) == 995

# ---------------------------------------------------------
# 3. Reconstruct historical channel eligibility
# ---------------------------------------------------------

channel_config = {
    "Email": {
        "active_days": "email_active_days",
        "p95": "email_count_p95",
    },
    "USB": {
        "active_days": "usb_active_days",
        "p95": "usb_copy_p95",
    },
    "HTTP": {
        "active_days": "http_upload_active_days",
        "p95": "http_upload_p95",
    },
}

for channel, config in channel_config.items():
    user_baseline_profile[f"{channel.lower()}_active"] = (
        user_baseline_profile[config["active_days"]] > 0
    )

active_columns = [
    "email_active",
    "usb_active",
    "http_active",
]

user_baseline_profile["verified_channel_count"] = (
    user_baseline_profile[active_columns].sum(axis=1)
)

assert (
    user_baseline_profile["verified_channel_count"]
    == user_baseline_profile["historical_channel_count"]
).all(), "Historical channel count mismatch."

user_baseline_profile["channel_combination"] = (
    user_baseline_profile.apply(
        lambda row: "+".join(
            channel
            for channel in channel_config
            if row[f"{channel.lower()}_active"]
        ),
        axis=1,
    )
)

# ---------------------------------------------------------
# 4. Audit P95 values on historically active channels
# ---------------------------------------------------------

p95_audit = []

for channel, config in channel_config.items():
    active = user_baseline_profile[
        f"{channel.lower()}_active"
    ]

    p95 = pd.to_numeric(
        user_baseline_profile[config["p95"]],
        errors="coerce",
    )

    invalid = active & (
        p95.isna()
        | ~np.isfinite(p95)
        | (p95 <= 0)
    )

    p95_audit.append({
        "channel": channel,
        "active_users": int(active.sum()),
        "valid_positive_p95": int(
            (active & ~invalid).sum()
        ),
        "invalid_active_p95": int(invalid.sum()),
        "min_active_p95": p95[active].min(),
        "median_active_p95": p95[active].median(),
        "p95_of_active_p95": p95[active].quantile(0.95),
        "max_active_p95": p95[active].max(),
    })

p95_audit_df = pd.DataFrame(p95_audit)

# ---------------------------------------------------------
# 5. Evaluate approved stealth multipliers
# ---------------------------------------------------------

stealth_multipliers = {
    "High": 0.5,
    "Medium": 1.0,
    "Low": 1.5,
}

stealth_summary = []
stealth_group_summary = []

for channel, config in channel_config.items():

    active = user_baseline_profile[
        f"{channel.lower()}_active"
    ]

    p95 = pd.to_numeric(
        user_baseline_profile[config["p95"]],
        errors="coerce",
    )

    for stealth, multiplier in stealth_multipliers.items():

        # Apply ceiling only to valid, nonnegative P95 values.
        valid = active & p95.notna() & np.isfinite(p95) & (p95 > 0)

        scaled = np.ceil(p95[valid] * multiplier)

        stealth_summary.append({
            "channel": channel,
            "stealth_level": stealth,
            "multiplier": multiplier,
            "active_users": int(active.sum()),
            "valid_scaled_users": int(valid.sum()),
            "zero_scaled_values": int((scaled == 0).sum()),
            "min_scaled_value": (
                scaled.min() if len(scaled) else np.nan
            ),
            "median_scaled_value": (
                scaled.median() if len(scaled) else np.nan
            ),
            "max_scaled_value": (
                scaled.max() if len(scaled) else np.nan
            ),
        })

        for group, group_data in user_baseline_profile.groupby(
            "assignment_group"
        ):
            group_active = group_data[
                f"{channel.lower()}_active"
            ]

            group_p95 = pd.to_numeric(
                group_data[config["p95"]],
                errors="coerce",
            )

            group_valid = (
                group_active
                & group_p95.notna()
                & np.isfinite(group_p95)
                & (group_p95 > 0)
            )

            group_scaled = np.ceil(
                group_p95[group_valid] * multiplier
            )

            stealth_group_summary.append({
                "assignment_group": group,
                "channel": channel,
                "stealth_level": stealth,
                "active_users": int(group_active.sum()),
                "valid_scaled_users": int(group_valid.sum()),
                "zero_scaled_values": int(
                    (group_scaled == 0).sum()
                ),
                "median_scaled_value": (
                    group_scaled.median()
                    if len(group_scaled) else np.nan
                ),
            })

stealth_summary_df = pd.DataFrame(stealth_summary)
stealth_group_summary_df = pd.DataFrame(
    stealth_group_summary
)

# ---------------------------------------------------------
# 6. Summarize historical channel combinations
# ---------------------------------------------------------

channel_summary = (
    user_baseline_profile
    .groupby(
        ["assignment_group", "channel_combination"],
        as_index=False
    )
    .agg(users=("user", "nunique"))
    .sort_values(
        ["assignment_group", "channel_combination"]
    )
)

# ---------------------------------------------------------
# 7. Identify any problematic active-channel P95 values
# ---------------------------------------------------------

invalid_records = []

for channel, config in channel_config.items():

    active = user_baseline_profile[
        f"{channel.lower()}_active"
    ]

    p95 = pd.to_numeric(
        user_baseline_profile[config["p95"]],
        errors="coerce",
    )

    invalid = active & (
        p95.isna()
        | ~np.isfinite(p95)
        | (p95 <= 0)
    )

    for _, row in user_baseline_profile.loc[invalid].iterrows():
        invalid_records.append({
            "user": row["user"],
            "assignment_group": row["assignment_group"],
            "channel": channel,
            "p95_value": row[config["p95"]],
        })

invalid_p95_df = pd.DataFrame(
    invalid_records,
    columns=[
        "user",
        "assignment_group",
        "channel",
        "p95_value",
    ],
)

# ---------------------------------------------------------
# 8. Display analysis results
# ---------------------------------------------------------

print("STEP 5.2 — HISTORICAL CHANNEL & BASELINE PROFILING")
print("=" * 75)

print("\nPROFILE VALIDATION")
print("Total users:", len(user_baseline_profile))
print("Unique users:", user_baseline_profile["user"].nunique())
print("Historical channel count verification: PASS")

print("\nHISTORICAL CHANNEL COMBINATIONS")
print(channel_summary.to_string(index=False))

print("\nACTIVE-CHANNEL P95 AUDIT")
print(p95_audit_df.to_string(index=False))

print("\nSTEALTH MULTIPLIER SUMMARY")
print(stealth_summary_df.to_string(index=False))

print("\nSTEALTH FEASIBILITY BY USER GROUP")
print(stealth_group_summary_df.to_string(index=False))

print("\nINVALID ACTIVE-CHANNEL P95 RECORDS")
print("Total:", len(invalid_p95_df))

if invalid_p95_df.empty:
    print("None detected.")
else:
    print(invalid_p95_df.to_string(index=False))

print("\nBASELINE USER PROFILE")
print("Full DataFrame shape:", user_baseline_profile.shape)
print("Preview: First 15 rows only (not the complete dataset)")

display(user_baseline_profile.head(15))


STEP 5.2 — HISTORICAL CHANNEL & BASELINE PROFILING

PROFILE VALIDATION
Total users: 995
Unique users: 995
Historical channel count verification: PASS

HISTORICAL CHANNEL COMBINATIONS
                      assignment_group channel_combination  users
           Three-channel long-duration      Email+USB+HTTP    289
Three-channel short-duration exception      Email+USB+HTTP      4
            Two-channel short-duration          Email+HTTP    274
            Two-channel short-duration           Email+USB    428

ACTIVE-CHANNEL P95 AUDIT
channel  active_users  valid_positive_p95  invalid_active_p95  min_active_p95  median_active_p95  p95_of_active_p95  max_active_p95
  Email           995                 995                   0             1.0                4.0                6.0            14.0
    USB           721                 721                   0             1.0                3.0                9.0            10.0
   HTTP           567                 567                   0    

,user,historical_channel_count,eligible_workdays,eligible_5d,eligible_10d,eligible_20d,eligible_30d,eligible_40d,eligible_50d,max_supported_duration,...,usb_active_days,http_upload_active_days,email_count_p95,usb_copy_p95,http_upload_p95,email_active,usb_active,http_active,verified_channel_count,channel_combination
0,AAC0610,2,105,True,True,True,True,True,True,50,...,133,0,2.0,4.00,NaN,True,True,False,2,Email+USB
1,AAF0819,2,105,True,True,True,True,True,True,50,...,203,0,4.0,4.00,NaN,True,True,False,2,Email+USB
2,AAP0352,2,105,True,True,True,True,True,True,50,...,79,0,4.0,2.10,NaN,True,True,False,2,Email+USB
3,AAP1919,2,105,True,True,True,True,True,True,50,...,40,0,5.0,3.05,NaN,True,True,False,2,Email+USB
4,AAS2987,2,105,True,True,True,True,True,True,50,...,0,66,2.0,NaN,3.0,True,False,True,2,Email+HTTP
5,ABD3426,3,106,True,True,True,True,True,True,50,...,257,59,6.0,9.00,3.0,True,True,True,3,Email+USB+HTTP
6,ABH0349,3,105,True,True,True,True,True,True,50,...,61,77,1.0,8.00,2.0,True,True,True,3,Email+USB+HTTP
7,ABK3081,2,106,True,True,True,True,True,True,50,...,53,0,4.0,3.00,NaN,True,True,False,2,Email+USB
8,ABM3203,2,106,True,True,True,True,True,True,50,...,0,63,5.0,NaN,2.9,True,False,True,2,Email+HTTP
9,ABM3772,2,105,True,True,True,True,True,True,50,...,96,0,6.0,3.00,NaN,True,True,False,2,Email+USB


كن هناك فرق بين صلاحية الحساب وبين تحقيق فصل فعلي بين مستويات التخفي.

مثلًا، إذا كانت قيمة P95 لمستخدم معين تساوي 1:
- High: ceil(1 × 0.5) = 1
- Medium: ceil(1 × 1.0) = 1
- Low: ceil(1 × 1.5) = 2
هنا High وMedium ينتجان العدد نفسه.

لذلك لا نستطيع الاستنتاج من النتائج الحالية أن مستويات التخفي الثلاثة متميزة لدى كل مستخدم. وهذا لا يعني ضرورة تعديل المضاعفات الآن، بل فقط قياس مدى حدوث هذا التداخل.


### 5.3 — Attack-Period Activity Profiling

**Objective:** Analyze the temporal availability of all 995 eligible users to determine how flexibly attack scenarios can be scheduled within the approved attack period.

**Attack Period:** January 1, 2011 – June 1, 2011 (inclusive).

**Eligible Workday Definition:**
- `is_weekend == 0`
- `logon_count > 0`

**Analysis Tasks:**
1. Verify the eligible attack-period workdays for all 995 users.
2. Identify each user's first and last eligible workday.
3. Measure calendar gaps between consecutive eligible workdays.
4. Calculate feasible attack start windows for durations of 5, 10, 20, 30, 40, and 50 eligible workdays.
5. Compare temporal availability across the three user groups.
6. Identify users with limited scheduling flexibility.

**Scheduling Assumptions:**
- Attack duration is measured in eligible workdays, not consecutive calendar days.
- Weekends and non-eligible days may occur between attack workdays.
- Attack-off days are included within the selected eligible-workday duration.
- No attack day may fall outside the approved attack period.
- A feasible start window is a sequence of the required number of eligible workdays, beginning at a distinct eligible date.
- This analysis evaluates temporal feasibility only; channel switching and detailed attack scheduling will be validated later.

**Expected Output:**
- User-level temporal activity profile.
- Eligible-workday and calendar-gap statistics.
- Feasible start-window counts by duration.
- Scheduling flexibility summaries by user group.
- Identification of users with limited temporal options.

**Scope:** Exploratory analysis only. No scenario assignment, attack injection, baseline modification, or file export.


In [30]:

# Purpose: Step 5.3 — Analyze eligible attack-period workdays,
# calendar gaps, and feasible start windows for all 995 users.
# Analysis only: no scenario assignment, injection, or export.

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Validate required inputs
# ---------------------------------------------------------

assert "eligible_user_profile" in globals(), (
    "Run Step 5.1 before Step 5.3."
)

assert "attack_period" in globals(), (
    "Run Step 3.1 to create attack_period first."
)

required_columns = [
    "user",
    "day",
    "is_weekend",
    "logon_count",
]

missing = [
    col for col in required_columns
    if col not in attack_period.columns
]

assert not missing, (
    f"Missing attack-period columns: {missing}"
)

assert eligible_user_profile["user"].is_unique

ATTACK_START_53 = pd.Timestamp("2011-01-01")
ATTACK_END_53 = pd.Timestamp("2011-06-01")

durations = [5, 10, 20, 30, 40, 50]

# ---------------------------------------------------------
# 2. Reconstruct eligible attack-period workdays
# ---------------------------------------------------------

activity = attack_period.loc[
    attack_period["user"].isin(
        eligible_user_profile["user"]
    ),
    required_columns
].copy()

activity["day"] = pd.to_datetime(
    activity["day"]
).dt.normalize()

assert activity["day"].notna().all(), (
    "Missing or invalid attack-period dates."
)

assert activity["day"].between(
    ATTACK_START_53,
    ATTACK_END_53
).all(), "Dates outside the approved attack period."

assert not activity.duplicated(
    ["user", "day"]
).any(), "Duplicate user-day records detected."

eligible_activity = activity.loc[
    (activity["is_weekend"] == 0)
    & (activity["logon_count"] > 0)
].copy()

eligible_activity = eligible_activity.sort_values(
    ["user", "day"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 3. Build user-level temporal profiles
# ---------------------------------------------------------

temporal_records = []

for user, group in eligible_activity.groupby(
    "user",
    sort=True
):

    days = group["day"].sort_values().reset_index(
        drop=True
    )

    n_days = len(days)

    # Calendar-day differences between eligible workdays.
    calendar_gaps = days.diff().dt.days.dropna()

    # Non-eligible calendar days strictly between workdays.
    intervening_days = calendar_gaps - 1

    record = {
        "user": user,
        "eligible_workdays_verified": n_days,
        "first_eligible_day": days.iloc[0],
        "last_eligible_day": days.iloc[-1],
        "max_calendar_gap": (
            int(calendar_gaps.max())
            if len(calendar_gaps) else 0
        ),
        "max_intervening_days": (
            int(intervening_days.max())
            if len(intervening_days) else 0
        ),
        "median_calendar_gap": (
            float(calendar_gaps.median())
            if len(calendar_gaps) else 0.0
        ),
    }

    # Each distinct eligible start day must leave enough
    # subsequent eligible workdays for the full duration.
    for duration in durations:

        window_count = max(
            0,
            n_days - duration + 1
        )

        record[f"start_windows_{duration}d"] = (
            window_count
        )

        record[f"first_start_{duration}d"] = (
            days.iloc[0]
            if window_count > 0 else pd.NaT
        )

        record[f"last_start_{duration}d"] = (
            days.iloc[n_days - duration]
            if window_count > 0 else pd.NaT
        )

    temporal_records.append(record)

temporal_profile = pd.DataFrame(
    temporal_records
)

# ---------------------------------------------------------
# 4. Merge with the existing eligible user profile
# ---------------------------------------------------------

user_temporal_profile = eligible_user_profile.merge(
    temporal_profile,
    on="user",
    how="left",
    validate="one_to_one",
    indicator=True,
)

assert (
    user_temporal_profile["_merge"] == "both"
).all(), "Some eligible users have no temporal profile."

user_temporal_profile = user_temporal_profile.drop(
    columns="_merge"
)

assert len(user_temporal_profile) == 995
assert user_temporal_profile["user"].is_unique

assert (
    user_temporal_profile["eligible_workdays"]
    == user_temporal_profile["eligible_workdays_verified"]
).all(), "Eligible-workday count mismatch."

for duration in durations:

    expected = (
        user_temporal_profile["eligible_workdays"]
        >= duration
    )

    actual = (
        user_temporal_profile[
            f"start_windows_{duration}d"
        ] > 0
    )

    assert (expected == actual).all(), (
        f"Scheduling feasibility mismatch: {duration}d"
    )

    assert (
        actual
        == user_temporal_profile[
            f"eligible_{duration}d"
        ].astype(bool)
    ).all(), (
        f"Existing duration flag mismatch: {duration}d"
    )

# ---------------------------------------------------------
# 5. Group-level temporal availability
# ---------------------------------------------------------

group_temporal_summary = (
    user_temporal_profile
    .groupby("assignment_group", as_index=False)
    .agg(
        users=("user", "nunique"),
        min_workdays=("eligible_workdays_verified", "min"),
        median_workdays=("eligible_workdays_verified", "median"),
        max_workdays=("eligible_workdays_verified", "max"),
        median_max_gap=("max_calendar_gap", "median"),
        largest_calendar_gap=("max_calendar_gap", "max"),
    )
    .sort_values("assignment_group")
)

# ---------------------------------------------------------
# 6. Feasible start windows by duration and group
# ---------------------------------------------------------

window_records = []

for group_name, group_data in (
    user_temporal_profile.groupby("assignment_group")
):

    for duration in durations:

        col = f"start_windows_{duration}d"

        feasible = group_data.loc[
            group_data[col] > 0,
            col
        ]

        window_records.append({
            "assignment_group": group_name,
            "duration": duration,
            "feasible_users": len(feasible),
            "min_start_windows": (
                int(feasible.min())
                if len(feasible) else 0
            ),
            "median_start_windows": (
                float(feasible.median())
                if len(feasible) else 0
            ),
            "max_start_windows": (
                int(feasible.max())
                if len(feasible) else 0
            ),
        })

start_window_summary = pd.DataFrame(
    window_records
).sort_values(
    ["assignment_group", "duration"]
)

# ---------------------------------------------------------
# 7. Identify users with limited temporal flexibility
# ---------------------------------------------------------

# Descriptive inspection only; this threshold does not
# create a new eligibility or exclusion rule.
limited_temporal_users = (
    user_temporal_profile.loc[
        user_temporal_profile[
            "start_windows_5d"
        ] <= 10,
        [
            "user",
            "assignment_group",
            "eligible_workdays_verified",
            "first_eligible_day",
            "last_eligible_day",
            "max_calendar_gap",
            "start_windows_5d",
            "start_windows_10d",
            "start_windows_20d",
            "start_windows_30d",
        ]
    ]
    .sort_values(
        ["start_windows_5d", "user"]
    )
)

# ---------------------------------------------------------
# 8. Display results
# ---------------------------------------------------------

print("STEP 5.3 — ATTACK-PERIOD ACTIVITY PROFILING")
print("=" * 75)

print("\nPOPULATION VALIDATION")
print("Total users:", len(user_temporal_profile))
print("Unique users:", user_temporal_profile["user"].nunique())
print("Eligible-workday verification: PASS")
print("Duration feasibility verification: PASS")

print("\nGROUP TEMPORAL SUMMARY")
print(group_temporal_summary.to_string(index=False))

print("\nFEASIBLE START WINDOWS BY DURATION")
print(start_window_summary.to_string(index=False))

print("\nUSERS WITH LIMITED TEMPORAL FLEXIBILITY")
print("Users with <= 10 possible 5-day starts:",
      len(limited_temporal_users))

if limited_temporal_users.empty:
    print("None detected.")
else:
    print(limited_temporal_users.to_string(index=False))

print("\nUSER TEMPORAL PROFILE")
print("Full DataFrame shape:", user_temporal_profile.shape)
print("Preview: First 15 rows only (not the complete dataset)")

display(
    user_temporal_profile
    .sort_values(["assignment_group", "user"])
    .head(15)
)


STEP 5.3 — ATTACK-PERIOD ACTIVITY PROFILING

POPULATION VALIDATION
Total users: 995
Unique users: 995
Eligible-workday verification: PASS
Duration feasibility verification: PASS

GROUP TEMPORAL SUMMARY
                      assignment_group  users  min_workdays  median_workdays  max_workdays  median_max_gap  largest_calendar_gap
           Three-channel long-duration    289            75            105.0           107             4.0                     4
Three-channel short-duration exception      4            15             29.0            30             3.0                     3
            Two-channel short-duration    702             7            105.0           108             4.0                     4

FEASIBLE START WINDOWS BY DURATION
                      assignment_group  duration  feasible_users  min_start_windows  median_start_windows  max_start_windows
           Three-channel long-duration         5             289                 71                 101.0                

,user,historical_channel_count,eligible_workdays,eligible_5d,eligible_10d,eligible_20d,eligible_30d,eligible_40d,eligible_50d,max_supported_duration,...,last_start_20d,start_windows_30d,first_start_30d,last_start_30d,start_windows_40d,first_start_40d,last_start_40d,start_windows_50d,first_start_50d,last_start_50d
5,ABD3426,3,106,True,True,True,True,True,True,50,...,2011-05-03,77,2011-01-03,2011-04-19,67,2011-01-03,2011-04-05,57,2011-01-03,2011-03-22
6,ABH0349,3,105,True,True,True,True,True,True,50,...,2011-05-03,76,2011-01-03,2011-04-18,66,2011-01-03,2011-04-04,56,2011-01-03,2011-03-21
10,ABO1173,3,105,True,True,True,True,True,True,50,...,2011-05-03,76,2011-01-03,2011-04-18,66,2011-01-03,2011-04-04,56,2011-01-03,2011-03-21
14,ACB0701,3,107,True,True,True,True,True,True,50,...,2011-05-04,78,2011-01-03,2011-04-20,68,2011-01-03,2011-04-06,58,2011-01-03,2011-03-23
16,ACG3819,3,106,True,True,True,True,True,True,50,...,2011-05-04,77,2011-01-03,2011-04-19,67,2011-01-03,2011-04-05,57,2011-01-03,2011-03-22
20,ACV1946,3,105,True,True,True,True,True,True,50,...,2011-05-03,76,2011-01-03,2011-04-18,66,2011-01-03,2011-04-04,56,2011-01-03,2011-03-21
21,ADG1737,3,105,True,True,True,True,True,True,50,...,2011-05-03,76,2011-01-03,2011-04-18,66,2011-01-03,2011-04-04,56,2011-01-03,2011-03-21
24,ADM0923,3,105,True,True,True,True,True,True,50,...,2011-05-03,76,2011-01-03,2011-04-18,66,2011-01-03,2011-04-04,56,2011-01-03,2011-03-21
27,AEB3249,3,105,True,True,True,True,True,True,50,...,2011-05-03,76,2011-01-03,2011-04-18,66,2011-01-03,2011-04-04,56,2011-01-03,2011-03-21
36,AGF0742,3,106,True,True,True,True,True,True,50,...,2011-05-04,77,2011-01-03,2011-04-19,67,2011-01-03,2011-04-05,57,2011-01-03,2011-03-22



### 5.4 — User–Scenario Compatibility Analysis

**Objective:** Evaluate user-level compatibility with the 1,143 approved attack scenarios before scenario assignment.

**Inputs:**
- `user_temporal_profile`: 995 eligible users and their attack-period availability.
- `user_baseline_profile`: Verified historical channel activity and baseline P95 values.
- `approved_scenarios`: 1,143 approved scenario templates.
- `scenario_channel_requirements`: Validated channel requirements for each scenario.

**Compatibility Rules:**
1. Each user must have sufficient eligible attack-period workdays for the scenario duration.
2. Two-channel scenarios require at least two historically active exfiltration channels.
3. Three-channel scenarios require all three historically active exfiltration channels.
4. The approved assignment policy restricts:
   - 289 three-channel long-duration users to 40- and 50-day scenarios.
   - 702 two-channel users to 5-, 10-, 20-, and 30-day scenarios.
   - 4 three-channel short-duration exceptions to compatible short-duration scenarios.
5. Historical P95 values must be valid for all active channels.
6. Approved scenario rules and stealth multipliers remain unchanged.

**Analysis Tasks:**
- Build the user–scenario compatibility matrix.
- Count compatible scenarios for each user.
- Count compatible users for each scenario.
- Summarize compatibility by user group, duration, and stealth level.
- Examine the 13 constrained users, including ACS0128 and JAW0551.
- Identify users or scenarios with zero compatible matches.
- Estimate the number of distinct scenario templates available under the approved assignment policy.

**Important Distinction:**

Compatibility does not mean assignment. A scenario may be compatible with multiple users, and the same template may be reused for different users during later assignment.

**Scope:** Compatibility analysis only. No scenario assignment, attack scheduling, injection, or file export.



In [31]:

# Purpose: Step 5.4 — Build and audit user–scenario compatibility
# using approved channel, duration, baseline, and group rules.
# No scenario assignment, injection, or file export.

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Validate required inputs
# ---------------------------------------------------------

required_objects = [
    "user_temporal_profile",
    "user_baseline_profile",
    "approved_scenarios",
    "scenario_channel_requirements",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Run earlier steps first: {missing_objects}"
)

users = user_temporal_profile.copy()
baseline = user_baseline_profile.copy()
scenarios = approved_scenarios.copy()
requirements = scenario_channel_requirements.copy()

assert len(users) == 995
assert users["user"].is_unique
assert baseline["user"].is_unique
assert len(scenarios) == 1143
assert "scenario_id" in scenarios.columns
assert scenarios["scenario_id"].is_unique

# ---------------------------------------------------------
# 2. Verify scenario requirements
# ---------------------------------------------------------

required_scenario_cols = [
    "scenario_id",
    "duration",
    "required_channel_count",
]

assert set(required_scenario_cols).issubset(
    requirements.columns
), "Missing scenario requirement columns."

assert requirements["scenario_id"].is_unique

scenario_info = scenarios.merge(
    requirements[required_scenario_cols],
    on="scenario_id",
    how="left",
    validate="one_to_one",
    suffixes=("", "_verified"),
)

if "duration_verified" in scenario_info.columns:
    assert (
        scenario_info["duration"]
        == scenario_info["duration_verified"]
    ).all(), "Scenario duration mismatch."

    scenario_info = scenario_info.drop(
        columns="duration_verified"
    )

assert scenario_info[
    "required_channel_count"
].notna().all()

assert set(
    scenario_info["required_channel_count"]
) == {2, 3}

assert set(
    scenario_info["duration"]
) == {5, 10, 20, 30, 40, 50}

# ---------------------------------------------------------
# 3. Verify historical channel and P95 readiness
# ---------------------------------------------------------

baseline_cols = [
    "user",
    "verified_channel_count",
    "channel_combination",
    "email_active",
    "usb_active",
    "http_active",
    "email_count_p95",
    "usb_copy_p95",
    "http_upload_p95",
]

assert set(baseline_cols).issubset(
    baseline.columns
)

users = users.merge(
    baseline[baseline_cols],
    on="user",
    how="left",
    validate="one_to_one",
)

assert users["verified_channel_count"].notna().all()

assert (
    users["historical_channel_count"]
    == users["verified_channel_count"]
).all(), "Historical channel count mismatch."

channel_p95_pairs = [
    ("email_active", "email_count_p95"),
    ("usb_active", "usb_copy_p95"),
    ("http_active", "http_upload_p95"),
]

for active_col, p95_col in channel_p95_pairs:
    invalid = (
        users[active_col].astype(bool)
        & (
            users[p95_col].isna()
            | ~np.isfinite(
                pd.to_numeric(users[p95_col], errors="coerce")
            )
            | (users[p95_col] <= 0)
        )
    )

    assert not invalid.any(), (
        f"Invalid active-channel P95: {p95_col}"
    )

# ---------------------------------------------------------
# 4. Apply approved assignment-group policy
# ---------------------------------------------------------

LONG_GROUP = "Three-channel long-duration"
SHORT_EXCEPTION_GROUP = (
    "Three-channel short-duration exception"
)
TWO_CHANNEL_GROUP = "Two-channel short-duration"

expected_groups = {
    LONG_GROUP: 289,
    SHORT_EXCEPTION_GROUP: 4,
    TWO_CHANNEL_GROUP: 702,
}

actual_groups = users[
    "assignment_group"
].value_counts().to_dict()

assert actual_groups == expected_groups, (
    f"Unexpected assignment groups: {actual_groups}"
)

# Long group: three-channel, 40/50 days.
# Short groups: two-channel scenarios, 5/10/20/30 days.

group_policy = {
    LONG_GROUP: {
        "durations": [40, 50],
        "required_channels": 3,
    },
    SHORT_EXCEPTION_GROUP: {
        "durations": [5, 10, 20, 30],
        "required_channels": 2,
    },
    TWO_CHANNEL_GROUP: {
        "durations": [5, 10, 20, 30],
        "required_channels": 2,
    },
}

# ---------------------------------------------------------
# 5. Construct compatible user–scenario pairs
# ---------------------------------------------------------

compatibility_parts = []

for group_name, policy in group_policy.items():

    group_users = users.loc[
        users["assignment_group"] == group_name,
        [
            "user",
            "assignment_group",
            "historical_channel_count",
            "eligible_workdays",
            "channel_combination",
        ]
    ].copy()

    group_scenarios = scenario_info.loc[
        scenario_info["duration"].isin(
            policy["durations"]
        )
        & (
            scenario_info["required_channel_count"]
            == policy["required_channels"]
        )
    ].copy()

    # Cross join candidates within the approved group.
    candidates = group_users.merge(
        group_scenarios,
        how="cross",
    )

    compatible = candidates.loc[
        (
            candidates["eligible_workdays"]
            >= candidates["duration"]
        )
        & (
            candidates["historical_channel_count"]
            >= candidates["required_channel_count"]
        )
    ].copy()

    compatibility_parts.append(compatible)

user_scenario_compatibility_54 = pd.concat(
    compatibility_parts,
    ignore_index=True,
)

assert not user_scenario_compatibility_54.empty

assert not user_scenario_compatibility_54.duplicated(
    ["user", "scenario_id"]
).any(), "Duplicate user–scenario compatibility pairs."

# ---------------------------------------------------------
# 6. Audit compatibility rules
# ---------------------------------------------------------

pairs = user_scenario_compatibility_54

assert (
    pairs["eligible_workdays"]
    >= pairs["duration"]
).all()

assert (
    pairs["historical_channel_count"]
    >= pairs["required_channel_count"]
).all()

assert (
    pairs.loc[
        pairs["assignment_group"] == LONG_GROUP,
        "duration"
    ].isin([40, 50])
).all()

assert (
    pairs.loc[
        pairs["assignment_group"] != LONG_GROUP,
        "duration"
    ].isin([5, 10, 20, 30])
).all()

assert (
    pairs.loc[
        pairs["assignment_group"] == LONG_GROUP,
        "required_channel_count"
    ] == 3
).all()

assert (
    pairs.loc[
        pairs["assignment_group"] != LONG_GROUP,
        "required_channel_count"
    ] == 2
).all()

# ---------------------------------------------------------
# 7. User-level compatibility summary
# ---------------------------------------------------------

user_compatibility_counts = (
    pairs.groupby("user")
    .agg(
        compatible_scenarios=("scenario_id", "nunique"),
        compatible_durations=("duration", "nunique"),
    )
    .reset_index()
)

user_compatibility_summary = users[
    [
        "user",
        "assignment_group",
        "historical_channel_count",
        "eligible_workdays",
        "channel_combination",
    ]
].merge(
    user_compatibility_counts,
    on="user",
    how="left",
    validate="one_to_one",
)

for col in [
    "compatible_scenarios",
    "compatible_durations",
]:
    user_compatibility_summary[col] = (
        user_compatibility_summary[col]
        .fillna(0)
        .astype(int)
    )

# ---------------------------------------------------------
# 8. Scenario-level compatibility summary
# ---------------------------------------------------------

scenario_compatibility_counts = (
    pairs.groupby("scenario_id")
    .agg(
        compatible_users=("user", "nunique")
    )
    .reset_index()
)

scenario_compatibility_summary = scenario_info.merge(
    scenario_compatibility_counts,
    on="scenario_id",
    how="left",
    validate="one_to_one",
)

scenario_compatibility_summary[
    "compatible_users"
] = (
    scenario_compatibility_summary[
        "compatible_users"
    ]
    .fillna(0)
    .astype(int)
)

# ---------------------------------------------------------
# 9. Group and duration summaries
# ---------------------------------------------------------

group_compatibility_summary = (
    user_compatibility_summary
    .groupby("assignment_group", as_index=False)
    .agg(
        users=("user", "nunique"),
        min_compatible_scenarios=(
            "compatible_scenarios", "min"
        ),
        median_compatible_scenarios=(
            "compatible_scenarios", "median"
        ),
        max_compatible_scenarios=(
            "compatible_scenarios", "max"
        ),
    )
)

duration_compatibility_summary = (
    pairs.groupby(
        ["assignment_group", "duration"],
        as_index=False
    )
    .agg(
        compatible_users=("user", "nunique"),
        available_scenarios=("scenario_id", "nunique"),
        compatible_pairs=("user", "size"),
    )
    .sort_values(
        ["assignment_group", "duration"]
    )
)

# ---------------------------------------------------------
# 10. Analyze the 13 constrained users
# ---------------------------------------------------------

# Nine constrained two-channel users are identified
# by their maximum supported duration (< 50 workdays).
# This includes ACS0128 and JAW0551.
#
# The four three-channel short-duration exceptions
# are included separately.

constrained_two_channel = users.loc[
    (users["assignment_group"] == TWO_CHANNEL_GROUP)
    & (users["max_supported_duration"] < 50),
    "user"
].tolist()

assert len(constrained_two_channel) == 9

short_exception_users = users.loc[
    users["assignment_group"] == SHORT_EXCEPTION_GROUP,
    "user"
].tolist()

assert len(short_exception_users) == 4

constrained_users = (
    constrained_two_channel + short_exception_users
)

assert len(set(constrained_users)) == 13

assert {
    "ACS0128",
    "JAW0551",
}.issubset(set(constrained_users))

constrained_compatibility = (
    user_compatibility_summary.loc[
        user_compatibility_summary[
            "user"
        ].isin(constrained_users)
    ]
    .sort_values(
        ["compatible_scenarios", "user"]
    )
)

# ---------------------------------------------------------
# 11. Identify unmatched users and scenarios
# ---------------------------------------------------------

unmatched_users = (
    user_compatibility_summary.loc[
        user_compatibility_summary[
            "compatible_scenarios"
        ] == 0
    ]
)

unmatched_scenarios = (
    scenario_compatibility_summary.loc[
        scenario_compatibility_summary[
            "compatible_users"
        ] == 0
    ]
)

# ---------------------------------------------------------
# 12. Display results
# ---------------------------------------------------------

print("STEP 5.4 — USER–SCENARIO COMPATIBILITY")
print("=" * 75)

print("\nCOMPATIBILITY MATRIX")
print("Full DataFrame shape:", pairs.shape)
print("Unique users:", pairs["user"].nunique())
print("Unique scenarios:", pairs["scenario_id"].nunique())
print("Compatible user–scenario pairs:", len(pairs))

print("\nGROUP COMPATIBILITY SUMMARY")
print(group_compatibility_summary.to_string(index=False))

print("\nDURATION COMPATIBILITY SUMMARY")
print(duration_compatibility_summary.to_string(index=False))

if "stealth_level" in pairs.columns:
    print("\nSTEALTH-LEVEL COMPATIBILITY")
    stealth_summary = (
        pairs.groupby(
            ["assignment_group", "stealth_level"],
            as_index=False
        )
        .agg(
            compatible_users=("user", "nunique"),
            available_scenarios=("scenario_id", "nunique"),
            compatible_pairs=("user", "size"),
        )
    )
    print(stealth_summary.to_string(index=False))
else:
    print(
        "\nStealth-level summary skipped: "
        "no stealth_level column found."
    )

print("\n13 CONSTRAINED USERS")
print(constrained_compatibility.to_string(index=False))

print("\nUNMATCHED USERS:", len(unmatched_users))
print("UNMATCHED SCENARIOS:", len(unmatched_scenarios))

print("\nUSER COMPATIBILITY SUMMARY")
print(
    "Full DataFrame shape:",
    user_compatibility_summary.shape
)
print("Preview: First 15 rows only (not the complete dataset)")
display(
    user_compatibility_summary
    .sort_values(["compatible_scenarios", "user"])
    .head(15)
)

print("\nSCENARIO COMPATIBILITY SUMMARY")
print(
    "Full DataFrame shape:",
    scenario_compatibility_summary.shape
)
print("Preview: First 15 rows only (not the complete dataset)")
display(
    scenario_compatibility_summary
    .sort_values(
        ["compatible_users", "scenario_id"]
    )
    .head(15)
)

print("\nVALIDATION: PASS")
print("No assignment or export performed.")


STEP 5.4 — USER–SCENARIO COMPATIBILITY

COMPATIBILITY MATRIX
Full DataFrame shape: (413196, 20)
Unique users: 995
Unique scenarios: 1143
Compatible user–scenario pairs: 413196

GROUP COMPATIBILITY SUMMARY
                      assignment_group  users  min_compatible_scenarios  median_compatible_scenarios  max_compatible_scenarios
           Three-channel long-duration    289                       942                        942.0                       942
Three-channel short-duration exception      4                        33                        151.5                       201
            Two-channel short-duration    702                         9                        201.0                       201

DURATION COMPATIBILITY SUMMARY
                      assignment_group  duration  compatible_users  available_scenarios  compatible_pairs
           Three-channel long-duration        40               289                  258             74562
           Three-channel long-duration     

,user,assignment_group,historical_channel_count,eligible_workdays,channel_combination,compatible_scenarios,compatible_durations
18,ACS0128,Two-channel short-duration,2,7,Email+HTTP,9,1
460,JAW0551,Two-channel short-duration,2,7,Email+HTTP,9,1
845,SKF2105,Three-channel short-duration exception,3,15,Email+USB+HTTP,33,2
117,BJE0351,Two-channel short-duration,2,28,Email+USB,102,3
287,DTH1333,Two-channel short-duration,2,27,Email+USB,102,3
332,FAJ3608,Three-channel short-duration exception,3,28,Email+USB+HTTP,102,3
450,IMS0862,Two-channel short-duration,2,29,Email+USB,102,3
0,AAC0610,Two-channel short-duration,2,105,Email+USB,201,4
1,AAF0819,Two-channel short-duration,2,105,Email+USB,201,4
2,AAP0352,Two-channel short-duration,2,105,Email+USB,201,4



SCENARIO COMPATIBILITY SUMMARY
Full DataFrame shape: (1143, 16)
Preview: First 15 rows only (not the complete dataset)


,scenario_id,duration,off_days,active_days,switching_level,min_switches,max_switches,target_switches,target_switching_rate,two_channel_days,three_channel_days,single_channel_days,stealth_level,stealth_multiplier,required_channel_count,compatible_users
201,SCN_0202,40,6,34,Low,4,8,4,0.121212,3,1,30,High,0.5,3,289
202,SCN_0203,40,6,34,Low,4,8,4,0.121212,3,1,30,Medium,1.0,3,289
203,SCN_0204,40,6,34,Low,4,8,4,0.121212,3,1,30,Low,1.5,3,289
204,SCN_0205,40,6,34,Low,4,8,4,0.121212,4,1,29,High,0.5,3,289
205,SCN_0206,40,6,34,Low,4,8,4,0.121212,4,1,29,Medium,1.0,3,289
206,SCN_0207,40,6,34,Low,4,8,4,0.121212,4,1,29,Low,1.5,3,289
207,SCN_0208,40,6,34,Low,4,8,5,0.151515,3,1,30,High,0.5,3,289
208,SCN_0209,40,6,34,Low,4,8,5,0.151515,3,1,30,Medium,1.0,3,289
209,SCN_0210,40,6,34,Low,4,8,5,0.151515,3,1,30,Low,1.5,3,289
210,SCN_0211,40,6,34,Low,4,8,5,0.151515,4,1,29,High,0.5,3,289



VALIDATION: PASS
No assignment or export performed.



### 5.5 — Scenario Reuse Feasibility

**Objective:** Evaluate whether the approved scenario templates can cover all 995 eligible users while maximizing scenario diversity and minimizing template reuse.

**Inputs:**
- `user_scenario_compatibility_54`: Validated user–scenario compatibility pairs.
- `user_compatibility_summary`: Compatible scenario counts for all 995 users.
- `scenario_compatibility_summary`: Compatible user counts for all 1,143 scenarios.
- `approved_scenarios`: Approved scenario templates.

**Approved Assignment Policy:**
- 289 three-channel users receive long-duration scenarios (40 or 50 workdays).
- 706 users receive short-duration scenarios (5, 10, 20, or 30 workdays).
- Each user receives exactly one synthetic attack.
- Scenario templates may be reused across different users.
- No fixed duration or stealth-level quotas are imposed.

**Analysis Tasks:**
1. Calculate user demand and available scenario templates for each assignment pool.
2. Calculate the theoretical minimum number of repeated template uses.
3. Check whether every short-duration scenario can be matched to a distinct compatible user.
4. Check whether 289 distinct long-duration scenarios can be matched to 289 distinct users.
5. Evaluate the 13 constrained users without consuming or reserving scenarios.
6. Assess scenario diversity by duration and stealth level.
7. Verify whether 995 unique-user assignments are feasible under the current compatibility rules.

**Methodology:**

Use bipartite matching to test the existence of distinct user–scenario pairings. This is a feasibility test, not a final assignment.

A scenario may be compatible with multiple users, but each user can receive only one attack.

**Expected Results:**
- Maximum number of distinct scenario templates usable.
- Minimum number of repeated template uses.
- Short- and long-duration pool feasibility.
- Compatibility coverage of the 13 constrained users.
- Evidence supporting the scenario assignment strategy.

**Scope:** Feasibility analysis only. No assignment export, attack scheduling, injection, or modification of approved scenario rules.


In [34]:

# Purpose: Step 5.5 — Evaluate maximum scenario diversity,
# minimum template reuse, and unique-user assignment feasibility.
# Feasibility analysis only: no final assignment or file export.

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching

# ---------------------------------------------------------
# 1. Validate inputs
# ---------------------------------------------------------

required_objects = [
    "user_scenario_compatibility_54",
    "user_compatibility_summary",
    "scenario_compatibility_summary",
    "approved_scenarios",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Missing inputs: {missing_objects}"
)

pairs = user_scenario_compatibility_54[
    [
        "user",
        "scenario_id",
        "assignment_group",
        "duration",
        "stealth_level",
    ]
].copy()

assert not pairs.duplicated(
    ["user", "scenario_id"]
).any()

assert pairs["user"].nunique() == 995
assert pairs["scenario_id"].nunique() == 1143

LONG_GROUP = "Three-channel long-duration"
SHORT_EXCEPTION_GROUP = (
    "Three-channel short-duration exception"
)
TWO_CHANNEL_GROUP = "Two-channel short-duration"

pairs["assignment_pool"] = np.where(
    pairs["assignment_group"] == LONG_GROUP,
    "Long-duration pool",
    "Short-duration pool",
)

# ---------------------------------------------------------
# 2. Bipartite matching feasibility helper
# ---------------------------------------------------------

def evaluate_matching(pool_pairs, orientation="scenario"):

    unique_users = sorted(pool_pairs["user"].unique())
    unique_scenarios = sorted(
        pool_pairs["scenario_id"].unique()
    )

    user_index = {
        user: idx
        for idx, user in enumerate(unique_users)
    }

    scenario_index = {
        scenario: idx
        for idx, scenario in enumerate(unique_scenarios)
    }

    row_indices = pool_pairs["user"].map(
        user_index
    ).to_numpy()

    col_indices = pool_pairs["scenario_id"].map(
        scenario_index
    ).to_numpy()

    adjacency = csr_matrix(
        (
            np.ones(len(pool_pairs), dtype=np.int8),
            (row_indices, col_indices),
        ),
        shape=(
            len(unique_users),
            len(unique_scenarios),
        ),
    )

    # perm_type="column" returns a matched scenario index
    # for each user, with -1 for unmatched users.
    user_to_scenario = maximum_bipartite_matching(
        adjacency,
        perm_type="column",
    )

    matched_users = int(
        np.count_nonzero(user_to_scenario >= 0)
    )

    matched_scenarios = matched_users

    # perm_type="row" returns a matched user index
    # for each scenario, with -1 for unmatched scenarios.
    scenario_to_user = maximum_bipartite_matching(
        adjacency,
        perm_type="row",
    )

    covered_scenarios = int(
        np.count_nonzero(scenario_to_user >= 0)
    )

    assert matched_scenarios == covered_scenarios

    return {
        "users": len(unique_users),
        "scenarios": len(unique_scenarios),
        "maximum_distinct_matches": matched_users,
        "unmatched_users_if_unique": (
            len(unique_users) - matched_users
        ),
        "uncovered_scenarios_if_unique": (
            len(unique_scenarios) - covered_scenarios
        ),
    }

# ---------------------------------------------------------
# 3. Evaluate short and long assignment pools
# ---------------------------------------------------------

pool_results = []

for pool_name, pool_data in pairs.groupby(
    "assignment_pool"
):

    result = evaluate_matching(pool_data)

    result["assignment_pool"] = pool_name
    result["compatible_pairs"] = len(pool_data)

    pool_results.append(result)

pool_feasibility = pd.DataFrame(
    pool_results
)[
    [
        "assignment_pool",
        "users",
        "scenarios",
        "compatible_pairs",
        "maximum_distinct_matches",
        "unmatched_users_if_unique",
        "uncovered_scenarios_if_unique",
    ]
].sort_values("assignment_pool")

# ---------------------------------------------------------
# 4. Calculate minimum template reuse
# ---------------------------------------------------------

total_users = int(
    pool_feasibility["users"].sum()
)

max_distinct_templates = int(
    pool_feasibility[
        "maximum_distinct_matches"
    ].sum()
)

minimum_repeated_uses = (
    total_users - max_distinct_templates
)

assert total_users == 995

# ---------------------------------------------------------
# 5. Verify complete coverage with permitted reuse
# ---------------------------------------------------------

user_coverage = (
    pairs.groupby("user")["scenario_id"]
    .nunique()
)

assert len(user_coverage) == 995

users_without_options = int(
    (user_coverage == 0).sum()
)

assert users_without_options == 0

# ---------------------------------------------------------
# 6. Analyze the 13 constrained users
# ---------------------------------------------------------

constrained_two_channel = (
    user_temporal_profile.loc[
        (
            user_temporal_profile[
                "assignment_group"
            ] == TWO_CHANNEL_GROUP
        )
        & (
            user_temporal_profile[
                "max_supported_duration"
            ] < 50
        ),
        "user",
    ].tolist()
)

short_exceptions = (
    user_temporal_profile.loc[
        user_temporal_profile[
            "assignment_group"
        ] == SHORT_EXCEPTION_GROUP,
        "user",
    ].tolist()
)

constrained_users = (
    constrained_two_channel + short_exceptions
)

assert len(set(constrained_users)) == 13

constrained_reuse_summary = (
    pairs.loc[
        pairs["user"].isin(constrained_users)
    ]
    .groupby(
        ["user", "assignment_group"],
        as_index=False
    )
    .agg(
        compatible_templates=(
            "scenario_id", "nunique"
        ),
        compatible_durations=(
            "duration", "nunique"
        ),
    )
    .sort_values(
        ["compatible_templates", "user"]
    )
)

# ---------------------------------------------------------
# 7. Diversity by duration and stealth level
# ---------------------------------------------------------

scenario_diversity = (
    pairs.groupby(
        [
            "assignment_pool",
            "duration",
            "stealth_level",
        ],
        as_index=False
    )
    .agg(
        compatible_users=("user", "nunique"),
        available_templates=(
            "scenario_id", "nunique"
        ),
    )
    .sort_values(
        [
            "assignment_pool",
            "duration",
            "stealth_level",
        ]
    )
)

# ---------------------------------------------------------
# 8. Additional feasibility assertions
# ---------------------------------------------------------

short_result = pool_feasibility.loc[
    pool_feasibility["assignment_pool"]
    == "Short-duration pool"
].iloc[0]

long_result = pool_feasibility.loc[
    pool_feasibility["assignment_pool"]
    == "Long-duration pool"
].iloc[0]

assert int(short_result["users"]) == 706
assert int(short_result["scenarios"]) == 201

assert int(long_result["users"]) == 289
assert int(long_result["scenarios"]) == 942

assert int(
    short_result["maximum_distinct_matches"]
) == 201, (
    "Not all short templates can be uniquely covered."
)

assert int(
    long_result["maximum_distinct_matches"]
) == 289, (
    "Cannot match all 289 long-duration users uniquely."
)

assert max_distinct_templates == 490
assert minimum_repeated_uses == 505

# ---------------------------------------------------------
# 9. Display results
# ---------------------------------------------------------

print("STEP 5.5 — SCENARIO REUSE FEASIBILITY")
print("=" * 75)

print("\nASSIGNMENT POOL FEASIBILITY")
print(pool_feasibility.to_string(index=False))

print("\nTHEORETICAL DIVERSITY CAPACITY")
print("Total unique users:", total_users)
print(
    "Maximum distinct scenario templates:",
    max_distinct_templates
)
print(
    "Minimum repeated template uses:",
    minimum_repeated_uses
)
print(
    "Users without compatible scenarios:",
    users_without_options
)

print("\n13 CONSTRAINED USERS")
print(
    constrained_reuse_summary.to_string(
        index=False
    )
)

print("\nSCENARIO DIVERSITY BY DURATION AND STEALTH")
print(scenario_diversity.to_string(index=False))

print("\nVALIDATION: PASS")
print(
    "Compatibility-based assignment feasibility "
    "confirmed."
)
print(
    "No final assignments or files were created."
)


STEP 5.5 — SCENARIO REUSE FEASIBILITY

ASSIGNMENT POOL FEASIBILITY
    assignment_pool  users  scenarios  compatible_pairs  maximum_distinct_matches  unmatched_users_if_unique  uncovered_scenarios_if_unique
 Long-duration pool    289        942            272238                       289                          0                            653
Short-duration pool    706        201            140958                       201                        505                              0

THEORETICAL DIVERSITY CAPACITY
Total unique users: 995
Maximum distinct scenario templates: 490
Minimum repeated template uses: 505
Users without compatible scenarios: 0

13 CONSTRAINED USERS
   user                       assignment_group  compatible_templates  compatible_durations
ACS0128             Two-channel short-duration                     9                     1
JAW0551             Two-channel short-duration                     9                     1
SKF2105 Three-channel short-duration exception 

In [33]:

# Purpose: Install SciPy in the active Jupyter kernel environment.

%pip install scipy


   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.6 MB ? eta -:--:--
   - -------------------------------------- 1.6/36.6 MB 6.0 MB/s eta 0:00:06
   ---- ----------------------------------- 3.9/36.6 MB 8.7 MB/s eta 0:00:04
   ------- -------------------------------- 6.6/36.6 MB 10.2 MB/s eta 0:00:03
   --------- ------------------------------ 8.4/36.6 MB 9.7 MB/s eta 0:00:03
   ------------ --------------------------- 11.0/36.6 MB 10.1 MB/s eta 0:00:03
   ------------- -------------------------- 12.6/36.6 MB 10.5 MB/s eta 0:00:03
   ---------------- ----------------------- 15.2/36.6 MB 10.2 MB/s eta 0:00:03
   --------------------- ------------------ 19.4/36.6 MB 11.3 MB/s eta 0:00:02
   ------------------------ --------------- 22.0/36.6 MB 11.6 MB/s eta 0:00:02
   --------------------------- ------------ 24.9/36.6 MB 11.8 MB/s eta 0:00:01
   ---------------------------- ----------- 26.0/36.6 MB 11.3 MB/s eta 0:00:0


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



## Step 6 — Scenario Assignment

### 6.1 — Assignment Strategy & Priority Planning

**Objective:** Prepare a deterministic, constraint-aware user assignment order before selecting scenario templates.

**Assignment policy:**
1. Assign at most one attack per unique user.
2. Use only previously validated user–scenario compatibility pairs.
3. Reserve the 289 three-channel long-duration users for 40- and 50-workday scenarios.
4. Assign the remaining 706 users to 5-, 10-, 20-, or 30-workday scenarios.
5. Prioritize users with fewer compatible scenario templates, followed by fewer compatible durations and fewer eligible attack workdays.
6. Maximize distinct scenario-template coverage before allowing reuse.
7. Preserve duration and stealth-level diversity where feasible, without introducing unapproved quotas.
8. Use deterministic ordering for reproducibility.

**Scope:** This step prepares and audits assignment priorities only. It does not select scenarios, schedule attacks, modify source datasets, or export files.


In [35]:

# Purpose: Step 6.1 — Prepare and validate a constraint-aware
# user assignment order without assigning any scenarios.

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Verify required inputs
# ---------------------------------------------------------

required_objects = [
    "user_compatibility_summary",
    "user_temporal_profile",
    "user_scenario_compatibility_54",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Missing required objects: {missing_objects}"
)

# ---------------------------------------------------------
# 2. Build user priority profile
# ---------------------------------------------------------

compatibility_counts = (
    user_scenario_compatibility_54
    .groupby("user")
    .agg(
        compatible_templates=("scenario_id", "nunique"),
        compatible_durations=("duration", "nunique")
    )
    .reset_index()
)

priority_columns = [
    "user",
    "assignment_group",
    "eligible_workdays"
]

assignment_priority = (
    user_temporal_profile[priority_columns]
    .merge(
        compatibility_counts,
        on="user",
        how="left",
        validate="one_to_one"
    )
)

assert assignment_priority.notna().all().all(), (
    "Missing values detected in assignment priority profile."
)

# ---------------------------------------------------------
# 3. Define assignment pools
# ---------------------------------------------------------

long_group = "Three-channel long-duration"

assignment_priority["assignment_pool"] = np.where(
    assignment_priority["assignment_group"].eq(long_group),
    "Long-duration pool",
    "Short-duration pool"
)

# ---------------------------------------------------------
# 4. Order users from most constrained to most flexible
# ---------------------------------------------------------

assignment_priority = (
    assignment_priority
    .sort_values(
        by=[
            "assignment_pool",
            "compatible_templates",
            "compatible_durations",
            "eligible_workdays",
            "user"
        ],
        ascending=[True, True, True, True, True],
        kind="stable"
    )
    .reset_index(drop=True)
)

assignment_priority["priority_rank"] = (
    assignment_priority
    .groupby("assignment_pool")
    .cumcount() + 1
)

# ---------------------------------------------------------
# 5. Summarize assignment priorities
# ---------------------------------------------------------

priority_summary = (
    assignment_priority
    .groupby("assignment_pool")
    .agg(
        users=("user", "nunique"),
        min_compatible_templates=("compatible_templates", "min"),
        median_compatible_templates=("compatible_templates", "median"),
        max_compatible_templates=("compatible_templates", "max"),
        min_eligible_workdays=("eligible_workdays", "min"),
        max_eligible_workdays=("eligible_workdays", "max")
    )
    .reset_index()
)

# ---------------------------------------------------------
# 6. Validate user coverage and pool sizes
# ---------------------------------------------------------

assert len(assignment_priority) == 995
assert assignment_priority["user"].is_unique

pool_counts = (
    assignment_priority["assignment_pool"]
    .value_counts()
)

assert pool_counts["Long-duration pool"] == 289
assert pool_counts["Short-duration pool"] == 706

assert (
    assignment_priority["compatible_templates"] > 0
).all()

assert (
    set(assignment_priority["user"])
    == set(user_scenario_compatibility_54["user"])
)

# ---------------------------------------------------------
# 7. Display results
# ---------------------------------------------------------

print("STEP 6.1 — ASSIGNMENT STRATEGY & PRIORITY PLANNING")
print("=" * 75)

print("\nASSIGNMENT PRIORITY SUMMARY")
print(priority_summary.to_string(index=False))

print("\nFIRST 15 USERS IN SHORT-DURATION PRIORITY ORDER")

short_priority = assignment_priority[
    assignment_priority["assignment_pool"]
    == "Short-duration pool"
]

print("Preview: First 15 rows only (not the complete dataset)")
print(short_priority.head(15).to_string(index=False))

print("\nFIRST 15 USERS IN LONG-DURATION PRIORITY ORDER")

long_priority = assignment_priority[
    assignment_priority["assignment_pool"]
    == "Long-duration pool"
]

print("Preview: First 15 rows only (not the complete dataset)")
print(long_priority.head(15).to_string(index=False))

print("\nDATAFRAME DIMENSIONS")
print("Assignment priority:", assignment_priority.shape)
print("Priority summary:", priority_summary.shape)

print("\nVALIDATION: PASS")
print("All 995 users have been prioritized.")
print("No scenario assignments or files were created.")


STEP 6.1 — ASSIGNMENT STRATEGY & PRIORITY PLANNING

ASSIGNMENT PRIORITY SUMMARY
    assignment_pool  users  min_compatible_templates  median_compatible_templates  max_compatible_templates  min_eligible_workdays  max_eligible_workdays
 Long-duration pool    289                       942                        942.0                       942                     75                    107
Short-duration pool    706                         9                        201.0                       201                      7                    108

FIRST 15 USERS IN SHORT-DURATION PRIORITY ORDER
Preview: First 15 rows only (not the complete dataset)
   user                       assignment_group  eligible_workdays  compatible_templates  compatible_durations     assignment_pool  priority_rank
ACS0128             Two-channel short-duration                  7                     9                     1 Short-duration pool              1
JAW0551             Two-channel short-duration                  


### 6.2 — Unique Scenario Matching

**Objective:** Construct a reproducible, maximum-cardinality matching between eligible users and distinct approved scenario templates.

**Method:**
1. Use the validated user–scenario compatibility pairs from Step 5.4.
2. Separate short-duration and long-duration assignment pools.
3. Apply maximum bipartite matching independently to each pool.
4. Assign each matched user exactly one compatible scenario.
5. Use each scenario template at most once in this initial assignment.
6. Preserve unmatched users for the subsequent scenario-reuse step.
7. Validate user uniqueness, scenario uniqueness, compatibility, and assignment coverage.

**Expected outcome:**
- 201 unique short-duration assignments.
- 289 unique long-duration assignments.
- 490 total unique-template assignments.
- 505 remaining users requiring scenario-template reuse.

**Scope:** This step creates provisional in-memory assignments only. It does not schedule attack events, validate detailed channel-switching feasibility, modify source datasets, or export files.


In [36]:

# Purpose: Step 6.2 — Create maximum-cardinality unique-template
# assignments using validated user–scenario compatibility.
# No attack scheduling, source-data modification, or file export.

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching

# ---------------------------------------------------------
# 1. Validate required inputs
# ---------------------------------------------------------

required_objects = [
    "assignment_priority",
    "user_scenario_compatibility_54",
    "approved_scenarios"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Missing required objects: {missing_objects}"
)

assert assignment_priority["user"].is_unique
assert approved_scenarios["scenario_id"].is_unique

# ---------------------------------------------------------
# 2. Prepare validated compatibility pairs
# ---------------------------------------------------------

compatibility_pairs_62 = (
    user_scenario_compatibility_54[
        ["user", "scenario_id", "duration"]
    ]
    .drop_duplicates(subset=["user", "scenario_id"])
    .merge(
        assignment_priority[
            ["user", "assignment_pool", "priority_rank"]
        ],
        on="user",
        how="inner",
        validate="many_to_one"
    )
)

assert len(compatibility_pairs_62) == 413196
assert compatibility_pairs_62["user"].nunique() == 995
assert compatibility_pairs_62["scenario_id"].nunique() == 1143

assert (
    (
        compatibility_pairs_62["assignment_pool"].eq(
            "Short-duration pool"
        )
        & compatibility_pairs_62["duration"].isin(
            [5, 10, 20, 30]
        )
    )
    |
    (
        compatibility_pairs_62["assignment_pool"].eq(
            "Long-duration pool"
        )
        & compatibility_pairs_62["duration"].isin(
            [40, 50]
        )
    )
).all()

# ---------------------------------------------------------
# 3. Maximum unique matching for one assignment pool
# ---------------------------------------------------------

def match_unique_templates(pool_name):

    pool_pairs = compatibility_pairs_62[
        compatibility_pairs_62["assignment_pool"].eq(pool_name)
    ].copy()

    pool_users = (
        assignment_priority[
            assignment_priority["assignment_pool"].eq(pool_name)
        ]
        .sort_values(
            ["priority_rank", "user"],
            kind="stable"
        )["user"]
        .tolist()
    )

    pool_scenarios = sorted(
        pool_pairs["scenario_id"].unique().tolist()
    )

    user_to_index = {
        user: index
        for index, user in enumerate(pool_users)
    }

    scenario_to_index = {
        scenario: index
        for index, scenario in enumerate(pool_scenarios)
    }

    row_indices = (
        pool_pairs["user"]
        .map(user_to_index)
        .to_numpy(dtype=np.int32)
    )

    column_indices = (
        pool_pairs["scenario_id"]
        .map(scenario_to_index)
        .to_numpy(dtype=np.int32)
    )

    adjacency = csr_matrix(
        (
            np.ones(len(pool_pairs), dtype=np.int8),
            (row_indices, column_indices)
        ),
        shape=(len(pool_users), len(pool_scenarios))
    )

    # One matched scenario index per user; -1 means unmatched.
    matched_scenario_indices = maximum_bipartite_matching(
        adjacency,
        perm_type="column"
    )

    matched_user_indices = np.flatnonzero(
        matched_scenario_indices >= 0
    )

    matched_users = [
        pool_users[index]
        for index in matched_user_indices
    ]

    matched_scenarios = [
        pool_scenarios[matched_scenario_indices[index]]
        for index in matched_user_indices
    ]

    matches = pd.DataFrame({
        "user": matched_users,
        "scenario_id": matched_scenarios
    })

    matches["assignment_pool"] = pool_name

    unmatched = pd.DataFrame({
        "user": [
            pool_users[index]
            for index in np.flatnonzero(
                matched_scenario_indices < 0
            )
        ]
    })

    unmatched["assignment_pool"] = pool_name

    return matches, unmatched

# ---------------------------------------------------------
# 4. Construct provisional unique assignments
# ---------------------------------------------------------

short_matches, short_unmatched = match_unique_templates(
    "Short-duration pool"
)

long_matches, long_unmatched = match_unique_templates(
    "Long-duration pool"
)

unique_scenario_assignments = pd.concat(
    [short_matches, long_matches],
    ignore_index=True
)

remaining_users_for_reuse = pd.concat(
    [short_unmatched, long_unmatched],
    ignore_index=True
)

# ---------------------------------------------------------
# 5. Enrich assignments with scenario and user metadata
# ---------------------------------------------------------

scenario_metadata_columns = [
    "scenario_id",
    "duration",
    "stealth_level"
]

unique_scenario_assignments = (
    unique_scenario_assignments
    .merge(
        approved_scenarios[scenario_metadata_columns],
        on="scenario_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        assignment_priority[
            ["user", "assignment_group",
             "eligible_workdays", "priority_rank"]
        ],
        on="user",
        how="left",
        validate="one_to_one"
    )
)

unique_scenario_assignments["assignment_type"] = (
    "Unique template"
)

remaining_users_for_reuse = (
    remaining_users_for_reuse
    .merge(
        assignment_priority[
            ["user", "assignment_group",
             "eligible_workdays", "priority_rank"]
        ],
        on="user",
        how="left",
        validate="one_to_one"
    )
)

# ---------------------------------------------------------
# 6. Verify compatibility and uniqueness
# ---------------------------------------------------------

assert len(short_matches) == 201
assert len(long_matches) == 289

assert len(unique_scenario_assignments) == 490
assert len(remaining_users_for_reuse) == 505

assert unique_scenario_assignments["user"].is_unique
assert unique_scenario_assignments["scenario_id"].is_unique

assert remaining_users_for_reuse["user"].is_unique

assert not (
    set(unique_scenario_assignments["user"])
    & set(remaining_users_for_reuse["user"])
)

assert (
    set(unique_scenario_assignments["user"])
    | set(remaining_users_for_reuse["user"])
) == set(assignment_priority["user"])

validated_pair_index = pd.MultiIndex.from_frame(
    compatibility_pairs_62[["user", "scenario_id"]]
)

assigned_pair_index = pd.MultiIndex.from_frame(
    unique_scenario_assignments[["user", "scenario_id"]]
)

assert assigned_pair_index.isin(
    validated_pair_index
).all(), "At least one assignment is incompatible."

assert unique_scenario_assignments[
    ["user", "scenario_id", "duration", "stealth_level"]
].notna().all().all()

# ---------------------------------------------------------
# 7. Summarize unique-template assignments
# ---------------------------------------------------------

unique_assignment_summary = (
    unique_scenario_assignments
    .groupby("assignment_pool")
    .agg(
        assigned_users=("user", "nunique"),
        distinct_templates=("scenario_id", "nunique")
    )
    .reset_index()
)

unique_duration_stealth_summary = (
    unique_scenario_assignments
    .groupby(
        ["assignment_pool", "duration", "stealth_level"]
    )
    .agg(
        assigned_users=("user", "nunique"),
        distinct_templates=("scenario_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["assignment_pool", "duration", "stealth_level"]
    )
)

remaining_pool_summary = (
    remaining_users_for_reuse
    .groupby("assignment_pool")
    .agg(
        remaining_users=("user", "nunique")
    )
    .reset_index()
)

# ---------------------------------------------------------
# 8. Display results
# ---------------------------------------------------------

print("STEP 6.2 — UNIQUE SCENARIO MATCHING")
print("=" * 75)

print("\nUNIQUE ASSIGNMENT SUMMARY")
print(unique_assignment_summary.to_string(index=False))

print("\nASSIGNMENT DIVERSITY BY DURATION AND STEALTH")
print(unique_duration_stealth_summary.to_string(index=False))

print("\nUSERS REMAINING FOR SCENARIO REUSE")
print(remaining_pool_summary.to_string(index=False))

print("\nFIRST 15 UNIQUE ASSIGNMENTS")
print("Preview: First 15 rows only (not the complete dataset)")
print(
    unique_scenario_assignments
    .sort_values(
        ["assignment_pool", "priority_rank", "user"]
    )
    .head(15)
    .to_string(index=False)
)

print("\nFIRST 15 USERS REMAINING FOR REUSE")
print("Preview: First 15 rows only (not the complete dataset)")
print(
    remaining_users_for_reuse
    .sort_values(
        ["assignment_pool", "priority_rank", "user"]
    )
    .head(15)
    .to_string(index=False)
)

print("\nDATAFRAME DIMENSIONS")
print(
    "Unique assignments:",
    unique_scenario_assignments.shape
)
print(
    "Remaining users:",
    remaining_users_for_reuse.shape
)

print("\nVALIDATION: PASS")
print("490 unique-template assignments created in memory.")
print("505 users remain for scenario-template reuse.")
print("No attack events were scheduled and no files were created.")


STEP 6.2 — UNIQUE SCENARIO MATCHING

UNIQUE ASSIGNMENT SUMMARY
    assignment_pool  assigned_users  distinct_templates
 Long-duration pool             289                 289
Short-duration pool             201                 201

ASSIGNMENT DIVERSITY BY DURATION AND STEALTH
    assignment_pool  duration stealth_level  assigned_users  distinct_templates
 Long-duration pool        40          High              86                  86
 Long-duration pool        40           Low              86                  86
 Long-duration pool        40        Medium              86                  86
 Long-duration pool        50          High              11                  11
 Long-duration pool        50           Low              10                  10
 Long-duration pool        50        Medium              10                  10
Short-duration pool         5          High               3                   3
Short-duration pool         5           Low               3                   3
Sho


### 6.2 — Balanced Unique Scenario Matching (Revised)

**Objective:** Construct provisional unique-template assignments while prioritizing stealth-level balance, then duration balance, and preserving maximum scenario diversity.

#### Problem Identified in the Previous Implementation

The original maximum bipartite matching implementation successfully produced 490 unique-template assignments:

- 201 short-duration assignments.
- 289 long-duration assignments.
- 505 users remaining for scenario-template reuse.

However, the matching algorithm maximized the number of unique matches without optimizing the distribution of attack durations or stealth levels.

In particular, the long-duration assignments were heavily concentrated in 40-workday scenarios:

- 258 assignments with 40-workday duration.
- 31 assignments with 50-workday duration.

This imbalance was influenced by the ordering of scenario IDs rather than an explicit experimental design decision.

Although all 289 long-duration users were compatible with both durations, the algorithm did not use that flexibility to improve duration balance.

Therefore, maximum-cardinality matching alone is insufficient for our assignment objectives.

#### Revised Assignment Decisions

**Priority 1 — Stealth-Level Balance**

- Distribute High, Medium, and Low stealth levels as evenly as possible.
- Target approximately one-third of assignments per stealth level.
- Maintain stealth balance within assignment pools and duration groups where feasible.
- Do not change the approved stealth multipliers (0.5, 1.0, and 1.5).

**Priority 2 — Duration Balance**

- Improve the distribution of attack durations after satisfying stealth-balance objectives.
- For the 289 long-duration users, target approximately 144–145 assignments per duration (40 and 50 workdays).
- For short-duration users, seek balanced coverage across 5, 10, 20, and 30 workdays while respecting individual eligibility and the availability of unique templates.
- Do not introduce rigid duration quotas that violate compatibility constraints.

**Priority 3 — Maximum Scenario Diversity**

- Preserve the maximum feasible 490 distinct scenario templates.
- Use each scenario template at most once during this initial unique-assignment phase.
- Reserve the remaining 505 users for the subsequent scenario-reuse phase.

#### Assignment Constraints

1. Each user may receive at most one scenario.
2. Every assigned user–scenario pair must exist in the validated compatibility dataset.
3. Long-duration users receive only 40- or 50-workday scenarios.
4. Short-duration users receive only 5-, 10-, 20-, or 30-workday scenarios.
5. Users with fewer compatible options should be protected during matching.
6. The assignment must be deterministic and reproducible.

#### Implementation Approach

- Use constrained-first bipartite matching for short-duration assignments.
- Include all 201 short-duration templates exactly once, preserving the approved template coverage.
- Use explicit duration and stealth targets for long-duration assignments.
- Verify user uniqueness, template uniqueness, compatibility, and distribution balance.
- Retain all unmatched users for the scenario-reuse phase.

**Scope:** This step creates provisional assignments in memory only. It does not schedule attacks, validate detailed channel-switching feasibility, modify source datasets, or export files.

**Important:** Stealth-level balance improves coverage of simulated attack categories but does not, by itself, guarantee better machine-learning performance.


In [37]:

# Purpose: Step 6.2 — Create balanced, maximum-diversity
# provisional scenario assignments.
# Priority: stealth balance, duration balance, unique templates.
# No attack scheduling, source modification, or file export.

import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching

# ---------------------------------------------------------
# 1. Validate required inputs
# ---------------------------------------------------------

required_objects = [
    "assignment_priority",
    "user_scenario_compatibility_54",
    "approved_scenarios"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Missing required objects: {missing_objects}"
)

assert assignment_priority["user"].is_unique
assert approved_scenarios["scenario_id"].is_unique

STEALTH_ORDER = ["High", "Medium", "Low"]

SHORT_DURATIONS = [5, 10, 20, 30]
LONG_DURATIONS = [40, 50]

# ---------------------------------------------------------
# 2. Prepare validated compatibility pairs
# ---------------------------------------------------------

compatibility_pairs_62 = (
    user_scenario_compatibility_54[
        ["user", "scenario_id", "duration"]
    ]
    .drop_duplicates(subset=["user", "scenario_id"])
    .merge(
        assignment_priority[
            ["user", "assignment_pool", "priority_rank"]
        ],
        on="user",
        how="inner",
        validate="many_to_one"
    )
    .merge(
        approved_scenarios[
            ["scenario_id", "stealth_level"]
        ],
        on="scenario_id",
        how="left",
        validate="many_to_one"
    )
)

assert len(compatibility_pairs_62) == 413196
assert compatibility_pairs_62["user"].nunique() == 995
assert compatibility_pairs_62["scenario_id"].nunique() == 1143

assert compatibility_pairs_62[
    ["duration", "stealth_level"]
].notna().all().all()

# ---------------------------------------------------------
# 3. Prepare short-duration unique-template matching
# ---------------------------------------------------------

short_pairs = compatibility_pairs_62[
    compatibility_pairs_62["assignment_pool"].eq(
        "Short-duration pool"
    )
].copy()

short_users = (
    assignment_priority[
        assignment_priority["assignment_pool"].eq(
            "Short-duration pool"
        )
    ]
    .sort_values(
        ["priority_rank", "user"],
        kind="stable"
    )["user"]
    .tolist()
)

short_scenarios = (
    approved_scenarios[
        approved_scenarios["duration"].isin(
            SHORT_DURATIONS
        )
    ][["scenario_id", "duration", "stealth_level"]]
    .copy()
)

assert len(short_users) == 706
assert len(short_scenarios) == 201

# Round-robin ordering across duration/stealth groups
# avoids relying on scenario_id ordering alone.

short_scenarios["stealth_rank"] = (
    short_scenarios["stealth_level"]
    .map({
        "High": 0,
        "Medium": 1,
        "Low": 2
    })
)

short_scenarios = short_scenarios.sort_values(
    ["duration", "stealth_rank", "scenario_id"],
    kind="stable"
).reset_index(drop=True)

short_scenarios["within_group_rank"] = (
    short_scenarios
    .groupby(["duration", "stealth_level"])
    .cumcount()
)

short_scenarios = short_scenarios.sort_values(
    [
        "within_group_rank",
        "stealth_rank",
        "duration",
        "scenario_id"
    ],
    kind="stable"
).reset_index(drop=True)

short_scenario_ids = (
    short_scenarios["scenario_id"].tolist()
)

short_user_index = {
    user: i for i, user in enumerate(short_users)
}

short_scenario_index = {
    scenario: i
    for i, scenario in enumerate(short_scenario_ids)
}

short_rows = (
    short_pairs["user"]
    .map(short_user_index)
    .to_numpy(dtype=np.int32)
)

short_cols = (
    short_pairs["scenario_id"]
    .map(short_scenario_index)
    .to_numpy(dtype=np.int32)
)

short_adjacency = csr_matrix(
    (
        np.ones(len(short_pairs), dtype=np.int8),
        (short_rows, short_cols)
    ),
    shape=(len(short_users), len(short_scenario_ids))
)

# Match every short template to one distinct eligible user.
short_matched_users = maximum_bipartite_matching(
    short_adjacency,
    perm_type="row"
)

assert len(short_matched_users) == 201
assert (short_matched_users >= 0).all(), (
    "Not all short-duration templates could be matched."
)

short_matches = pd.DataFrame({
    "user": [
        short_users[i]
        for i in short_matched_users
    ],
    "scenario_id": short_scenario_ids
})

short_matches["assignment_pool"] = (
    "Short-duration pool"
)

# ---------------------------------------------------------
# 4. Prepare balanced long-duration assignments
# ---------------------------------------------------------

long_users = (
    assignment_priority[
        assignment_priority["assignment_pool"].eq(
            "Long-duration pool"
        )
    ]
    .sort_values(
        ["priority_rank", "user"],
        kind="stable"
    )["user"]
    .tolist()
)

long_scenarios = (
    approved_scenarios[
        approved_scenarios["duration"].isin(
            LONG_DURATIONS
        )
    ][["scenario_id", "duration", "stealth_level"]]
    .copy()
)

assert len(long_users) == 289
assert len(long_scenarios) == 942

# Explicit targets:
# 40 days = 144 users, 50 days = 145 users.
# Stealth totals = High 97, Medium 96, Low 96.

long_targets = pd.DataFrame([
    {"duration": 40, "stealth_level": "High", "target": 48},
    {"duration": 40, "stealth_level": "Medium", "target": 48},
    {"duration": 40, "stealth_level": "Low", "target": 48},
    {"duration": 50, "stealth_level": "High", "target": 49},
    {"duration": 50, "stealth_level": "Medium", "target": 48},
    {"duration": 50, "stealth_level": "Low", "target": 48}
])

assert long_targets["target"].sum() == 289

long_selected_parts = []

for target in long_targets.itertuples(index=False):

    candidates = (
        long_scenarios[
            long_scenarios["duration"].eq(
                target.duration
            )
            &
            long_scenarios["stealth_level"].eq(
                target.stealth_level
            )
        ]
        .sort_values("scenario_id")
    )

    assert len(candidates) >= target.target, (
        "Insufficient long-duration scenario templates."
    )

    # Spread selected templates across the candidate list,
    # rather than taking only the earliest scenario IDs.
    selected_indices = np.linspace(
        0,
        len(candidates) - 1,
        target.target,
        dtype=int
    )

    selected = candidates.iloc[
        selected_indices
    ].copy()

    long_selected_parts.append(selected)

long_selected = pd.concat(
    long_selected_parts,
    ignore_index=True
)

assert len(long_selected) == 289
assert long_selected["scenario_id"].is_unique

# Interleave duration and stealth groups to avoid
# assigning one entire duration group first.

long_selected["stealth_rank"] = (
    long_selected["stealth_level"]
    .map({
        "High": 0,
        "Medium": 1,
        "Low": 2
    })
)

long_selected["within_group_rank"] = (
    long_selected
    .groupby(["duration", "stealth_level"])
    .cumcount()
)

long_selected = (
    long_selected
    .sort_values(
        [
            "within_group_rank",
            "stealth_rank",
            "duration",
            "scenario_id"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

long_matches = pd.DataFrame({
    "user": long_users,
    "scenario_id": long_selected["scenario_id"].tolist()
})

long_matches["assignment_pool"] = (
    "Long-duration pool"
)

# ---------------------------------------------------------
# 5. Combine provisional unique assignments
# ---------------------------------------------------------

unique_scenario_assignments = pd.concat(
    [short_matches, long_matches],
    ignore_index=True
)

unique_scenario_assignments = (
    unique_scenario_assignments
    .merge(
        approved_scenarios[
            ["scenario_id", "duration", "stealth_level"]
        ],
        on="scenario_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        assignment_priority[
            [
                "user",
                "assignment_group",
                "eligible_workdays",
                "priority_rank"
            ]
        ],
        on="user",
        how="left",
        validate="one_to_one"
    )
)

unique_scenario_assignments["assignment_type"] = (
    "Unique template"
)

# ---------------------------------------------------------
# 6. Identify remaining users for scenario reuse
# ---------------------------------------------------------

assigned_users = set(
    unique_scenario_assignments["user"]
)

remaining_users_for_reuse = (
    assignment_priority[
        ~assignment_priority["user"].isin(
            assigned_users
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# 7. Validate uniqueness, compatibility, and coverage
# ---------------------------------------------------------

assert len(short_matches) == 201
assert len(long_matches) == 289

assert len(unique_scenario_assignments) == 490
assert len(remaining_users_for_reuse) == 505

assert unique_scenario_assignments["user"].is_unique
assert unique_scenario_assignments["scenario_id"].is_unique

assert remaining_users_for_reuse["user"].is_unique

assert (
    set(unique_scenario_assignments["user"])
    | set(remaining_users_for_reuse["user"])
) == set(assignment_priority["user"])

assert not (
    set(unique_scenario_assignments["user"])
    & set(remaining_users_for_reuse["user"])
)

valid_pairs = pd.MultiIndex.from_frame(
    compatibility_pairs_62[
        ["user", "scenario_id"]
    ]
)

assigned_pairs = pd.MultiIndex.from_frame(
    unique_scenario_assignments[
        ["user", "scenario_id"]
    ]
)

assert assigned_pairs.isin(valid_pairs).all(), (
    "Incompatible user–scenario assignment detected."
)

assert unique_scenario_assignments[
    ["duration", "stealth_level"]
].notna().all().all()

# ---------------------------------------------------------
# 8. Validate long-duration balance
# ---------------------------------------------------------

long_assigned = unique_scenario_assignments[
    unique_scenario_assignments["assignment_pool"].eq(
        "Long-duration pool"
    )
]

actual_long_targets = (
    long_assigned
    .groupby(["duration", "stealth_level"])
    .size()
    .rename("actual")
    .reset_index()
)

long_balance_check = long_targets.merge(
    actual_long_targets,
    on=["duration", "stealth_level"],
    how="left",
    validate="one_to_one"
)

assert (
    long_balance_check["target"]
    == long_balance_check["actual"]
).all()

# ---------------------------------------------------------
# 9. Summarize stealth and duration distributions
# ---------------------------------------------------------

assignment_summary = (
    unique_scenario_assignments
    .groupby("assignment_pool")
    .agg(
        assigned_users=("user", "nunique"),
        distinct_templates=("scenario_id", "nunique")
    )
    .reset_index()
)

stealth_summary = (
    unique_scenario_assignments
    .groupby(["assignment_pool", "stealth_level"])
    .size()
    .rename("assigned_users")
    .reset_index()
)

duration_stealth_summary = (
    unique_scenario_assignments
    .groupby(
        ["assignment_pool", "duration", "stealth_level"]
    )
    .size()
    .rename("assigned_users")
    .reset_index()
    .sort_values(
        ["assignment_pool", "duration", "stealth_level"]
    )
)

overall_stealth = (
    unique_scenario_assignments
    .groupby("stealth_level")
    .size()
    .reindex(STEALTH_ORDER, fill_value=0)
)

assert overall_stealth.max() - overall_stealth.min() <= 1, (
    "Overall stealth distribution is not balanced."
)

# ---------------------------------------------------------
# 10. Display results
# ---------------------------------------------------------

print("STEP 6.2 — BALANCED UNIQUE SCENARIO MATCHING")
print("=" * 75)

print("\nUNIQUE ASSIGNMENT SUMMARY")
print(assignment_summary.to_string(index=False))

print("\nSTEALTH DISTRIBUTION BY ASSIGNMENT POOL")
print(stealth_summary.to_string(index=False))

print("\nDURATION AND STEALTH DISTRIBUTION")
print(duration_stealth_summary.to_string(index=False))

print("\nLONG-DURATION TARGET VALIDATION")
print(long_balance_check.to_string(index=False))

print("\nOVERALL STEALTH DISTRIBUTION")
print(overall_stealth.to_string())

print("\nUSERS REMAINING FOR SCENARIO REUSE")
print(
    remaining_users_for_reuse["assignment_pool"]
    .value_counts()
    .to_string()
)

print("\nFIRST 15 PROVISIONAL ASSIGNMENTS")
print("Preview: First 15 rows only (not the complete dataset)")
print(
    unique_scenario_assignments
    .sort_values(
        ["assignment_pool", "priority_rank", "user"]
    )
    .head(15)
    .to_string(index=False)
)

print("\nDATAFRAME DIMENSIONS")
print(
    "Unique assignments:",
    unique_scenario_assignments.shape
)
print(
    "Remaining users:",
    remaining_users_for_reuse.shape
)

print("\nVALIDATION: PASS")
print("490 unique-template assignments created in memory.")
print("Stealth balance and long-duration targets verified.")
print("505 users remain for scenario-template reuse.")
print("No attack events were scheduled and no files were created.")


STEP 6.2 — BALANCED UNIQUE SCENARIO MATCHING

UNIQUE ASSIGNMENT SUMMARY
    assignment_pool  assigned_users  distinct_templates
 Long-duration pool             289                 289
Short-duration pool             201                 201

STEALTH DISTRIBUTION BY ASSIGNMENT POOL
    assignment_pool stealth_level  assigned_users
 Long-duration pool          High              97
 Long-duration pool           Low              96
 Long-duration pool        Medium              96
Short-duration pool          High              67
Short-duration pool           Low              67
Short-duration pool        Medium              67

DURATION AND STEALTH DISTRIBUTION
    assignment_pool  duration stealth_level  assigned_users
 Long-duration pool        40          High              48
 Long-duration pool        40           Low              48
 Long-duration pool        40        Medium              48
 Long-duration pool        50          High              49
 Long-duration pool        50     


### 6.3 — Balanced Scenario Reuse Assignment

**Objective:** Assign compatible approved scenario templates to the remaining 505 users while preserving the 490 unique-template assignments established in Step 6.2.

#### Starting Point

Step 6.2 produced:

- 490 provisional assignments using 490 distinct scenario templates.
- 289 long-duration assignments and 201 short-duration assignments.
- 505 unassigned users, all belonging to the short-duration pool.
- An initial stealth distribution of High = 164, Medium = 163, and Low = 163.

All 201 approved short-duration templates have already been used once. Therefore, additional short-duration assignments require template reuse.

#### Assignment Priorities

**Priority 1 — Stealth-Level Balance**

Maintain the closest possible distribution across High, Medium, and Low stealth levels.

For 995 total assignments, the target is:

- High: 332
- Medium: 332
- Low: 331

These are assignment-count targets, not guarantees of equal behavioral difficulty or machine-learning performance.

**Priority 2 — Duration Balance**

Improve the distribution of short-duration attacks across 5, 10, 20, and 30 eligible workdays.

- Respect each user's validated duration eligibility.
- Prioritize users with fewer compatible duration options.
- Avoid assigning longer durations to users without sufficient eligible workdays.
- Do not impose an equal-duration quota when it conflicts with approved compatibility constraints.

**Priority 3 — Template Reuse Distribution**

- Reuse only approved, compatible short-duration templates.
- Prefer templates with fewer previous uses when stealth and duration priorities are equivalent.
- Avoid excessive concentration on a small number of templates.
- Preserve the 490 distinct templates already selected in Step 6.2.

#### Assignment Method

Use deterministic, constrained-first assignment.

For each remaining user:

1. Identify compatible approved short-duration scenarios.
2. Select the stealth level with the largest remaining target deficit.
3. Within that stealth level, prefer the duration with the lowest current assignment count, subject to eligibility.
4. Prefer the least-used compatible scenario template.
5. Apply scenario ID as a deterministic final tie-breaker.

After assignment, validate user uniqueness, compatibility, scenario coverage, stealth balance, duration distributions, and template reuse.

#### Scope and Limitations

This step creates a complete provisional assignment for the 995 currently eligible users.

It does not:

- Resolve the remaining five-user shortage against the original 1,000-attack target.
- Schedule individual attack events.
- Validate detailed channel-switching or attack-off-day feasibility.
- Modify source datasets or export files.

Final assignment acceptance remains subject to later scheduling and end-to-end validation.


In [38]:

# Purpose: Step 6.3 — Assign compatible reused scenario templates
# to the remaining 505 users, prioritizing stealth balance,
# duration balance, and controlled template reuse.
# No attack scheduling, source modification, or file export.

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Validate required inputs
# ---------------------------------------------------------

required_objects = [
    "unique_scenario_assignments",
    "remaining_users_for_reuse",
    "assignment_priority",
    "user_scenario_compatibility_54",
    "approved_scenarios"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Missing required objects: {missing_objects}"
)

assert len(unique_scenario_assignments) == 490
assert len(remaining_users_for_reuse) == 505

assert unique_scenario_assignments["user"].is_unique
assert unique_scenario_assignments["scenario_id"].is_unique
assert remaining_users_for_reuse["user"].is_unique

STEALTH_LEVELS = ["High", "Medium", "Low"]
SHORT_DURATIONS = [5, 10, 20, 30]

STEALTH_TARGETS = {
    "High": 332,
    "Medium": 332,
    "Low": 331
}

# ---------------------------------------------------------
# 2. Prepare compatibility lookup for remaining users
# ---------------------------------------------------------

remaining_users = (
    remaining_users_for_reuse
    .sort_values(
        [
            "priority_rank",
            "user"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)

assert (
    remaining_users["assignment_pool"]
    == "Short-duration pool"
).all()

reuse_candidates = (
    user_scenario_compatibility_54[
        ["user", "scenario_id", "duration"]
    ]
    .merge(
        remaining_users[["user"]],
        on="user",
        how="inner",
        validate="many_to_one"
    )
    .merge(
        approved_scenarios[
            ["scenario_id", "stealth_level"]
        ],
        on="scenario_id",
        how="left",
        validate="many_to_one"
    )
    .drop_duplicates(
        subset=["user", "scenario_id"]
    )
)

assert reuse_candidates["user"].nunique() == 505

assert reuse_candidates["duration"].isin(
    SHORT_DURATIONS
).all()

assert reuse_candidates[
    ["scenario_id", "duration", "stealth_level"]
].notna().all().all()

candidate_lookup = {
    user: group.reset_index(drop=True)
    for user, group in reuse_candidates.groupby("user")
}

# ---------------------------------------------------------
# 3. Initialize assignment distribution counters
# ---------------------------------------------------------

stealth_counts = (
    unique_scenario_assignments["stealth_level"]
    .value_counts()
    .reindex(STEALTH_LEVELS, fill_value=0)
    .to_dict()
)

duration_counts = (
    unique_scenario_assignments["duration"]
    .value_counts()
    .reindex(SHORT_DURATIONS, fill_value=0)
    .to_dict()
)

template_usage = (
    unique_scenario_assignments["scenario_id"]
    .value_counts()
    .to_dict()
)

initial_stealth_counts = stealth_counts.copy()
initial_duration_counts = duration_counts.copy()

reuse_assignment_records = []

# ---------------------------------------------------------
# 4. Assign remaining users with explicit priorities
# ---------------------------------------------------------

for user_row in remaining_users.itertuples(index=False):

    user = user_row.user

    candidates = candidate_lookup[user].copy()

    # Priority 1: Stealth target deficit.
    candidates["stealth_deficit"] = (
        candidates["stealth_level"]
        .map(
            lambda level:
            STEALTH_TARGETS[level] - stealth_counts[level]
        )
    )

    # Priority 2: Duration balance.
    candidates["duration_current_count"] = (
        candidates["duration"]
        .map(duration_counts)
    )

    # Priority 3: Prefer least-used compatible templates.
    candidates["template_current_uses"] = (
        candidates["scenario_id"]
        .map(
            lambda scenario:
            template_usage.get(scenario, 0)
        )
    )

    selected = (
        candidates
        .sort_values(
            [
                "stealth_deficit",
                "duration_current_count",
                "template_current_uses",
                "scenario_id"
            ],
            ascending=[
                False,
                True,
                True,
                True
            ],
            kind="stable"
        )
        .iloc[0]
    )

    scenario_id = selected["scenario_id"]
    duration = int(selected["duration"])
    stealth_level = selected["stealth_level"]

    reuse_assignment_records.append({
        "user": user,
        "scenario_id": scenario_id,
        "assignment_pool": "Short-duration pool",
        "duration": duration,
        "stealth_level": stealth_level,
        "assignment_group": user_row.assignment_group,
        "eligible_workdays": user_row.eligible_workdays,
        "priority_rank": user_row.priority_rank,
        "assignment_type": "Reused template"
    })

    stealth_counts[stealth_level] += 1
    duration_counts[duration] += 1

    template_usage[scenario_id] = (
        template_usage.get(scenario_id, 0) + 1
    )

reused_scenario_assignments = pd.DataFrame(
    reuse_assignment_records
)

# ---------------------------------------------------------
# 5. Combine all provisional assignments
# ---------------------------------------------------------

final_provisional_assignments = pd.concat(
    [
        unique_scenario_assignments,
        reused_scenario_assignments
    ],
    ignore_index=True
)

# ---------------------------------------------------------
# 6. Validate user uniqueness and scenario compatibility
# ---------------------------------------------------------

assert len(reused_scenario_assignments) == 505
assert len(final_provisional_assignments) == 995

assert final_provisional_assignments["user"].is_unique

assert (
    set(final_provisional_assignments["user"])
    == set(assignment_priority["user"])
)

assert (
    final_provisional_assignments["scenario_id"].nunique()
    == 490
)

assert (
    final_provisional_assignments["assignment_type"]
    .value_counts()["Unique template"]
    == 490
)

assert (
    final_provisional_assignments["assignment_type"]
    .value_counts()["Reused template"]
    == 505
)

valid_pairs = pd.MultiIndex.from_frame(
    user_scenario_compatibility_54[
        ["user", "scenario_id"]
    ]
)

assigned_pairs = pd.MultiIndex.from_frame(
    final_provisional_assignments[
        ["user", "scenario_id"]
    ]
)

assert assigned_pairs.isin(valid_pairs).all(), (
    "Incompatible user–scenario assignment detected."
)

# ---------------------------------------------------------
# 7. Validate stealth targets
# ---------------------------------------------------------

final_stealth_counts = (
    final_provisional_assignments["stealth_level"]
    .value_counts()
    .reindex(STEALTH_LEVELS, fill_value=0)
)

assert final_stealth_counts.to_dict() == (
    STEALTH_TARGETS
), "Final stealth distribution differs from targets."

assert (
    final_stealth_counts.max()
    - final_stealth_counts.min()
    <= 1
)

# ---------------------------------------------------------
# 8. Summarize duration and template reuse
# ---------------------------------------------------------

final_duration_stealth_summary = (
    final_provisional_assignments
    .groupby(
        ["assignment_pool", "duration", "stealth_level"]
    )
    .agg(
        assignments=("user", "count"),
        distinct_templates=("scenario_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["assignment_pool", "duration", "stealth_level"]
    )
)

final_duration_summary = (
    final_provisional_assignments
    .groupby(["assignment_pool", "duration"])
    .agg(
        assignments=("user", "count"),
        distinct_templates=("scenario_id", "nunique")
    )
    .reset_index()
)

template_reuse_summary = (
    final_provisional_assignments
    .groupby("scenario_id")
    .agg(
        total_uses=("user", "count"),
        duration=("duration", "first"),
        stealth_level=("stealth_level", "first")
    )
    .reset_index()
)

reuse_distribution = (
    template_reuse_summary["total_uses"]
    .value_counts()
    .sort_index()
    .rename_axis("uses_per_template")
    .reset_index(name="number_of_templates")
)

# ---------------------------------------------------------
# 9. Display results
# ---------------------------------------------------------

print("STEP 6.3 — BALANCED SCENARIO REUSE ASSIGNMENT")
print("=" * 75)

print("\nFINAL ASSIGNMENT COVERAGE")
print("Total assignments:", len(final_provisional_assignments))
print(
    "Unique users:",
    final_provisional_assignments["user"].nunique()
)
print(
    "Distinct scenario templates:",
    final_provisional_assignments["scenario_id"].nunique()
)
print(
    "Reused-template assignments:",
    len(reused_scenario_assignments)
)

print("\nFINAL STEALTH DISTRIBUTION")
print(final_stealth_counts.to_string())

print("\nDURATION DISTRIBUTION")
print(final_duration_summary.to_string(index=False))

print("\nDURATION AND STEALTH DISTRIBUTION")
print(
    final_duration_stealth_summary
    .to_string(index=False)
)

print("\nTEMPLATE REUSE DISTRIBUTION")
print(reuse_distribution.to_string(index=False))

print("\nTEMPLATE USAGE STATISTICS")
print(
    template_reuse_summary["total_uses"]
    .describe()
    .round(2)
    .to_string()
)

print("\nFIRST 15 REUSED-TEMPLATE ASSIGNMENTS")
print("Preview: First 15 rows only (not the complete dataset)")
print(
    reused_scenario_assignments
    .head(15)
    .to_string(index=False)
)

print("\nDATAFRAME DIMENSIONS")
print(
    "Reused assignments:",
    reused_scenario_assignments.shape
)
print(
    "Complete provisional assignments:",
    final_provisional_assignments.shape
)
print(
    "Template reuse summary:",
    template_reuse_summary.shape
)

print("\nVALIDATION: PASS")
print("All 995 eligible users have one compatible assignment.")
print("Stealth targets achieved.")
print("490 distinct scenario templates preserved.")
print("No attack events were scheduled and no files were created.")


STEP 6.3 — BALANCED SCENARIO REUSE ASSIGNMENT

FINAL ASSIGNMENT COVERAGE
Total assignments: 995
Unique users: 995
Distinct scenario templates: 490
Reused-template assignments: 505

FINAL STEALTH DISTRIBUTION
stealth_level
High      332
Medium    332
Low       331

DURATION DISTRIBUTION
    assignment_pool  duration  assignments  distinct_templates
 Long-duration pool        40          144                 144
 Long-duration pool        50          145                 145
Short-duration pool         5          176                   9
Short-duration pool        10          176                  24
Short-duration pool        20          177                  69
Short-duration pool        30          177                  99

DURATION AND STEALTH DISTRIBUTION
    assignment_pool  duration stealth_level  assignments  distinct_templates
 Long-duration pool        40          High           48                  48
 Long-duration pool        40           Low           48                  48
 Long-


### 6.4 — Complete Structural Diversity Audit

**Objective:** Reassess scenario diversity using the complete set of approved attack-design parameters.

**Problem Identified in Step 6.4:**
The previous audit reported 90 structural patterns but excluded important fields such as `off_days`, `target_switches`, and `target_switching_rate`. Consequently, different attack configurations may have been incorrectly grouped into the same structural pattern.

**Analysis:**
- Include all approved scenario-rule parameters, excluding identifiers.
- Compare distinct templates and structural patterns in the approved catalog and assigned attacks.
- Measure structural-pattern reuse and coverage by duration.
- Identify any missing or disproportionately represented configurations.

**Decision Criteria:** Preserve stealth balance first, duration balance second, and maximize structural diversity where feasible.

**Output:** Read-only audit with no reassignment, scheduling, or file export.


In [41]:

# Purpose: Step 6.4 — Recalculate structural diversity using
# all approved scenario-design fields and compare the full
# scenario catalog against the 995 provisional assignments.
# Read-only analysis; no reassignment, scheduling, or export.

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Validate inputs
# ---------------------------------------------------------

assert "approved_scenarios" in globals()
assert "final_provisional_assignments" in globals()

scenarios_65 = approved_scenarios.copy()
assignments_65 = final_provisional_assignments.copy()

assert len(scenarios_65) == 1143
assert len(assignments_65) == 995
assert scenarios_65["scenario_id"].is_unique
assert assignments_65["user"].is_unique

# Explicitly use all approved scenario-design fields.
rule_columns_65 = [
    "duration",
    "off_days",
    "active_days",
    "switching_level",
    "min_switches",
    "max_switches",
    "target_switches",
    "target_switching_rate",
    "two_channel_days",
    "three_channel_days",
    "single_channel_days",
    "stealth_level",
    "stealth_multiplier"
]

missing_columns_65 = [
    col for col in rule_columns_65
    if col not in scenarios_65.columns
]

assert not missing_columns_65, (
    f"Missing scenario-rule columns: {missing_columns_65}"
)

assert scenarios_65[
    rule_columns_65
].notna().all().all(), (
    "Missing values detected in approved scenario rules."
)

# Ensure no unreviewed scenario columns are silently excluded.
unexpected_columns_65 = (
    set(scenarios_65.columns)
    - set(rule_columns_65)
    - {"scenario_id"}
)

assert not unexpected_columns_65, (
    "Unreviewed scenario columns: "
    f"{sorted(unexpected_columns_65)}"
)

# ---------------------------------------------------------
# 2. Construct complete structural-pattern identifiers
# ---------------------------------------------------------

# Group by exact approved parameter values.
# No rounding or conversion to descriptive categories.

scenarios_65["structural_pattern_id"] = (
    scenarios_65
    .groupby(rule_columns_65, sort=True)
    .ngroup()
    + 1
)

pattern_catalog_65 = (
    scenarios_65
    .groupby(
        ["structural_pattern_id"] + rule_columns_65,
        dropna=False
    )
    .agg(
        catalog_templates=("scenario_id", "nunique")
    )
    .reset_index()
)

# ---------------------------------------------------------
# 3. Attach complete pattern IDs to assignments
# ---------------------------------------------------------

audit_65 = assignments_65.merge(
    scenarios_65[
        ["scenario_id", "structural_pattern_id"]
        + rule_columns_65
    ],
    on="scenario_id",
    how="left",
    validate="many_to_one",
    suffixes=("_assigned", "")
)

assert len(audit_65) == 995
assert audit_65["structural_pattern_id"].notna().all()

assert (
    audit_65["duration_assigned"]
    == audit_65["duration"]
).all()

assert (
    audit_65["stealth_level_assigned"]
    == audit_65["stealth_level"]
).all()

# ---------------------------------------------------------
# 4. Compare catalog and assigned pattern coverage
# ---------------------------------------------------------

catalog_duration_65 = (
    scenarios_65
    .groupby("duration")
    .agg(
        catalog_templates=("scenario_id", "nunique"),
        catalog_patterns=("structural_pattern_id", "nunique")
    )
    .reset_index()
)

assigned_duration_65 = (
    audit_65
    .groupby("duration")
    .agg(
        assignments=("user", "count"),
        assigned_templates=("scenario_id", "nunique"),
        assigned_patterns=("structural_pattern_id", "nunique")
    )
    .reset_index()
)

pattern_coverage_65 = (
    catalog_duration_65
    .merge(
        assigned_duration_65,
        on="duration",
        how="left",
        validate="one_to_one"
    )
)

pattern_coverage_65["template_coverage_pct"] = (
    100
    * pattern_coverage_65["assigned_templates"]
    / pattern_coverage_65["catalog_templates"]
)

pattern_coverage_65["pattern_coverage_pct"] = (
    100
    * pattern_coverage_65["assigned_patterns"]
    / pattern_coverage_65["catalog_patterns"]
)

# ---------------------------------------------------------
# 5. Measure structural-pattern reuse
# ---------------------------------------------------------

pattern_usage_65 = (
    audit_65
    .groupby(
        ["duration", "structural_pattern_id"]
    )
    .agg(
        assignments=("user", "count"),
        distinct_templates=("scenario_id", "nunique")
    )
    .reset_index()
)

pattern_concentration_records_65 = []

for duration, group in pattern_usage_65.groupby("duration"):

    counts = group["assignments"].to_numpy(
        dtype=float
    )

    proportions = counts / counts.sum()

    effective_patterns = (
        1.0 / np.sum(proportions ** 2)
    )

    pattern_concentration_records_65.append({
        "duration": duration,
        "effective_patterns": effective_patterns,
        "max_pattern_assignments": int(counts.max()),
        "mean_pattern_assignments": counts.mean()
    })

pattern_concentration_65 = pd.DataFrame(
    pattern_concentration_records_65
)

pattern_coverage_65 = pattern_coverage_65.merge(
    pattern_concentration_65,
    on="duration",
    validate="one_to_one"
)

# ---------------------------------------------------------
# 6. Identify missing and most-used patterns
# ---------------------------------------------------------

catalog_pattern_ids_65 = set(
    scenarios_65["structural_pattern_id"]
)

assigned_pattern_ids_65 = set(
    audit_65["structural_pattern_id"]
)

missing_pattern_ids_65 = (
    catalog_pattern_ids_65 - assigned_pattern_ids_65
)

missing_patterns_65 = (
    pattern_catalog_65[
        pattern_catalog_65["structural_pattern_id"].isin(
            missing_pattern_ids_65
        )
    ]
    .copy()
)

top_patterns_65 = (
    pattern_usage_65
    .sort_values(
        ["assignments", "duration", "structural_pattern_id"],
        ascending=[False, True, True]
    )
    .head(15)
    .merge(
        pattern_catalog_65,
        on="structural_pattern_id",
        how="left",
        validate="many_to_one",
        suffixes=("_assigned", "_catalog")
    )
)

# ---------------------------------------------------------
# 7. Check structural uniqueness in the approved catalog
# ---------------------------------------------------------

duplicate_pattern_groups_65 = (
    pattern_catalog_65[
        pattern_catalog_65["catalog_templates"] > 1
    ]
    .copy()
)

# ---------------------------------------------------------
# 8. Validate audit consistency
# ---------------------------------------------------------

assert (
    pattern_coverage_65["assignments"].sum()
    == 995
)

assert (
    pattern_usage_65["assignments"].sum()
    == 995
)

assert (
    pattern_coverage_65["assigned_patterns"]
    <= pattern_coverage_65["assigned_templates"]
).all()

assert (
    pattern_coverage_65["assigned_patterns"]
    <= pattern_coverage_65["catalog_patterns"]
).all()

assert (
    pattern_coverage_65["effective_patterns"]
    <= pattern_coverage_65["assigned_patterns"] + 1e-9
).all()

assert (
    len(assigned_pattern_ids_65)
    + len(missing_pattern_ids_65)
    == len(catalog_pattern_ids_65)
)

# ---------------------------------------------------------
# 9. Display audit results
# ---------------------------------------------------------

print("STEP 6.4 — COMPLETE STRUCTURAL DIVERSITY AUDIT")
print("=" * 78)

print("\nSTRUCTURAL RULE COLUMNS")
print(rule_columns_65)

print("\nOVERALL STRUCTURAL DIVERSITY")
print(
    "Approved scenario templates:",
    scenarios_65["scenario_id"].nunique()
)
print(
    "Catalog structural patterns:",
    scenarios_65["structural_pattern_id"].nunique()
)
print(
    "Assigned scenario templates:",
    audit_65["scenario_id"].nunique()
)
print(
    "Assigned structural patterns:",
    audit_65["structural_pattern_id"].nunique()
)
print(
    "Unassigned catalog patterns:",
    len(missing_pattern_ids_65)
)

print("\nSTRUCTURAL COVERAGE BY DURATION")
print(
    pattern_coverage_65
    .round(3)
    .to_string(index=False)
)

print("\nDUPLICATE STRUCTURAL PATTERNS IN CATALOG")
print(
    "Patterns shared by multiple scenario IDs:",
    len(duplicate_pattern_groups_65)
)

print("\nTOP 15 MOST ASSIGNED STRUCTURAL PATTERNS")
print("Preview: First 15 rows only (not the complete dataset)")
print(
    top_patterns_65[
        [
            "duration_assigned",
            "structural_pattern_id",
            "assignments",
            "distinct_templates",
            "catalog_templates",
            "off_days",
            "target_switches",
            "switching_level",
            "stealth_level"
        ]
    ].to_string(index=False)
)

print("\nMISSING STRUCTURAL PATTERNS BY DURATION")

if missing_patterns_65.empty:
    print("No missing structural patterns.")
else:
    print(
        missing_patterns_65
        .groupby("duration")
        .size()
        .rename("missing_patterns")
        .to_string()
    )

print("\nDATAFRAME DIMENSIONS")
print("Catalog patterns:", pattern_catalog_65.shape)
print("Assignment audit:", audit_65.shape)
print("Pattern usage:", pattern_usage_65.shape)
print("Pattern coverage:", pattern_coverage_65.shape)

print("\nVALIDATION: PASS")
print("Complete approved scenario-rule fields audited.")
print("No assignments were modified and no files were created.")


STEP 6.4 — COMPLETE STRUCTURAL DIVERSITY AUDIT

STRUCTURAL RULE COLUMNS
['duration', 'off_days', 'active_days', 'switching_level', 'min_switches', 'max_switches', 'target_switches', 'target_switching_rate', 'two_channel_days', 'three_channel_days', 'single_channel_days', 'stealth_level', 'stealth_multiplier']

OVERALL STRUCTURAL DIVERSITY
Approved scenario templates: 1143
Catalog structural patterns: 1143
Assigned scenario templates: 490
Assigned structural patterns: 490
Unassigned catalog patterns: 653

STRUCTURAL COVERAGE BY DURATION
 duration  catalog_templates  catalog_patterns  assignments  assigned_templates  assigned_patterns  template_coverage_pct  pattern_coverage_pct  effective_patterns  max_pattern_assignments  mean_pattern_assignments
        5                  9                 9          176                   9                  9                100.000               100.000               8.994                       20                    19.556
       10                 24

In [42]:
# Purpose: Display the exact reuse count for every assigned scenario template.

scenario_reuse_counts = (
    final_provisional_assignments
    .groupby(["duration", "scenario_id"])
    .size()
    .reset_index(name="times_used")
    .sort_values(["duration", "scenario_id"])
)

print(scenario_reuse_counts.to_string(index=False))

 duration scenario_id  times_used
        5    SCN_0001          20
        5    SCN_0002          20
        5    SCN_0003          20
        5    SCN_0004          20
        5    SCN_0005          19
        5    SCN_0006          20
        5    SCN_0007          19
        5    SCN_0008          19
        5    SCN_0009          19
       10    SCN_0010           8
       10    SCN_0011           8
       10    SCN_0012           8
       10    SCN_0013           8
       10    SCN_0014           8
       10    SCN_0015           8
       10    SCN_0016           7
       10    SCN_0017           8
       10    SCN_0018           8
       10    SCN_0019           7
       10    SCN_0020           7
       10    SCN_0021           7
       10    SCN_0022           7
       10    SCN_0023           7
       10    SCN_0024           7
       10    SCN_0025           7
       10    SCN_0026           7
       10    SCN_0027           7
       10    SCN_0028           7
       10    S


### Step 6 — Scenario Assignment: Final Summary

**Objective:** Assign approved attack scenarios to eligible users while maximizing scenario diversity and maintaining balanced attack characteristics.

#### 1. Problems Identified and Solutions

**Problem 1 — Scenario ID Ordering Bias (Step 6.2)**

The initial maximum bipartite matching achieved 490 unique assignments but produced an unintended long-duration imbalance: 258 attacks of 40 days versus only 31 attacks of 50 days.

**Solution:** Revised the matching strategy to explicitly prioritize stealth balance, followed by duration balance, while preserving 490 unique scenario templates.

**Problem 2 — Uneven Template Reuse (Step 6.3)**

After assigning the remaining 505 users, some 5-day templates were reused up to 20 times, raising concerns about repetitive attack patterns.

**Solution:** Conducted a complete template-reuse and structural-diversity audit. The analysis confirmed that reuse was evenly distributed within each duration. High reuse of 5-day templates resulted primarily from having only nine approved templates for 176 assignments.

**Problem 3 — Structural Diversity Verification (Step 6.4)**

Template uniqueness alone could not establish whether the assigned scenarios represented different attack configurations.

**Solution:** Evaluated all 13 approved scenario-rule parameters. The audit confirmed that the 490 assigned templates represented 490 distinct structural configurations.

#### 2. Final Assignment Decisions

1. **Stealth balance is the highest priority**, followed by duration balance and scenario diversity.
2. Preserve the approved scenario catalog and all existing eligibility constraints.
3. Assign exactly one compatible scenario per eligible user.
4. Allow scenario-template reuse across different users, but never duplicate users.
5. Preserve maximum feasible unique-template coverage before applying reuse.
6. Retain the current assignment distribution because the diversity audit found no sufficient reason to modify it.
7. Keep assignments provisional until detailed attack scheduling and subsequent validation are completed.

#### 3. Final Results

| Metric | Result |
|---|---:|
| Eligible users assigned | 995 |
| Distinct scenario templates | 490 |
| Unique-template assignments | 490 |
| Reused-template assignments | 505 |
| Distinct structural configurations | 490 |
| High stealth attacks | 332 |
| Medium stealth attacks | 332 |
| Low stealth attacks | 331 |

**Duration Distribution:**

| Duration | Assigned Attacks |
|---|---:|
| 5 days | 176 |
| 10 days | 176 |
| 20 days | 177 |
| 30 days | 177 |
| 40 days | 144 |
| 50 days | 145 |
| **Total** | **995** |

#### 4. Remaining Limitations

- The original target is 1,000 attacks, but only 995 users satisfy the current eligibility constraints, leaving a five-attack shortfall.
- Template-level compatibility has been verified, but detailed attack scheduling and channel-level execution feasibility remain unvalidated.
- Structural diversity reflects differences in approved scenario parameters, not necessarily differences in realized attack behavior.
- Balanced simulated stealth categories do not guarantee balanced detection difficulty or improved model performance.

**Final Decision:** Retain the 995 provisional assignments and proceed to Step 7 — Assignment Validation & Export without changing the approved simulation rules.



### 7.1 — Strict Assignment Integrity Validation

**Objective:** Independently verify the integrity of all provisional scenario assignments before final approval.

**Risk:** Although Step 6 produced 995 balanced assignments, successful distribution does not guarantee data integrity. Duplicate users, missing assignments, unauthorized scenario IDs, inconsistent attributes, or incorrect reuse records could remain undetected.

**Validation:**
- Verify schema, data types, missing values, and identifier integrity.
- Confirm exact coverage of the independently verified eligible-user cohort.
- Validate scenario IDs and assigned attributes against the approved catalog.
- Reconcile unique and reused assignments with the final dataset.
- Independently recompute assignment counts and distributions.
- Detect unexpected modifications or inconsistent records.

**Decision:** Any failed mandatory check blocks approval. No data correction, reassignment, or export is permitted during validation.

**Output:** Read-only integrity report with explicit PASS/FAIL results.


In [43]:

# Purpose: Step 7.1 — Perform strict, independent, fail-closed
# integrity validation of the 995 provisional assignments.
# Read-only: no corrections, reassignment, or file export.

import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype

print("STEP 7.1 — STRICT ASSIGNMENT INTEGRITY VALIDATION")
print("=" * 80)

checks_71 = []

def check_71(name, condition, details=""):
    passed = bool(condition)
    checks_71.append({
        "check": name,
        "status": "PASS" if passed else "FAIL",
        "details": str(details)
    })
    return passed

# ---------------------------------------------------------
# 1. Verify required inputs
# ---------------------------------------------------------

required_inputs_71 = [
    "final_provisional_assignments",
    "unique_scenario_assignments",
    "reused_scenario_assignments",
    "approved_scenarios",
    "eligible_user_profile"
]

missing_inputs_71 = [
    name for name in required_inputs_71
    if name not in globals()
]

check_71(
    "Required source datasets exist",
    len(missing_inputs_71) == 0,
    f"Missing: {missing_inputs_71}"
)

if missing_inputs_71:
    raise RuntimeError(
        "VALIDATION BLOCKED — Required inputs missing: "
        + ", ".join(missing_inputs_71)
    )

final_71 = final_provisional_assignments.copy(deep=True)
unique_71 = unique_scenario_assignments.copy(deep=True)
reused_71 = reused_scenario_assignments.copy(deep=True)
catalog_71 = approved_scenarios.copy(deep=True)
eligible_71 = eligible_user_profile.copy(deep=True)

sources_71 = {
    "Final assignments": final_71,
    "Unique assignments": unique_71,
    "Reused assignments": reused_71,
    "Approved catalog": catalog_71,
    "Eligible cohort": eligible_71
}

# ---------------------------------------------------------
# 2. Validate schemas and column names
# ---------------------------------------------------------

required_assignment_columns_71 = [
    "user",
    "scenario_id",
    "assignment_pool",
    "duration",
    "stealth_level",
    "assignment_group",
    "eligible_workdays",
    "priority_rank",
    "assignment_type"
]

for name, df in sources_71.items():
    check_71(
        f"{name}: no duplicate column names",
        df.columns.is_unique
    )

for name, df in [
    ("Final", final_71),
    ("Unique", unique_71),
    ("Reused", reused_71)
]:
    missing = sorted(
        set(required_assignment_columns_71) - set(df.columns)
    )
    extra = sorted(
        set(df.columns) - set(required_assignment_columns_71)
    )

    check_71(
        f"{name}: exact assignment schema",
        not missing and not extra,
        f"Missing={missing}; Extra={extra}"
    )

check_71(
    "Catalog: required columns",
    {"scenario_id", "duration", "stealth_level"}
    .issubset(catalog_71.columns)
)

check_71(
    "Eligible cohort: user column",
    "user" in eligible_71.columns
)

if any(
    set(required_assignment_columns_71) != set(df.columns)
    for df in [final_71, unique_71, reused_71]
) or not {"scenario_id", "duration", "stealth_level"}.issubset(
    catalog_71.columns
) or "user" not in eligible_71.columns:
    raise RuntimeError(
        "VALIDATION BLOCKED — Required schema is invalid."
    )

# ---------------------------------------------------------
# 3. Check missing values and identifiers
# ---------------------------------------------------------

for name, df in sources_71.items():
    check_71(
        f"{name}: no entirely duplicated rows",
        not df.duplicated().any(),
        f"Duplicates={int(df.duplicated().sum())}"
    )

for name, df in [
    ("Final", final_71),
    ("Unique", unique_71),
    ("Reused", reused_71)
]:
    missing_count = int(
        df[required_assignment_columns_71].isna().sum().sum()
    )

    check_71(
        f"{name}: no missing assignment values",
        missing_count == 0,
        f"Missing values={missing_count}"
    )

for name, df, col in [
    ("Final users", final_71, "user"),
    ("Unique users", unique_71, "user"),
    ("Reused users", reused_71, "user"),
    ("Catalog scenario IDs", catalog_71, "scenario_id"),
    ("Eligible cohort users", eligible_71, "user")
]:
    invalid = (
        df[col].isna()
        | df[col].astype(str).str.strip().eq("")
    )

    check_71(
        f"{name}: valid nonblank identifiers",
        not invalid.any(),
        f"Invalid={int(invalid.sum())}"
    )

    check_71(
        f"{name}: unique identifiers",
        df[col].is_unique,
        f"Duplicates={int(df[col].duplicated().sum())}"
    )

# ---------------------------------------------------------
# 4. Independently verify eligible-user coverage
# ---------------------------------------------------------

eligible_users_71 = set(eligible_71["user"])
assigned_users_71 = set(final_71["user"])

missing_users_71 = sorted(
    eligible_users_71 - assigned_users_71
)
unauthorized_users_71 = sorted(
    assigned_users_71 - eligible_users_71
)

check_71(
    "Eligible cohort contains exactly 995 users",
    len(eligible_71) == 995,
    f"Actual={len(eligible_71)}"
)

check_71(
    "Final assignments contain exactly 995 rows",
    len(final_71) == 995,
    f"Actual={len(final_71)}"
)

check_71(
    "All eligible users assigned exactly once",
    not missing_users_71
    and not unauthorized_users_71
    and final_71["user"].is_unique,
    (
        f"Missing users={missing_users_71[:10]}; "
        f"Unauthorized users={unauthorized_users_71[:10]}"
    )
)

# ---------------------------------------------------------
# 5. Validate assignment partition and provenance
# ---------------------------------------------------------

unique_users_71 = set(unique_71["user"])
reused_users_71 = set(reused_71["user"])

check_71(
    "Unique assignments: exactly 490",
    len(unique_71) == 490,
    f"Actual={len(unique_71)}"
)

check_71(
    "Reused assignments: exactly 505",
    len(reused_71) == 505,
    f"Actual={len(reused_71)}"
)

check_71(
    "Unique and reused user sets do not overlap",
    unique_users_71.isdisjoint(reused_users_71)
)

check_71(
    "Combined assignment users match final users",
    unique_users_71 | reused_users_71 == assigned_users_71
)

check_71(
    "Unique assignments use distinct scenario IDs",
    unique_71["scenario_id"].is_unique
)

check_71(
    "Unique assignment labels are correct",
    unique_71["assignment_type"].eq(
        "Unique template"
    ).all()
)

check_71(
    "Final assignment type labels are preserved",
    final_71["assignment_type"].value_counts(
        dropna=False
    ).to_dict()
    == {
        "Unique template": 490,
        "Reused template": 505
    },
    str(
        final_71["assignment_type"].value_counts(
            dropna=False
        ).to_dict()
    )
)

combined_71 = pd.concat(
    [unique_71, reused_71],
    ignore_index=True
)

# Compare all columns, not just user/scenario IDs.
canonical_columns_71 = sorted(
    required_assignment_columns_71
)

def canonical_rows_71(df):
    return (
        df[canonical_columns_71]
        .sort_values(
            canonical_columns_71,
            kind="stable"
        )
        .reset_index(drop=True)
    )

check_71(
    "Final records exactly equal unique + reused records",
    canonical_rows_71(final_71).equals(
        canonical_rows_71(combined_71)
    )
)

# ---------------------------------------------------------
# 6. Validate scenario catalog membership and attributes
# ---------------------------------------------------------

approved_ids_71 = set(catalog_71["scenario_id"])
assigned_ids_71 = set(final_71["scenario_id"])

unknown_scenarios_71 = sorted(
    assigned_ids_71 - approved_ids_71
)

check_71(
    "All assigned scenarios exist in approved catalog",
    not unknown_scenarios_71,
    f"Unknown IDs={unknown_scenarios_71[:10]}"
)

check_71(
    "Approved catalog contains exactly 1143 scenarios",
    len(catalog_71) == 1143,
    f"Actual={len(catalog_71)}"
)

catalog_lookup_71 = catalog_71[
    ["scenario_id", "duration", "stealth_level"]
].rename(
    columns={
        "duration": "approved_duration",
        "stealth_level": "approved_stealth"
    }
)

verified_71 = final_71.merge(
    catalog_lookup_71,
    on="scenario_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

check_71(
    "All scenario IDs resolve to one approved record",
    verified_71["_merge"].eq("both").all()
)

check_71(
    "Assigned duration exactly matches approved duration",
    verified_71["duration"].eq(
        verified_71["approved_duration"]
    ).all()
)

check_71(
    "Assigned stealth exactly matches approved stealth",
    verified_71["stealth_level"].eq(
        verified_71["approved_stealth"]
    ).all()
)

# ---------------------------------------------------------
# 7. Validate numeric and categorical constraints
# ---------------------------------------------------------

valid_durations_71 = {5, 10, 20, 30, 40, 50}
valid_stealth_71 = {"High", "Medium", "Low"}

check_71(
    "Duration values belong to approved duration set",
    final_71["duration"].isin(
        valid_durations_71
    ).all()
)

check_71(
    "Stealth values belong to approved stealth set",
    final_71["stealth_level"].isin(
        valid_stealth_71
    ).all()
)

check_71(
    "Eligible workdays are numeric",
    is_numeric_dtype(final_71["eligible_workdays"])
)

check_71(
    "Priority ranks are numeric",
    is_numeric_dtype(final_71["priority_rank"])
)

for column in ["eligible_workdays", "priority_rank"]:
    numeric_values = pd.to_numeric(
        final_71[column],
        errors="coerce"
    )

    check_71(
        f"{column}: positive finite integers",
        (
            numeric_values.notna()
            & np.isfinite(numeric_values)
            & numeric_values.gt(0)
            & numeric_values.eq(
                np.floor(numeric_values)
            )
        ).all()
    )

check_71(
    "All users have at least five eligible workdays",
    pd.to_numeric(
        final_71["eligible_workdays"],
        errors="coerce"
    ).ge(5).all()
)

check_71(
    "Assigned duration does not exceed eligible workdays",
    final_71["duration"].le(
        pd.to_numeric(
            final_71["eligible_workdays"],
            errors="coerce"
        )
    ).all()
)

# ---------------------------------------------------------
# 8. Independently recalculate distributions
# ---------------------------------------------------------

stealth_counts_71 = (
    final_71["stealth_level"]
    .value_counts()
    .reindex(["High", "Medium", "Low"], fill_value=0)
)

duration_counts_71 = (
    final_71["duration"]
    .value_counts()
    .reindex([5, 10, 20, 30, 40, 50], fill_value=0)
)

expected_stealth_71 = pd.Series(
    [332, 332, 331],
    index=["High", "Medium", "Low"]
)

expected_durations_71 = pd.Series(
    [176, 176, 177, 177, 144, 145],
    index=[5, 10, 20, 30, 40, 50]
)

check_71(
    "Stealth distribution matches approved Step 6 result",
    stealth_counts_71.equals(expected_stealth_71),
    str(stealth_counts_71.to_dict())
)

check_71(
    "Duration distribution matches approved Step 6 result",
    duration_counts_71.equals(expected_durations_71),
    str(duration_counts_71.to_dict())
)

check_71(
    "Exactly 490 distinct scenarios assigned",
    final_71["scenario_id"].nunique() == 490,
    f"Actual={final_71['scenario_id'].nunique()}"
)

check_71(
    "Stealth imbalance does not exceed one assignment",
    stealth_counts_71.max() - stealth_counts_71.min() <= 1
)

# ---------------------------------------------------------
# 9. Verify exact scenario reuse accounting
# ---------------------------------------------------------

reuse_counts_71 = (
    final_71["scenario_id"]
    .value_counts()
)

reuse_distribution_71 = (
    reuse_counts_71
    .value_counts()
    .sort_index()
)

expected_reuse_71 = pd.Series({
    1: 310,
    2: 108,
    3: 39,
    7: 16,
    8: 8,
    19: 4,
    20: 5
})

check_71(
    "Template reuse frequency matches Step 6 audit",
    reuse_distribution_71.equals(
        expected_reuse_71
    ),
    str(reuse_distribution_71.to_dict())
)

check_71(
    "Template use frequencies sum to 995",
    int(reuse_counts_71.sum()) == 995
)

# ---------------------------------------------------------
# 10. Print complete validation summary
# ---------------------------------------------------------

validation_report_71 = pd.DataFrame(checks_71)

failed_71 = validation_report_71[
    validation_report_71["status"] == "FAIL"
]

print("\nDATASET DIMENSIONS")
for name, df in sources_71.items():
    print(f"{name}: {df.shape}")

print("\nSTEALTH DISTRIBUTION")
print(stealth_counts_71.to_string())

print("\nDURATION DISTRIBUTION")
print(duration_counts_71.to_string())

print("\nTEMPLATE REUSE DISTRIBUTION")
print(reuse_distribution_71.to_string())

print("\nCOMPLETE VALIDATION REPORT")
print(
    validation_report_71.to_string(
        index=False,
        max_colwidth=100
    )
)

print("\nVALIDATION SUMMARY")
print("Total checks:", len(validation_report_71))
print(
    "Passed:",
    int((validation_report_71["status"] == "PASS").sum())
)
print("Failed:", len(failed_71))

if not failed_71.empty:
    print("\nFAILED CHECKS")
    print(
        failed_71.to_string(
            index=False,
            max_colwidth=150
        )
    )
    raise AssertionError(
        "STEP 7.1 FAILED — Do not approve or export assignments."
    )

print("\nSTEP 7.1 — PASS")
print("All assignment integrity checks passed.")
print("No assignments were modified or exported.")
print("Channel and temporal eligibility still require Step 7.2.")


STEP 7.1 — STRICT ASSIGNMENT INTEGRITY VALIDATION

DATASET DIMENSIONS
Final assignments: (995, 9)
Unique assignments: (490, 9)
Reused assignments: (505, 9)
Approved catalog: (1143, 14)
Eligible cohort: (995, 11)

STEALTH DISTRIBUTION
stealth_level
High      332
Medium    332
Low       331

DURATION DISTRIBUTION
duration
5     176
10    176
20    177
30    177
40    144
50    145

TEMPLATE REUSE DISTRIBUTION
count
1     310
2     108
3      39
7      16
8       8
19      4
20      5

COMPLETE VALIDATION REPORT
                                               check status                                               details
                      Required source datasets exist   PASS                                           Missing: []
        Final assignments: no duplicate column names   PASS                                                      
       Unique assignments: no duplicate column names   PASS                                                      
       Reused assignments: no


### 7.2 — Independent User Eligibility & Scenario Compatibility Validation

**Objective:** Independently verify that every assigned user satisfies the historical channel and attack-period duration requirements of their assigned scenario.

**Risk:** Step 7.1 confirmed assignment integrity, but valid records may still contain incompatible user–scenario combinations.

**Validation:**
- Recalculate historical channel eligibility from the benign baseline.
- Recalculate eligible attack-period workdays from daily activity records.
- Verify duration feasibility and the three-channel requirement for 40/50-day attacks.
- Compare recalculated eligibility with stored assignment values.
- Exclude known malicious users and detect missing source records.

**Decision:** Any incompatibility or source inconsistency blocks approval. Detailed scheduling feasibility remains a separate Stage 3 requirement.

**Output:** Independent, read-only compatibility validation report.


In [46]:

# Purpose: Step 7.2 — Independently validate historical
# channel eligibility, daily activity, attack-period workdays,
# and scenario compatibility using the verified user_day schema.
# Read-only: no data modification, reassignment, or export.

import pandas as pd
import numpy as np

print("STEP 7.2 — STRICT INDEPENDENT ELIGIBILITY VALIDATION")
print("=" * 82)

checks_72 = []

def check_72(name, condition, details=""):
    passed = bool(condition)
    checks_72.append({
        "check": name,
        "status": "PASS" if passed else "FAIL",
        "details": str(details)
    })

def require_72(condition, message):
    if not bool(condition):
        raise RuntimeError("VALIDATION BLOCKED — " + message)

# ---------------------------------------------------------
# 1. Verify required source datasets and schemas
# ---------------------------------------------------------

required_sources_72 = [
    "final_provisional_assignments",
    "user_baseline",
    "user_day",
    "approved_scenarios"
]

missing_sources_72 = [
    name for name in required_sources_72
    if name not in globals()
]

require_72(
    not missing_sources_72,
    f"Missing datasets: {missing_sources_72}"
)

assignments_72 = final_provisional_assignments.copy(deep=True)
baseline_72 = user_baseline.copy(deep=True)
daily_72 = user_day.copy(deep=True)
catalog_72 = approved_scenarios.copy(deep=True)

baseline_fields_72 = [
    "user",
    "email_active_days",
    "usb_active_days",
    "http_upload_active_days",
    "email_count_p95",
    "usb_copy_p95",
    "http_upload_p95"
]

daily_fields_72 = [
    "user",
    "day",
    "day_of_week",
    "is_weekend",
    "logon_count"
]

assignment_fields_72 = [
    "user",
    "scenario_id",
    "duration",
    "stealth_level",
    "eligible_workdays"
]

catalog_fields_72 = [
    "scenario_id",
    "duration",
    "stealth_level"
]

for name, df, fields in [
    ("Historical baseline", baseline_72, baseline_fields_72),
    ("Daily activity", daily_72, daily_fields_72),
    ("Assignments", assignments_72, assignment_fields_72),
    ("Approved catalog", catalog_72, catalog_fields_72)
]:
    require_72(
        df.columns.is_unique,
        f"{name} has duplicate column names."
    )

    missing = sorted(set(fields) - set(df.columns))

    require_72(
        not missing,
        f"{name} missing columns: {missing}"
    )

for name, df, key in [
    ("Baseline", baseline_72, "user"),
    ("Assignments", assignments_72, "user"),
    ("Catalog", catalog_72, "scenario_id")
]:
    require_72(
        df[key].notna().all()
        and df[key].is_unique,
        f"{name} has missing or duplicate identifiers."
    )

check_72(
    "Source datasets and required schemas are valid",
    True
)

# ---------------------------------------------------------
# 2. Independently validate historical channel activity
# ---------------------------------------------------------

historical_72 = baseline_72[
    baseline_fields_72
].copy()

channel_columns_72 = [
    "email_active_days",
    "usb_active_days",
    "http_upload_active_days"
]

p95_columns_72 = [
    "email_count_p95",
    "usb_copy_p95",
    "http_upload_p95"
]

for col in channel_columns_72 + p95_columns_72:
    historical_72[col] = pd.to_numeric(
        historical_72[col],
        errors="coerce"
    )

for col in channel_columns_72:
    values = historical_72[col]

    check_72(
        f"{col}: valid nonnegative integer counts",
        (
            values.notna()
            & np.isfinite(values)
            & values.ge(0)
            & values.eq(np.floor(values))
        ).all(),
        f"Invalid={int((~(
            values.notna()
            & np.isfinite(values)
            & values.ge(0)
            & values.eq(np.floor(values))
        )).sum())}"
    )

historical_72["historical_channel_count"] = (
    historical_72[channel_columns_72]
    .gt(0)
    .sum(axis=1)
)

historical_72["historical_channels"] = (
    historical_72[channel_columns_72]
    .gt(0)
    .apply(
        lambda row: "+".join(
            channel
            for channel, active in zip(
                ["Email", "USB", "HTTP"],
                row
            )
            if active
        ),
        axis=1
    )
)

for activity_col, p95_col in zip(
    channel_columns_72,
    p95_columns_72
):
    active = historical_72[activity_col].gt(0)
    p95 = historical_72[p95_col]

    valid = (
        ~active
        | (
            p95.notna()
            & np.isfinite(p95)
            & p95.ge(0)
        )
    )

    check_72(
        f"{p95_col}: valid for all historically active users",
        valid.all(),
        f"Invalid={int((~valid).sum())}"
    )

# ---------------------------------------------------------
# 3. Validate dates and daily activity records
# ---------------------------------------------------------

daily_72["day"] = pd.to_datetime(
    daily_72["day"],
    errors="coerce"
)

check_72(
    "All daily dates parse successfully",
    daily_72["day"].notna().all(),
    f"Invalid={int(daily_72['day'].isna().sum())}"
)

check_72(
    "Daily records have nonmissing user identifiers",
    daily_72["user"].notna().all()
    and daily_72["user"].astype(str).str.strip().ne("").all()
)

duplicate_days_72 = int(
    daily_72.duplicated(["user", "day"]).sum()
)

check_72(
    "No duplicate user-day records in full dataset",
    duplicate_days_72 == 0,
    f"Duplicates={duplicate_days_72}"
)

day_of_week_72 = pd.to_numeric(
    daily_72["day_of_week"],
    errors="coerce"
)

weekend_72 = pd.to_numeric(
    daily_72["is_weekend"],
    errors="coerce"
)

logon_72 = pd.to_numeric(
    daily_72["logon_count"],
    errors="coerce"
)

check_72(
    "Day-of-week values are valid (Monday=1 through Sunday=7)",
    day_of_week_72.isin(range(1, 8)).all()
)

check_72(
    "Day-of-week matches actual calendar dates",
    day_of_week_72.eq(
        daily_72["day"].dt.dayofweek + 1
    ).all()
)

check_72(
    "Weekend indicator is binary",
    weekend_72.isin([0, 1]).all()
)

# Check consistency with the source dataset's
# Monday=1 ... Sunday=7 convention.
expected_weekend_72 = day_of_week_72.isin([6, 7])

check_72(
    "Weekend indicator matches Saturday/Sunday convention",
    weekend_72.eq(expected_weekend_72.astype(int)).all()
)

check_72(
    "Logon counts are valid nonnegative integers",
    (
        logon_72.notna()
        & np.isfinite(logon_72)
        & logon_72.ge(0)
        & logon_72.eq(np.floor(logon_72))
    ).all()
)

# ---------------------------------------------------------
# 4. Independently reconstruct attack-period eligible days
# ---------------------------------------------------------

attack_start_72 = pd.Timestamp("2011-01-01")
attack_end_72 = pd.Timestamp("2011-06-01")

attack_period_72 = daily_72.loc[
    daily_72["day"].between(
        attack_start_72,
        attack_end_72,
        inclusive="both"
    )
].copy()

attack_period_72["eligible_workday_flag"] = (
    attack_period_72["is_weekend"].eq(0)
    & attack_period_72["logon_count"].gt(0)
)

recalculated_days_72 = (
    attack_period_72.loc[
        attack_period_72["eligible_workday_flag"]
    ]
    .groupby("user")
    .size()
    .rename("recalculated_eligible_workdays")
    .reset_index()
)

check_72(
    "Attack-period dates remain inside approved boundaries",
    attack_period_72["day"].between(
        attack_start_72,
        attack_end_72
    ).all()
)

check_72(
    "Eligible workday calculation uses only approved criteria",
    (
        attack_period_72["eligible_workday_flag"]
        == (
            attack_period_72["is_weekend"].eq(0)
            & attack_period_72["logon_count"].gt(0)
        )
    ).all()
)

# ---------------------------------------------------------
# 5. Reconcile assignments with independent sources
# ---------------------------------------------------------

validation_72 = (
    assignments_72
    .merge(
        historical_72[
            baseline_fields_72
            + [
                "historical_channel_count",
                "historical_channels"
            ]
        ],
        on="user",
        how="left",
        validate="one_to_one",
        indicator="baseline_match"
    )
    .merge(
        recalculated_days_72,
        on="user",
        how="left",
        validate="one_to_one",
        indicator="daily_match"
    )
)

check_72(
    "All assigned users exist in historical baseline",
    validation_72["baseline_match"].eq("both").all()
)

check_72(
    "All assigned users have eligible attack-period days",
    validation_72["daily_match"].eq("both").all()
)

stored_days_72 = pd.to_numeric(
    validation_72["eligible_workdays"],
    errors="coerce"
)

recomputed_days_72 = pd.to_numeric(
    validation_72["recalculated_eligible_workdays"],
    errors="coerce"
)

workday_match_72 = stored_days_72.eq(recomputed_days_72)

check_72(
    "Stored eligible workdays equal independently recalculated values",
    workday_match_72.all(),
    f"Mismatches={int((~workday_match_72).sum())}"
)

# ---------------------------------------------------------
# 6. Verify channel and duration compatibility
# ---------------------------------------------------------

validation_72["required_historical_channels"] = np.where(
    validation_72["duration"].isin([40, 50]),
    3,
    2
)

validation_72["channel_compatible"] = (
    validation_72["historical_channel_count"]
    >= validation_72["required_historical_channels"]
)

validation_72["duration_compatible"] = (
    recomputed_days_72
    >= validation_72["duration"]
)

validation_72["fully_compatible"] = (
    validation_72["channel_compatible"]
    & validation_72["duration_compatible"]
)

check_72(
    "All assigned users meet historical channel requirements",
    validation_72["channel_compatible"].all(),
    f"Incompatible={int((~validation_72['channel_compatible']).sum())}"
)

check_72(
    "All assigned users have enough eligible workdays",
    validation_72["duration_compatible"].all(),
    f"Incompatible={int((~validation_72['duration_compatible']).sum())}"
)

check_72(
    "All long-duration assignments have three historical channels",
    validation_72.loc[
        validation_72["duration"].isin([40, 50]),
        "historical_channel_count"
    ].eq(3).all()
)

check_72(
    "All 995 user-scenario pairs satisfy channel and duration rules",
    validation_72["fully_compatible"].all(),
    f"Incompatible={int((~validation_72['fully_compatible']).sum())}"
)

# ---------------------------------------------------------
# 7. Verify exclusion of known malicious users
# ---------------------------------------------------------

known_malicious_72 = {
    "ACM2278",
    "CDE1846",
    "CMP2946",
    "MBG3183",
    "PLJ1771"
}

malicious_overlap_72 = sorted(
    set(validation_72["user"])
    & known_malicious_72
)

check_72(
    "No known malicious users receive synthetic attacks",
    not malicious_overlap_72,
    f"Overlap={malicious_overlap_72}"
)

check_72(
    "Validation includes exactly 995 assignments",
    len(validation_72) == 995
)

# ---------------------------------------------------------
# 8. Produce diagnostic summaries
# ---------------------------------------------------------

duration_summary_72 = (
    validation_72
    .groupby("duration")
    .agg(
        assignments=("user", "count"),
        compatible=("fully_compatible", "sum"),
        min_eligible_workdays=(
            "recalculated_eligible_workdays", "min"
        ),
        max_eligible_workdays=(
            "recalculated_eligible_workdays", "max"
        ),
        min_historical_channels=(
            "historical_channel_count", "min"
        )
    )
    .reset_index()
)

channel_summary_72 = (
    validation_72
    .groupby("historical_channels", dropna=False)
    .agg(
        assignments=("user", "count"),
        compatible=("fully_compatible", "sum")
    )
    .reset_index()
)

incompatible_72 = validation_72.loc[
    ~validation_72["fully_compatible"]
    | ~workday_match_72
    | validation_72["baseline_match"].ne("both")
    | validation_72["daily_match"].ne("both")
].copy()

report_72 = pd.DataFrame(checks_72)
failed_72 = report_72[
    report_72["status"].eq("FAIL")
]

# ---------------------------------------------------------
# 9. Display validation results
# ---------------------------------------------------------

print("\nSOURCE DIMENSIONS")
print("Assignments:", assignments_72.shape)
print("Historical baseline:", baseline_72.shape)
print("Daily activity:", daily_72.shape)
print("Attack-period daily activity:", attack_period_72.shape)

print("\nINDEPENDENT ELIGIBILITY BY DURATION")
print(duration_summary_72.to_string(index=False))

print("\nHISTORICAL CHANNEL DISTRIBUTION")
print(channel_summary_72.to_string(index=False))

print("\nINCOMPATIBLE OR INCONSISTENT ASSIGNMENTS")
print("Total:", len(incompatible_72))

if not incompatible_72.empty:
    print("Preview: First 15 rows only (not the complete dataset)")
    print(
        incompatible_72[
            [
                "user",
                "scenario_id",
                "duration",
                "eligible_workdays",
                "recalculated_eligible_workdays",
                "historical_channels",
                "historical_channel_count",
                "required_historical_channels",
                "channel_compatible",
                "duration_compatible"
            ]
        ].head(15).to_string(index=False)
    )

print("\nCOMPLETE VALIDATION REPORT")
print(
    report_72.to_string(
        index=False,
        max_colwidth=110
    )
)

print("\nVALIDATION SUMMARY")
print("Total checks:", len(report_72))
print("Passed:", int(report_72["status"].eq("PASS").sum()))
print("Failed:", len(failed_72))

if not failed_72.empty:
    print("\nFAILED CHECKS")
    print(
        failed_72.to_string(
            index=False,
            max_colwidth=150
        )
    )
    raise AssertionError(
        "STEP 7.2 FAILED — Investigate source or compatibility issues."
    )

print("\nSTEP 7.2 — PASS")
print("All independently checked channel and duration rules passed.")
print("No assignments were modified or exported.")
print("Detailed scheduling feasibility remains for Stage 3.")


STEP 7.2 — STRICT INDEPENDENT ELIGIBILITY VALIDATION

SOURCE DIMENSIONS
Assignments: (995, 9)
Historical baseline: (3995, 36)
Daily activity: (1394010, 34)
Attack-period daily activity: (395562, 35)

INDEPENDENT ELIGIBILITY BY DURATION
 duration  assignments  compatible  min_eligible_workdays  max_eligible_workdays  min_historical_channels
        5          176         176                      7                    107                        2
       10          176         176                     15                    107                        2
       20          177         177                     27                    108                        2
       30          177         177                     30                    107                        2
       40          144         144                     75                    107                        3
       50          145         145                     83                    107                        3

HISTORICAL CHANNEL DI

In [45]:

# Purpose: Step 7.2 — Inspect the actual daily dataset schema
# and identify the correct date field before validation.

print("USER_DAY DATAFRAME SHAPE")
print(user_day.shape)

print("\nALL COLUMN NAMES")
print(user_day.columns.tolist())

print("\nINDEX INFORMATION")
print("Index type:", type(user_day.index).__name__)
print("Index names:", user_day.index.names)

print("\nDATE-RELATED COLUMNS")
date_candidates = [
    col for col in user_day.columns
    if any(
        keyword in str(col).lower()
        for keyword in ["date", "day", "time"]
    )
]
print(date_candidates)

print("\nDATA TYPES")
print(user_day.dtypes.to_string())

print("\nDATA PREVIEW")
print("Preview: First 15 rows only (not the complete dataset)")
print(user_day.head(15).to_string(index=False))


USER_DAY DATAFRAME SHAPE
(1394010, 34)

ALL COLUMN NAMES
['user', 'day', 'day_of_week', 'is_weekend', 'email_event_count', 'email_send_count', 'email_view_count', 'external_email_count', 'internal_email_count', 'total_sent_email_bytes', 'external_email_bytes', 'emails_with_attachments', 'attachment_count', 'unique_external_recipients', 'unique_external_domains', 'file_event_count', 'file_copy_count', 'file_open_count', 'file_write_count', 'file_delete_count', 'usb_copy_count', 'from_removable_count', 'unique_usb_files', 'unique_usb_extensions', 'http_event_count', 'www_visit_count', 'www_download_count', 'www_upload_count', 'unique_domains', 'unique_upload_domains', 'logon_count', 'logoff_count', 'first_logon', 'last_logoff']

INDEX INFORMATION
Index type: RangeIndex
Index names: [None]

DATE-RELATED COLUMNS
['day', 'day_of_week']

DATA TYPES
user                                     str
day                                   object
day_of_week                            int32
is_weekend


### 7.3 — Final Distribution, Diversity & Provenance Validation

**Objective:** Verify that the final assignments preserve the approved distribution priorities, scenario diversity, and assignment provenance.

**Risks:** Earlier integrity and eligibility checks do not fully establish that template reuse, structural coverage, and duration–stealth balance remain consistent with the final Step 6 decisions.

**Validation:**
- Independently reconstruct scenario usage and structural diversity from the approved catalog.
- Verify stealth and duration distributions, including their combinations.
- Validate template reuse, assignment provenance, and coverage by duration.
- Confirm that short-duration templates are fully represented and long-duration templates are not reused.
- Identify any changes from the approved Step 6 results.

**Decision:** Any mandatory failure blocks export. Successful validation confirms assignment-level readiness, not detailed scheduling or detection performance.

**Output:** Read-only final distribution and diversity report.


In [47]:

# Purpose: Step 7.3 — Independently validate final assignment
# distributions, complete structural diversity, template reuse,
# and assignment provenance against approved Step 6 results.
# Read-only: no modifications, reassignment, or file export.

import pandas as pd
import numpy as np

print("STEP 7.3 — FINAL DISTRIBUTION, DIVERSITY & PROVENANCE VALIDATION")
print("=" * 84)

checks_73 = []

def check_73(name, condition, details=""):
    passed = bool(condition)
    checks_73.append({
        "check": name,
        "status": "PASS" if passed else "FAIL",
        "details": str(details)
    })

def require_73(condition, message):
    if not bool(condition):
        raise RuntimeError("VALIDATION BLOCKED — " + message)

# ---------------------------------------------------------
# 1. Validate independent inputs
# ---------------------------------------------------------

required_sources_73 = [
    "final_provisional_assignments",
    "unique_scenario_assignments",
    "reused_scenario_assignments",
    "approved_scenarios"
]

missing_sources_73 = [
    name for name in required_sources_73
    if name not in globals()
]

require_73(
    not missing_sources_73,
    f"Missing source datasets: {missing_sources_73}"
)

assignments_73 = final_provisional_assignments.copy(deep=True)
unique_73 = unique_scenario_assignments.copy(deep=True)
reused_73 = reused_scenario_assignments.copy(deep=True)
catalog_73 = approved_scenarios.copy(deep=True)

assignment_columns_73 = [
    "user", "scenario_id", "assignment_pool",
    "duration", "stealth_level", "assignment_group",
    "eligible_workdays", "priority_rank", "assignment_type"
]

rule_columns_73 = [
    "duration",
    "off_days",
    "active_days",
    "switching_level",
    "min_switches",
    "max_switches",
    "target_switches",
    "target_switching_rate",
    "two_channel_days",
    "three_channel_days",
    "single_channel_days",
    "stealth_level",
    "stealth_multiplier"
]

for name, df, required in [
    ("Final assignments", assignments_73, assignment_columns_73),
    ("Unique assignments", unique_73, assignment_columns_73),
    ("Reused assignments", reused_73, assignment_columns_73),
    ("Approved catalog", catalog_73, ["scenario_id"] + rule_columns_73)
]:
    require_73(
        df.columns.is_unique,
        f"{name} has duplicate column names."
    )
    require_73(
        set(required).issubset(df.columns),
        f"{name} missing columns: {sorted(set(required) - set(df.columns))}"
    )

require_73(
    catalog_73["scenario_id"].is_unique,
    "Catalog scenario IDs are not unique."
)

require_73(
    assignments_73["user"].is_unique,
    "Final assignments contain duplicate users."
)

require_73(
    not catalog_73[rule_columns_73].isna().any().any(),
    "Approved scenario rules contain missing values."
)

check_73("Required schemas and source keys are valid", True)

# ---------------------------------------------------------
# 2. Independently join approved scenario rules
# ---------------------------------------------------------

audit_73 = assignments_73.merge(
    catalog_73[["scenario_id"] + rule_columns_73],
    on="scenario_id",
    how="left",
    validate="many_to_one",
    indicator=True,
    suffixes=("_assigned", "")
)

check_73(
    "Every assignment resolves to the approved catalog",
    audit_73["_merge"].eq("both").all()
)

check_73(
    "Assigned duration matches catalog",
    audit_73["duration_assigned"].eq(audit_73["duration"]).all()
)

check_73(
    "Assigned stealth matches catalog",
    audit_73["stealth_level_assigned"].eq(
        audit_73["stealth_level"]
    ).all()
)

# ---------------------------------------------------------
# 3. Independently reconstruct structural configurations
# ---------------------------------------------------------

# Exact comparison of all 13 approved design parameters.
# No rounding, grouping into broad categories, or use of IDs.

catalog_73["structural_pattern_id"] = (
    catalog_73
    .groupby(rule_columns_73, sort=True)
    .ngroup() + 1
)

audit_73 = audit_73.merge(
    catalog_73[["scenario_id", "structural_pattern_id"]],
    on="scenario_id",
    how="left",
    validate="many_to_one"
)

catalog_patterns_73 = catalog_73["structural_pattern_id"].nunique()
assigned_patterns_73 = audit_73["structural_pattern_id"].nunique()

check_73(
    "Approved catalog contains 1143 distinct structural patterns",
    catalog_patterns_73 == 1143,
    f"Actual={catalog_patterns_73}"
)

check_73(
    "Assigned attacks use exactly 490 structural patterns",
    assigned_patterns_73 == 490,
    f"Actual={assigned_patterns_73}"
)

check_73(
    "Assigned templates and structural patterns are one-to-one",
    assigned_patterns_73 == audit_73["scenario_id"].nunique()
)

# ---------------------------------------------------------
# 4. Independently calculate duration and stealth counts
# ---------------------------------------------------------

durations_73 = [5, 10, 20, 30, 40, 50]
stealth_levels_73 = ["High", "Medium", "Low"]

duration_counts_73 = (
    audit_73["duration"]
    .value_counts()
    .reindex(durations_73, fill_value=0)
)

stealth_counts_73 = (
    audit_73["stealth_level"]
    .value_counts()
    .reindex(stealth_levels_73, fill_value=0)
)

expected_duration_73 = pd.Series(
    [176, 176, 177, 177, 144, 145],
    index=durations_73
)

expected_stealth_73 = pd.Series(
    [332, 332, 331],
    index=stealth_levels_73
)

check_73(
    "Duration distribution matches final Step 6 decision",
    duration_counts_73.equals(expected_duration_73),
    str(duration_counts_73.to_dict())
)

check_73(
    "Stealth distribution matches final Step 6 decision",
    stealth_counts_73.equals(expected_stealth_73),
    str(stealth_counts_73.to_dict())
)

check_73(
    "Overall stealth difference does not exceed one",
    int(stealth_counts_73.max() - stealth_counts_73.min()) <= 1
)

cross_distribution_73 = pd.crosstab(
    audit_73["duration"],
    audit_73["stealth_level"]
).reindex(
    index=durations_73,
    columns=stealth_levels_73,
    fill_value=0
)

# Explicit expected combinations from Step 6.3.
expected_cross_73 = pd.DataFrame(
    [
        [59, 58, 59],
        [58, 59, 59],
        [59, 60, 58],
        [59, 59, 59],
        [48, 48, 48],
        [49, 48, 48]
    ],
    index=durations_73,
    columns=stealth_levels_73
)

check_73(
    "Every duration-stealth combination matches Step 6.3",
    cross_distribution_73.equals(expected_cross_73)
)

check_73(
    "Stealth spread within each duration is at most two",
    cross_distribution_73.apply(
        lambda row: row.max() - row.min(),
        axis=1
    ).le(2).all()
)

# ---------------------------------------------------------
# 5. Reconstruct template usage and reuse frequency
# ---------------------------------------------------------

template_usage_73 = (
    audit_73
    .groupby(["duration", "scenario_id"], as_index=False)
    .agg(uses=("user", "count"))
)

reuse_frequency_73 = (
    template_usage_73["uses"]
    .value_counts()
    .sort_index()
)

expected_reuse_frequency_73 = pd.Series({
    1: 310,
    2: 108,
    3: 39,
    7: 16,
    8: 8,
    19: 4,
    20: 5
})

check_73(
    "Exactly 490 templates used",
    len(template_usage_73) == 490,
    f"Actual={len(template_usage_73)}"
)

check_73(
    "Template reuse frequency matches Step 6 audit",
    reuse_frequency_73.equals(expected_reuse_frequency_73),
    str(reuse_frequency_73.to_dict())
)

check_73(
    "Template usage accounts for all 995 assignments",
    int(template_usage_73["uses"].sum()) == 995
)

# ---------------------------------------------------------
# 6. Verify coverage and reuse by duration
# ---------------------------------------------------------

catalog_by_duration_73 = (
    catalog_73
    .groupby("duration")
    .agg(
        available_templates=("scenario_id", "nunique"),
        available_patterns=("structural_pattern_id", "nunique")
    )
)

assigned_by_duration_73 = (
    audit_73
    .groupby("duration")
    .agg(
        assignments=("user", "count"),
        used_templates=("scenario_id", "nunique"),
        used_patterns=("structural_pattern_id", "nunique")
    )
)

duration_coverage_73 = (
    catalog_by_duration_73
    .join(assigned_by_duration_73, how="outer")
    .reindex(durations_73)
)

duration_coverage_73["coverage_pct"] = (
    100
    * duration_coverage_73["used_templates"]
    / duration_coverage_73["available_templates"]
)

expected_available_73 = pd.Series(
    [9, 24, 69, 99, 258, 684],
    index=durations_73
)

expected_used_73 = pd.Series(
    [9, 24, 69, 99, 144, 145],
    index=durations_73
)

check_73(
    "Catalog template counts match approved scenario rules",
    duration_coverage_73["available_templates"].equals(
        expected_available_73.rename("available_templates")
    )
)

check_73(
    "Used template counts match final Step 6 results",
    duration_coverage_73["used_templates"].equals(
        expected_used_73.rename("used_templates")
    )
)

check_73(
    "All approved short-duration templates are represented",
    (
        duration_coverage_73.loc[
            [5, 10, 20, 30], "used_templates"
        ]
        ==
        duration_coverage_73.loc[
            [5, 10, 20, 30], "available_templates"
        ]
    ).all()
)

long_usage_73 = template_usage_73[
    template_usage_73["duration"].isin([40, 50])
]

check_73(
    "Long-duration templates are never reused",
    long_usage_73["uses"].eq(1).all()
)

check_73(
    "All 289 long-duration assignments use distinct templates",
    len(long_usage_73) == 289
    and long_usage_73["uses"].sum() == 289
)

# ---------------------------------------------------------
# 7. Verify assignment provenance
# ---------------------------------------------------------

check_73(
    "Unique-template assignments contain 490 records",
    len(unique_73) == 490
)

check_73(
    "Reused-template assignments contain 505 records",
    len(reused_73) == 505
)

check_73(
    "Unique-template records use distinct scenario IDs",
    unique_73["scenario_id"].is_unique
)

check_73(
    "Unique assignment labels are correct",
    unique_73["assignment_type"].eq("Unique template").all()
)

check_73(
    "Reused assignment labels are correct",
    reused_73["assignment_type"].eq("Reused template").all()
)

check_73(
    "No user appears in both assignment partitions",
    set(unique_73["user"]).isdisjoint(
        set(reused_73["user"])
    )
)

check_73(
    "Reused scenario IDs originate from unique assignments",
    set(reused_73["scenario_id"]).issubset(
        set(unique_73["scenario_id"])
    )
)

# Compare the complete final dataset with the two
# original assignment partitions, including all columns.

reconstructed_73 = pd.concat(
    [unique_73, reused_73],
    ignore_index=True
)

def canonical_73(df):
    return (
        df[assignment_columns_73]
        .sort_values(
            assignment_columns_73,
            kind="stable"
        )
        .reset_index(drop=True)
    )

check_73(
    "Final assignments exactly match reconstructed partitions",
    canonical_73(assignments_73).equals(
        canonical_73(reconstructed_73)
    )
)

# ---------------------------------------------------------
# 8. Verify pool allocation rules
# ---------------------------------------------------------

check_73(
    "Short-duration assignments total 706",
    int(audit_73["duration"].isin([5, 10, 20, 30]).sum())
    == 706
)

check_73(
    "Long-duration assignments total 289",
    int(audit_73["duration"].isin([40, 50]).sum())
    == 289
)

check_73(
    "All reused assignments have short durations",
    reused_73["duration"].isin([5, 10, 20, 30]).all()
)

# ---------------------------------------------------------
# 9. Produce complete diagnostic report
# ---------------------------------------------------------

validation_report_73 = pd.DataFrame(checks_73)

failed_73 = validation_report_73[
    validation_report_73["status"].eq("FAIL")
]

print("\nSOURCE DIMENSIONS")
print("Final assignments:", assignments_73.shape)
print("Unique assignments:", unique_73.shape)
print("Reused assignments:", reused_73.shape)
print("Approved catalog:", catalog_73.shape)

print("\nDURATION × STEALTH DISTRIBUTION")
print(cross_distribution_73.to_string())

print("\nSCENARIO COVERAGE BY DURATION")
print(duration_coverage_73.round(3).to_string())

print("\nTEMPLATE REUSE FREQUENCY")
print(reuse_frequency_73.to_string())

print("\nSTRUCTURAL DIVERSITY")
print("Catalog templates:", catalog_73["scenario_id"].nunique())
print("Catalog structural patterns:", catalog_patterns_73)
print("Assigned templates:", audit_73["scenario_id"].nunique())
print("Assigned structural patterns:", assigned_patterns_73)

print("\nCOMPLETE VALIDATION REPORT")
print(
    validation_report_73.to_string(
        index=False,
        max_colwidth=110
    )
)

print("\nVALIDATION SUMMARY")
print("Total checks:", len(validation_report_73))
print(
    "Passed:",
    int(validation_report_73["status"].eq("PASS").sum())
)
print("Failed:", len(failed_73))

if not failed_73.empty:
    print("\nFAILED CHECKS")
    print(
        failed_73.to_string(
            index=False,
            max_colwidth=150
        )
    )
    raise AssertionError(
        "STEP 7.3 FAILED — Do not approve or export assignments."
    )

print("\nSTEP 7.3 — PASS")
print("Final distributions, structural diversity, template reuse,")
print("and assignment provenance match approved Step 6 results.")
print("No assignments were modified or exported.")


STEP 7.3 — FINAL DISTRIBUTION, DIVERSITY & PROVENANCE VALIDATION

SOURCE DIMENSIONS
Final assignments: (995, 9)
Unique assignments: (490, 9)
Reused assignments: (505, 9)
Approved catalog: (1143, 15)

DURATION × STEALTH DISTRIBUTION
stealth_level  High  Medium  Low
duration                        
5                59      58   59
10               58      59   59
20               59      60   58
30               59      59   59
40               48      48   48
50               49      48   48

SCENARIO COVERAGE BY DURATION
          available_templates  available_patterns  assignments  used_templates  used_patterns  coverage_pct
duration                                                                                                   
5                           9                   9          176               9              9       100.000
10                         24                  24          176              24             24       100.000
20                         69            


### 7.4 — Final Assignment Approval & Export

**Objective:** Finalize and export the validated Stage 2 user–scenario assignments, preserve the derived structural pattern identifiers, and produce a reproducible audit record.

**Validation Gate:**
- Confirm that all 116 checks from Steps 7.1–7.3 passed.
- Verify that all 995 eligible users are assigned exactly once.
- Confirm that the 490 unique and 505 reused assignments reconcile with the final dataset.
- Verify that the original 14 approved catalog columns remain unchanged.
- Preserve and validate the derived `structural_pattern_id` as the 15th catalog column.
- Confirm assignment compatibility and approved scenario membership.

**Target Limitation:** The original target is 1,000 attacks, but only 995 eligible users are available under the approved constraints. The remaining five attacks represent an unresolved feasibility gap, not missing assignments or a validation failure.

**Export:** Save the validated assignments, the 15-column approved scenario catalog, and a machine-readable validation manifest to `Attack_Simulation/data/`. Compute SHA256 hashes and independently reload the exported files to verify record integrity.

**Decision:** Approve the 995 assignments for Stage 3 scheduling only after all validation and post-export integrity checks pass. This approval does not establish detailed scheduling feasibility, successful injection, or detection performance.


In [50]:

# STEP 7.4 — Final Assignment Approval & Export
# Goal: Validate all Stage 2 results, preserve structural_pattern_id,
# export approved records and an audit manifest, and verify file integrity.
# Fail-closed: no reassignment, no synthetic data, no silent overwrites.

import pandas as pd
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from pandas.testing import assert_frame_equal

print("STEP 7.4 — FINAL ASSIGNMENT APPROVAL & EXPORT")
print("=" * 82)

# ---------------------------------------------------------
# 1. Resolve approved output directory
# ---------------------------------------------------------

project_root_74 = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
).resolve()

output_dir_74 = (
    project_root_74 / "Attack_Simulation" / "data"
).resolve()

if output_dir_74.parent != (
    project_root_74 / "Attack_Simulation"
).resolve():
    raise RuntimeError("EXPORT BLOCKED — Invalid output directory.")

# ---------------------------------------------------------
# 2. Verify prior validation reports
# ---------------------------------------------------------

required_74 = [
    "validation_report_71", "report_72", "validation_report_73",
    "final_provisional_assignments",
    "unique_scenario_assignments",
    "reused_scenario_assignments",
    "approved_scenarios", "eligible_user_profile",
    "assignments_73", "unique_73", "reused_73",
    "catalog_73", "validation_72"
]

missing_74 = [x for x in required_74 if x not in globals()]

if missing_74:
    raise RuntimeError(
        f"EXPORT BLOCKED — Missing objects: {missing_74}"
    )

reports_74 = {
    "7.1": (validation_report_71, 59),
    "7.2": (report_72, 26),
    "7.3": (validation_report_73, 31)
}

validation_summary_74 = {}

for step, (report, expected) in reports_74.items():
    if not {"check", "status"}.issubset(report.columns):
        raise RuntimeError(f"Invalid Step {step} report schema.")

    if len(report) != expected or not report["status"].eq("PASS").all():
        raise RuntimeError(
            f"EXPORT BLOCKED — Step {step} validation failed."
        )

    validation_summary_74[step] = {
        "total_checks": int(len(report)),
        "passed": int(report["status"].eq("PASS").sum()),
        "failed": 0
    }

if sum(x["passed"] for x in validation_summary_74.values()) != 116:
    raise RuntimeError("EXPORT BLOCKED — Validation count mismatch.")

print("Prior validation: 116/116 PASS")

# ---------------------------------------------------------
# 3. Protect approved assignments and catalog
# ---------------------------------------------------------

final_74 = final_provisional_assignments.copy(deep=True)
unique_74 = unique_scenario_assignments.copy(deep=True)
reused_74 = reused_scenario_assignments.copy(deep=True)
eligible_74 = eligible_user_profile.copy(deep=True)

original_catalog_74 = approved_scenarios.copy(deep=True)
catalog_74 = catalog_73.copy(deep=True)

assignment_columns_74 = [
    "user", "scenario_id", "assignment_pool",
    "duration", "stealth_level", "assignment_group",
    "eligible_workdays", "priority_rank", "assignment_type"
]

def canonical_assignments_74(df):
    if not df.columns.is_unique or set(df.columns) != set(assignment_columns_74):
        raise RuntimeError("EXPORT BLOCKED — Assignment schema mismatch.")
    return (
        df[assignment_columns_74]
        .sort_values(["user", "scenario_id"], kind="stable")
        .reset_index(drop=True)
    )

def canonical_catalog_74(df):
    if not df.columns.is_unique or not df["scenario_id"].is_unique:
        raise RuntimeError("EXPORT BLOCKED — Invalid catalog identifiers.")
    return df.sort_values("scenario_id", kind="stable").reset_index(drop=True)

for name, current, snapshot in [
    ("Final", final_74, assignments_73),
    ("Unique", unique_74, unique_73),
    ("Reused", reused_74, reused_73)
]:
    if not canonical_assignments_74(current).equals(
        canonical_assignments_74(snapshot)
    ):
        raise RuntimeError(
            f"EXPORT BLOCKED — {name} assignments changed."
        )

# Compare all original columns without discarding the derived column.
if len(original_catalog_74.columns) != 14:
    raise RuntimeError("EXPORT BLOCKED — Original catalog schema changed.")

if set(catalog_74.columns) != (
    set(original_catalog_74.columns) | {"structural_pattern_id"}
):
    raise RuntimeError("EXPORT BLOCKED — Unexpected catalog columns.")

if len(catalog_74.columns) != 15:
    raise RuntimeError("EXPORT BLOCKED — Expected 15 catalog columns.")

original_sorted_74 = canonical_catalog_74(original_catalog_74)
catalog_sorted_74 = canonical_catalog_74(catalog_74)

assert_frame_equal(
    original_sorted_74,
    catalog_sorted_74[original_catalog_74.columns],
    check_exact=True,
    check_dtype=True
)

# Independently reconstruct structural IDs using Step 7.3 rules.
rule_columns_74 = [
    "duration", "off_days", "active_days",
    "switching_level", "min_switches", "max_switches",
    "target_switches", "target_switching_rate",
    "two_channel_days", "three_channel_days",
    "single_channel_days", "stealth_level",
    "stealth_multiplier"
]

if catalog_sorted_74[rule_columns_74].isna().any().any():
    raise RuntimeError("EXPORT BLOCKED — Missing structural rules.")

expected_patterns_74 = (
    catalog_sorted_74
    .groupby(rule_columns_74, sort=True)
    .ngroup() + 1
)

if not catalog_sorted_74["structural_pattern_id"].reset_index(
    drop=True
).equals(expected_patterns_74.astype(
    catalog_sorted_74["structural_pattern_id"].dtype
)):
    raise RuntimeError(
        "EXPORT BLOCKED — Structural pattern IDs are inconsistent."
    )

if len(catalog_sorted_74) != 1143:
    raise RuntimeError("EXPORT BLOCKED — Catalog size mismatch.")

print("Catalog integrity: PASS")
print("Approved catalog shape:", catalog_sorted_74.shape)
print(
    "Structural patterns:",
    catalog_sorted_74["structural_pattern_id"].nunique()
)

# ---------------------------------------------------------
# 4. Reconfirm assignment approval constraints
# ---------------------------------------------------------

if (
    len(final_74) != 995
    or len(eligible_74) != 995
    or not final_74["user"].is_unique
    or not eligible_74["user"].is_unique
    or set(final_74["user"]) != set(eligible_74["user"])
):
    raise RuntimeError("EXPORT BLOCKED — Eligible-user coverage failed.")

if len(unique_74) != 490 or len(reused_74) != 505:
    raise RuntimeError("EXPORT BLOCKED — Partition counts changed.")

reconstructed_74 = pd.concat(
    [unique_74, reused_74], ignore_index=True
)

if not canonical_assignments_74(final_74).equals(
    canonical_assignments_74(reconstructed_74)
):
    raise RuntimeError("EXPORT BLOCKED — Partitions do not reconcile.")

if not set(final_74["scenario_id"]).issubset(
    set(catalog_sorted_74["scenario_id"])
):
    raise RuntimeError("EXPORT BLOCKED — Unknown scenario IDs.")

approved_attributes_74 = catalog_sorted_74[
    ["scenario_id", "duration", "stealth_level"]
].rename(columns={
    "duration": "approved_duration",
    "stealth_level": "approved_stealth"
})

matched_74 = final_74.merge(
    approved_attributes_74,
    on="scenario_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

if not (
    matched_74["_merge"].eq("both").all()
    and matched_74["duration"].eq(
        matched_74["approved_duration"]
    ).all()
    and matched_74["stealth_level"].eq(
        matched_74["approved_stealth"]
    ).all()
):
    raise RuntimeError("EXPORT BLOCKED — Scenario attributes mismatch.")

if (
    not validation_72["fully_compatible"].all()
    or not validation_72["user"].is_unique
    or set(validation_72["user"]) != set(final_74["user"])
):
    raise RuntimeError("EXPORT BLOCKED — Compatibility validation failed.")

target_74 = 1000
shortfall_74 = target_74 - len(final_74)

if shortfall_74 != 5:
    raise RuntimeError("EXPORT BLOCKED — Target shortfall changed.")

export_74 = canonical_assignments_74(final_74)
catalog_export_74 = catalog_sorted_74.copy(deep=True)

if export_74.isna().any().any() or catalog_export_74.isna().any().any():
    raise RuntimeError("EXPORT BLOCKED — Missing export values.")

print("Assignment approval gate: PASS")
print("Validated users:", len(export_74))
print("Target shortfall:", shortfall_74)

# ---------------------------------------------------------
# 5. Protect export destinations
# ---------------------------------------------------------

assignment_path_74 = (
    output_dir_74 / "validated_scenario_assignments_r62.parquet"
)
catalog_path_74 = (
    output_dir_74 / "approved_scenario_catalog_r62.parquet"
)
manifest_path_74 = (
    output_dir_74 / "stage2_assignment_validation_manifest_r62.json"
)

export_paths_74 = [
    assignment_path_74, catalog_path_74, manifest_path_74
]

existing_74 = [str(p) for p in export_paths_74 if p.exists()]

if existing_74:
    raise FileExistsError(
        "EXPORT BLOCKED — Existing files must be reviewed: "
        + str(existing_74)
    )

# ---------------------------------------------------------
# 6. Export assignments and approved catalog
# ---------------------------------------------------------

output_dir_74.mkdir(parents=True, exist_ok=True)

export_74.to_parquet(assignment_path_74, index=False)
catalog_export_74.to_parquet(catalog_path_74, index=False)

def file_sha256_74(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assignment_hash_74 = file_sha256_74(assignment_path_74)
catalog_hash_74 = file_sha256_74(catalog_path_74)

# ---------------------------------------------------------
# 7. Independently reload and verify data files
# ---------------------------------------------------------

reloaded_assignments_74 = pd.read_parquet(assignment_path_74)
reloaded_catalog_74 = pd.read_parquet(catalog_path_74)

assert_frame_equal(
    reloaded_assignments_74,
    export_74,
    check_exact=True
)

assert_frame_equal(
    reloaded_catalog_74,
    catalog_export_74,
    check_exact=True
)

if (
    file_sha256_74(assignment_path_74) != assignment_hash_74
    or file_sha256_74(catalog_path_74) != catalog_hash_74
):
    raise RuntimeError("POST-EXPORT VERIFICATION FAILED — Hash mismatch.")

# ---------------------------------------------------------
# 8. Create machine-readable audit manifest
# ---------------------------------------------------------

manifest_74 = {
    "project": "CERT r6.2 Cross-Channel Data Exfiltration",
    "stage": "Stage 2 — User Eligibility & Scenario Assignment",
    "status": "APPROVED_WITH_TARGET_SHORTFALL",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "target_attacks": target_74,
    "validated_assignments": int(len(export_74)),
    "target_shortfall": int(shortfall_74),
    "unique_users": int(export_74["user"].nunique()),
    "unique_templates": int(export_74["scenario_id"].nunique()),
    "unique_assignment_records": int(len(unique_74)),
    "reused_assignment_records": int(len(reused_74)),
    "duration_distribution": {
        str(k): int(v)
        for k, v in export_74["duration"].value_counts().sort_index().items()
    },
    "stealth_distribution": {
        str(k): int(v)
        for k, v in export_74["stealth_level"].value_counts().items()
    },
    "maximum_template_usage": int(
        export_74["scenario_id"].value_counts().max()
    ),
    "catalog_scenarios": int(len(catalog_export_74)),
    "catalog_columns": int(len(catalog_export_74.columns)),
    "structural_patterns": int(
        catalog_export_74["structural_pattern_id"].nunique()
    ),
    "structural_pattern_column": "structural_pattern_id",
    "validation": validation_summary_74,
    "limitations": [
        "Five attacks remain below the original 1000-attack target.",
        "Scenario assignment does not prove detailed scheduling feasibility.",
        "Attack injection and feature recalculation are not yet performed.",
        "Detection performance has not been evaluated."
    ],
    "assignment_file": assignment_path_74.name,
    "assignment_sha256": assignment_hash_74,
    "catalog_file": catalog_path_74.name,
    "catalog_sha256": catalog_hash_74
}

with open(manifest_path_74, "w", encoding="utf-8") as f:
    json.dump(manifest_74, f, indent=2, ensure_ascii=False)

with open(manifest_path_74, "r", encoding="utf-8") as f:
    reloaded_manifest_74 = json.load(f)

if reloaded_manifest_74 != manifest_74:
    raise RuntimeError(
        "POST-EXPORT VERIFICATION FAILED — Manifest mismatch."
    )

if (
    reloaded_manifest_74["validated_assignments"]
    != len(reloaded_assignments_74)
    or reloaded_manifest_74["catalog_scenarios"]
    != len(reloaded_catalog_74)
    or reloaded_manifest_74["assignment_sha256"]
    != file_sha256_74(assignment_path_74)
    or reloaded_manifest_74["catalog_sha256"]
    != file_sha256_74(catalog_path_74)
):
    raise RuntimeError(
        "POST-EXPORT VERIFICATION FAILED — Manifest consistency."
    )

# ---------------------------------------------------------
# 9. Final approval summary
# ---------------------------------------------------------

print("\nVALIDATION GATE")
for step, result in validation_summary_74.items():
    print(
        f"Step {step}: {result['passed']}/"
        f"{result['total_checks']} PASS"
    )

print("\nFINAL ASSIGNMENT SUMMARY")
print("Target attacks:", target_74)
print("Validated assignments:", len(export_74))
print("Target shortfall:", shortfall_74)
print("Unique users:", export_74["user"].nunique())
print("Unique-template records:", len(unique_74))
print("Reused-template records:", len(reused_74))
print("Unique scenario templates:", export_74["scenario_id"].nunique())

print("\nFINAL CATALOG")
print("Shape:", catalog_export_74.shape)
print(
    "Structural patterns:",
    catalog_export_74["structural_pattern_id"].nunique()
)

print("\nEXPORTED FILES")
print("Assignments:", assignment_path_74)
print("Catalog:", catalog_path_74)
print("Manifest:", manifest_path_74)
print("Assignment SHA256:", assignment_hash_74)
print("Catalog SHA256:", catalog_hash_74)

print("\nPOST-EXPORT VERIFICATION")
print("Assignment record equality: PASS")
print("Catalog record equality: PASS")
print("SHA256 integrity: PASS")
print("Manifest consistency: PASS")

print("\nSTAGE 2 STATUS: APPROVED WITH TARGET SHORTFALL")
print("995 assignments approved for Stage 3 scheduling.")
print("Five attacks remain below the original target.")
print("No files were written to Clear_Attack_Simulation.")


STEP 7.4 — FINAL ASSIGNMENT APPROVAL & EXPORT
Prior validation: 116/116 PASS
Catalog integrity: PASS
Approved catalog shape: (1143, 15)
Structural patterns: 1143
Assignment approval gate: PASS
Validated users: 995
Target shortfall: 5

VALIDATION GATE
Step 7.1: 59/59 PASS
Step 7.2: 26/26 PASS
Step 7.3: 31/31 PASS

FINAL ASSIGNMENT SUMMARY
Target attacks: 1000
Validated assignments: 995
Target shortfall: 5
Unique users: 995
Unique-template records: 490
Reused-template records: 505
Unique scenario templates: 490

FINAL CATALOG
Shape: (1143, 15)
Structural patterns: 1143

EXPORTED FILES
Assignments: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Attack_Simulation\data\validated_scenario_assignments_r62.parquet
Catalog: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Attack_Simulation\data\approved_scenario_catalog_r62.parquet
Manifest: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Attack_Simulation\data\stage2_assignment_validation_manifest_r62.json
Assignment SHA256: 039c15f15d6a8bad90

In [51]:

# STAGE 2 — Display Approved User–Scenario Assignments
# Goal: Display the actual approved scenario assigned to each user.
# Read-only: No modifications to the approved dataset.

import pandas as pd
from IPython.display import display

assignments_view = pd.read_parquet(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Attack_Simulation\data\validated_scenario_assignments_r62.parquet"
)

print("Total assignments:", len(assignments_view))
print("Unique users:", assignments_view["user"].nunique())
print("Unique scenarios:", assignments_view["scenario_id"].nunique())

display(
    assignments_view
    .sort_values("user")
    .reset_index(drop=True)
)


Total assignments: 995
Unique users: 995
Unique scenarios: 490


,user,scenario_id,assignment_pool,duration,stealth_level,assignment_group,eligible_workdays,priority_rank,assignment_type
0,AAC0610,SCN_0108,Short-duration pool,30,Low,Two-channel short-duration,105,24,Unique template
1,AAF0819,SCN_0007,Short-duration pool,5,High,Two-channel short-duration,105,25,Unique template
2,AAP0352,SCN_0016,Short-duration pool,10,High,Two-channel short-duration,105,26,Unique template
3,AAP1919,SCN_0040,Short-duration pool,20,High,Two-channel short-duration,105,27,Unique template
4,AAS2987,SCN_0109,Short-duration pool,30,High,Two-channel short-duration,105,28,Unique template
...,...,...,...,...,...,...,...,...,...
990,ZDC2523,SCN_0180,Short-duration pool,30,Low,Two-channel short-duration,107,705,Reused template
991,ZGP2957,SCN_0149,Short-duration pool,30,Medium,Two-channel short-duration,105,577,Reused template
992,ZHF1006,SCN_0037,Short-duration pool,20,High,Two-channel short-duration,105,578,Reused template
993,ZNB3313,SCN_0011,Short-duration pool,10,Medium,Two-channel short-duration,105,579,Reused template
